> **Fork note.** Unmodified copy of the public notebook [CASMI 26: From Spectra to Structures](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-from-spectra-to-structures) by dmitriigluzdov (which builds on Ahmed Berat Özer's public v4g–v4m pipeline, Apache 2.0, and seyitkaangunes' engine fusion), re-run under dalloliogm's account to measure its public-leaderboard score. All credit to the original authors.

# CASMI 26: From Spectra to Structures

### From measured fragments to ranked molecular structures

An unknown molecule can yield several tandem mass spectra (MS/MS), each recording the fragment ions of one selected precursor ion. The competition asks for one ordered list of up to 25 structures per molecule. This notebook connects the measured fragments to candidate structures, examines the available evidence and runs a reproducible ranking method. Figure 1 sets the scene with caffeine: its structure, two of its measured spectra and the size of the reference libraries.

**How to read it.** Sections 1–2 explain the task with one worked example; Sections 3–9 explore the data; Sections 10–12 describe and run the model; Sections 13–17 discuss next steps and sources. Readers new to mass spectrometry may start with [a domain primer for non-chemists](https://www.kaggle.com/code/mohammadhmozafary/casmi-2026-domain-knowledge-for-non-chemists).

In [ ]:
import sys, subprocess, importlib.metadata, time
NOTEBOOK_STARTED = time.monotonic()   # the second engine is given only the time the later stages do not need
from pathlib import Path
if not Path('/kaggle/input').exists() and Path('.runtime/rdkit2026033').exists():
    sys.path.insert(0, str(Path('.runtime/rdkit2026033').resolve()))
try:
    installed = importlib.metadata.version('rdkit')
except importlib.metadata.PackageNotFoundError:
    installed = ''
if installed.replace('.03.', '.3.') != '2026.3.3':
    tag=f'cp{sys.version_info.major}{sys.version_info.minor}'
    wheels=sorted(Path('/kaggle/input').rglob(f'rdkit-2026.3.3-{tag}-*.whl'))
    assert wheels, f'Attach the RDKit 2026.3.3 offline wheel for {tag}.'
    subprocess.check_call([sys.executable,'-m','pip','install','--no-index','--no-deps','-q',str(wheels[0])])
from rdkit import rdBase
assert rdBase.rdkitVersion == '2026.03.3', rdBase.rdkitVersion
print('Official-metric RDKit:', rdBase.rdkitVersion)


In [ ]:
from pathlib import Path
import os
import hashlib
import platform
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, Draw, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import DataStructs
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

SEED = 20260915
rng = np.random.default_rng(SEED)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (11, 5.5), "axes.titleweight": "bold", "figure.dpi": 120})

# Sharp inline figures, with PNG retained for readers that do not support SVG.
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg', 'png')
plt.rcParams.update({
    'figure.dpi': 160, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'font.size': 12, 'axes.labelsize': 12, 'axes.titlesize': 13,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'svg.fonttype': 'none', 'figure.facecolor': 'white',
})


In [ ]:
def locate_data_dir():
    candidates = [
        Path("/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra"),
        Path("/kaggle/input/enveda-casmi26-molecule-id-mass-spectra"),
        Path("/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra"),
        Path("data/raw"),
        Path("../data/raw"),
    ]
    for path in candidates:
        if all((path / name).exists() for name in ("train.parquet", "test.parquet", "sample_submission.csv")):
            return path.resolve()
    raise FileNotFoundError("Could not find train.parquet, test.parquet, and sample_submission.csv")


DATA = locate_data_dir()
TRAIN_PATH = DATA / "train.parquet"
TEST_PATH = Path(os.environ.get("CASMI_TEST_PATH", DATA / "test.parquet"))
SAMPLE_PATH = Path(os.environ.get("CASMI_SAMPLE_PATH", DATA / "sample_submission.csv"))

In [ ]:
# The journal-style header is assembled from measured competition data: caffeine, the worked example of Sections 1-2.
CAFFEINE_KEY = "RYYVLZVUVIJVGH"
header_columns = ["normalized_smiles", "ms2_mzs", "ms2_normalized_intensities", "adduct", "precursor_mz",
                  "ingest_lib", "instrument_type", "collision_energy_ev"]
header_file = pq.ParquetFile(TRAIN_PATH)
caffeine_parts = []
for group in range(header_file.num_row_groups):
    keys = header_file.read_row_group(group, columns=["inchikey14", "adduct"]).to_pandas()
    hit = np.where((keys["inchikey14"] == CAFFEINE_KEY) & (keys["adduct"] == "[M+H]+"))[0]
    if len(hit):
        caffeine_parts.append(header_file.read_row_group(group, columns=header_columns).to_pandas().iloc[hit])
caffeine = pd.concat(caffeine_parts, ignore_index=True) if caffeine_parts else pd.DataFrame(columns=header_columns)


def single_energy(value):
    try:
        values = [float(v) for v in value]
    except TypeError:
        return np.nan
    return values[0] if len(values) == 1 else np.nan


caffeine["energy"] = caffeine["collision_energy_ev"].map(single_energy)
caffeine["peaks"] = caffeine["ms2_mzs"].map(len)
timstof = caffeine[(caffeine["ingest_lib"] == "enveda-np-examples") & caffeine["energy"].notna()].sort_values("energy")
if len(timstof) >= 2:
    views = [timstof.iloc[0], timstof.iloc[-1]]
else:
    views = [row for _, row in caffeine.sort_values("peaks", ascending=False).head(2).iterrows()]
is_caffeine = len(views) == 2 and len(timstof) >= 2
if not views:
    preview_pool = header_file.read_row_group(0, columns=header_columns).slice(0, 700).to_pandas()
    preview_pool["peaks"] = preview_pool["ms2_mzs"].map(len)
    views = [preview_pool.sort_values("peaks", ascending=False).iloc[0]]
mol = Chem.MolFromSmiles(views[0]["normalized_smiles"])
mol_image = Draw.MolToImage(mol, size=(900, 650), fitImage=True)
header_meta = pd.read_parquet(TRAIN_PATH, columns=["ingest_lib", "inchikey14"])
header_connectivities = header_meta["inchikey14"].nunique()
source_counts = header_meta["ingest_lib"].value_counts().head(6).sort_values()
del header_meta


def draw_view(ax, row, title, labels, step):
    """One measured spectrum; labels the given fragment m/z values and draws the mass step between two of them."""
    mz = np.asarray(row["ms2_mzs"], dtype=float)
    intensity = np.asarray(row["ms2_normalized_intensities"], dtype=float)
    precursor = float(row["precursor_mz"])
    ax.vlines(mz, 0, intensity, color="#b74759", linewidth=1.3)
    ax.axvline(precursor, color="#46566b", linestyle=":", linewidth=1)
    ax.text(precursor, 1.37, f"precursor\n{precursor:.3f}", ha="center", va="bottom", fontsize=9, color="#46566b")
    ax.set(xlabel="Fragment m/z", ylabel="Relative intensity", title=title, ylim=(0, 1.55), xlim=(30, precursor + 18))
    ax.set_yticks([0, 0.5, 1.0])
    for target in labels:
        j = int(np.argmin(np.abs(mz - target)))
        if abs(mz[j] - target) < 0.01:
            ax.text(mz[j], intensity[j] + 0.03, f"{mz[j]:.3f}", ha="center", va="bottom", fontsize=9)
    if step:
        start, end, text = step
        if not np.any(np.abs(mz - start) < 0.01):          # the arrow starts at a fragment that this spectrum no longer shows
            ax.axvline(start, color="#46566b", linestyle=":", linewidth=0.8)
            ax.text(start, 1.37, f"{start:.3f}\nnot seen", ha="center", va="bottom", fontsize=9, color="#46566b")
        ax.annotate("", xy=(end, 1.16), xytext=(start, 1.16), arrowprops=dict(arrowstyle="->", color="#16324f", lw=1.4))
        ax.text((start + end) / 2, 1.19, text, ha="center", va="bottom", fontsize=10, color="#16324f")


fig = plt.figure(figsize=(11.0, 10.5), facecolor="white")
grid = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=.5, wspace=.3, left=.1, right=.96, bottom=.1, top=.86)
ax0 = fig.add_subplot(grid[0, 0])
ax0.imshow(mol_image); ax0.axis("off")
if is_caffeine:
    ax0.set_title("A. Caffeine, C$_8$H$_{10}$N$_4$O$_2$", loc="left")
    precursor = float(views[0]["precursor_mz"])
    draw_view(fig.add_subplot(grid[0, 1]), views[0], f"B. {views[0]['energy']:.0f} eV: one piece breaks off",
              [138.0666], (precursor, 138.0666, "−57.021 Da (C$_2$H$_3$NO)"))
    draw_view(fig.add_subplot(grid[1, 0]), views[1], f"C. {views[1]['energy']:.0f} eV: the ion breaks further",
              [123.0427], (138.0666, 123.0427, "−15.023 Da (CH$_3$)"))
    ax2 = fig.add_subplot(grid[1, 1])
    subtitle = f"Caffeine · [M+H]+ at m/z {precursor:.3f} · measured on timsTOF (enveda-np-examples)"
else:
    ax0.set_title("A. Known molecular graph", loc="left")
    for k, row in enumerate(views[:2]):
        draw_view(fig.add_subplot(grid[0, 1] if k == 0 else grid[1, 0]), row, "BC"[k] + ". Measured fragments", [], None)
    ax2 = fig.add_subplot(grid[1, 1] if len(views) > 1 else grid[1, :])
    subtitle = f"One measured example · precursor {float(views[0]['precursor_mz']):.4f} m/z · {views[0]['adduct']}"
ax2.barh(source_counts.index, source_counts.values / 1e3, color="#287a8c")
ax2.set(xlabel="Measured spectra (thousands)", title="D. Largest reference libraries")
ax2.set_xlim(0, source_counts.max() / 1e3 * 1.3)
ax2.grid(axis="y", visible=False)
for y, value in enumerate(source_counts.values):
    ax2.text(value / 1e3 + source_counts.max() / 1e3 * .014, y, f"{value / 1e3:,.0f}k", va="center", fontsize=10)
fig.suptitle("FROM SPECTRA TO STRUCTURES", fontsize=21, weight="bold", color="#16324f", y=.965)
fig.text(.5, .912, subtitle, ha="center", fontsize=12)
fig.text(.5, .025, f"{header_file.metadata.num_rows:,} spectra · {header_connectivities:,} stored connectivity groups",
         ha="center", fontsize=12, color="#46566b")
plt.show()

*Figure 1. Graphical abstract built from the competition corpus. (A) Caffeine, the worked example of Sections 1–2. (B, C) Two of its `[M+H]+` spectra in the `enveda-np-examples` library, measured on timsTOF, the instrument family of the hidden test. At 35 eV the protonated ion (m/z 195.088) mostly loses 57.021 Da, a C₂H₃NO piece, and m/z 138.067 is the strongest peak; at 60 eV neither survives, and m/z 123.043, 15.023 Da (the mass of a methyl group) below 138.067, dominates. (D) The six largest source libraries. The task runs the other way: from spectra like B and C back to the structure in A.*

## Abstract

Precursor mass removes most impossible structures, yet dozens or thousands of candidates can share almost the same mass. The corpus holds millions of measured spectra and repeated acquisitions of many molecules, so the central task is to turn fragment patterns into an ordered candidate list. This notebook shows how candidate coverage limits identification accuracy, then runs an attributed public method: it searches training and COCONUT structures, ranks them with spectral and molecular features, re-scores same-formula isomers with two predicted-spectrum models, merges in PubChem proposals and, when no strong library match exists, fuses the result with the ranking of a second, independently built engine.

The method builds on [Ahmed Berat Özer's public inference pipeline](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4h-inference), the [engine fusion by seyitkaangunes](https://www.kaggle.com/code/seyitkaangunes/casmi26-v4n-engine-fusion-union-lb-0-399), the deeper [PubChem retrieval screen](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4m-inference) and [wangpenghua's popularity prior](https://www.kaggle.com/code/wangpenghua/casmi26-v21-v4i-pop). PubChem substance and article counts favour documented candidates; Figure 18 shows the database enrichment that motivates this choice. The [attributed model assets](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets) and the [rebuilt popularity dataset](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior) make every inference resource available for reuse. Public scores of the successive versions are listed in Appendix A, and Section 11 describes an identity-disjoint validation protocol and reports a development benchmark on 250 natural products.


## 1. From a molecule to a spectrum — and back to SMILES

**The idea in one paragraph.** A mass spectrometer works like a very precise scale combined with a hammer. It weighs a charged molecule, breaks it into pieces and weighs the pieces. We never see the molecule itself, only these weights and how strongly each piece was detected. The task is to name the molecule: to write up to 25 candidate structures as **SMILES** strings ([Simplified Molecular Input Line Entry System](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System), a one-line text notation for a molecular graph), most likely first.

### How a mass spectrometer weighs a molecule

No balance can weigh a single molecule. A mass spectrometer instead gives the molecule an electric charge and watches how the charged particle, an **ion**, moves in an electric field. The hidden test was measured on timsTOF instruments, whose mass analyser is a **time-of-flight (TOF)** tube, so that is the example here.

1. **Give the molecule a charge.** In electrospray ionisation (**ESI**), the usual source for liquid samples such as plant extracts, the dissolved molecule picks up a proton and becomes a positive ion (`[M+H]+`), or loses one and becomes a negative ion (`[M-H]-`, **negative mode**). The molecule stays intact. Harsher sources, such as a beam of fast electrons that knocks an electron out, often break the molecule at once; tandem mass spectrometry breaks it later and in a controlled way (step 4).
2. **Accelerate the ions.** A voltage $V$ gives every ion with $z$ elementary charges $e$ the same energy, $zeV$, which turns into motion:
$$
zeV=\frac{1}{2}mv^2 \quad\Rightarrow\quad v=\sqrt{\frac{2zeV}{m}}
$$
3. **Time the flight.** Over a flight tube of length $L$ an ion needs $t = L/v = L\sqrt{m/(2zeV)}$: heavier ions arrive later, and the arrival time depends only on the ratio $m/z$. The instrument converts every arrival time into a mass-to-charge ratio, **m/z**. Most ions here carry a single charge ($z = 1$), so m/z is simply the mass of the ion.
4. **Break and weigh again (MS/MS).** The instrument keeps only ions of one m/z, the **precursor**, lets them collide with gas molecules so that they break, and times the pieces in the same way. Only charged pieces reach the detector; a neutral piece that falls off is invisible and appears as a mass difference, a **neutral loss** (the arrows in Figure 1).

**From mass to formula.** Masses are given in daltons (Da). One dalton is one twelfth of the mass of a carbon-12 atom, close to the mass of a hydrogen atom, and a mole of molecules of $M$ Da each weighs about $M$ grams. Atomic masses are not whole numbers (H 1.00783, C 12 exactly, N 14.00307, O 15.99491), so an accurately measured mass allows only a few elemental formulas. Caffeine's neutral mass, 194.0804 Da, fits C₈H₁₀N₄O₂; C₇H₁₄O₆ has the same whole-number mass, 194, but weighs 194.0790 Da. The difference, 7 parts per million, is within reach of an accurate instrument, although the 10-ppm search window used below still admits both. A formula, however, does not fix the structure: the same atoms can be connected in many ways, and the fragments have to tell those apart.

### One molecule from start to finish: caffeine

Caffeine (C₈H₁₀N₄O₂, Figure 1A) has 645 spectra in the training file, from nine source libraries; four of them were measured on timsTOF, the instrument family of the hidden test. Following it through the experiment shows each step that the model has to invert; Figure 2 draws the same steps as a diagram.

| Step | What happens | Caffeine |
|---|---|---|
| Separation (**LC**, liquid chromatography) | a mixture is spread out in time, so its molecules reach the instrument one after another | the files contain no retention times, so this step leaves no trace in the data |
| Ionisation (**ESI**, electrospray) | the molecule picks up a charge (step 1 above); the **adduct** names the ion form | `[M+H]+`, one added proton, in 474 spectra; `[M+Na]+`, an added sodium ion, in 149 |
| Weighing the ion (**MS1**) | the instrument measures the mass-to-charge ratio, **m/z** | m/z 195.0877; subtracting the proton (1.0073) gives the neutral **monoisotopic** mass (computed with each element's most abundant isotope), 194.0804 Da |
| Breaking the ion (**CID**, collision-induced dissociation) | the selected ion is accelerated into gas molecules and breaks apart | at a collision energy of 35 eV (electron volts) the ion mostly loses 57.021 Da, a C₂H₃NO piece (methyl isocyanate), leaving m/z 138.067 (Figure 1B); this fragment is the strongest one in about 60% of caffeine's `[M+H]+` spectra |
| Weighing the pieces (**MS2**, or MS/MS) | a spectrum: each peak is one fragment m/z with its intensity, scaled so that the strongest peak (the **base peak**) is 1 | at 60 eV the ion breaks further and m/z 123.043 dominates (Figure 1C): the collision energy changes the pattern; Figure 11 shows more spectra |

### Running the experiment backwards

The neutral mass narrows the search, but not to one answer. Within 10 parts per million (ppm) of 194.0804 Da, about ±0.002 Da, the 710,701-structure candidate pool of this notebook holds 48 structures. Thirty-one are sugar-like compounds (C₇H₁₄O₆, for example methyl glycosides) whose mass differs from caffeine's by only 7 ppm, and two have a third formula. Fifteen share caffeine's formula exactly, among them isocaffeine, which differs only in the position of one methyl group. Mass cannot separate these fifteen; the fragments have to. A loss of 57.021 Da fits caffeine, but one loss rarely settles the question: isomers can lose the same piece, and rearrangements make a bond-by-bond reading unreliable. The useful evidence is the whole pattern of fragments across all spectra of a molecule.

Caffeine is the easy case, because its own reference spectra are in the library and a direct spectral search finds it ([section 14 of A Complete Statistical Tour](https://www.kaggle.com/code/josefreitasalvesneto/casmi-2026-a-complete-statistical-tour) measures how far that alone goes; [a public retrieval notebook](https://www.kaggle.com/code/haideptry/enveda-casmi-2026-fast-spectral-cosine-baseline) records the same ceiling in its version history before adding analogue, fragmentation and fingerprint channels). A molecule without a reference spectrum needs indirect evidence: a predicted fingerprint (a vector of yes/no answers about substructures), a predicted spectrum or measured relatives. A molecule that no database contains has to be generated. Section 2 lists these three situations.

[CSI:FingerID](https://www.pnas.org/doi/10.1073/pnas.1509788112) and [MIST](https://www.nature.com/articles/s42256-023-00708-3) predict fingerprints that help rank database structures; [MSNovelist](https://www.nature.com/articles/s41592-022-01486-3) generates structures de novo, as does the [de novo tutorial notebook](https://www.kaggle.com/code/inversion/casmi-denovo-tutorial-notebook) published for this competition; [MassSpecGym](https://github.com/pluskal-lab/MassSpecGym) provides evaluation tasks for both settings. Here, database retrieval is supplemented by a constrained generator that modifies spectral analogues, as described in Section 10.

**Illustrated background.** The Wikipedia articles on [time-of-flight mass spectrometry](https://en.wikipedia.org/wiki/Time-of-flight_mass_spectrometry), [tandem mass spectrometry](https://en.wikipedia.org/wiki/Tandem_mass_spectrometry) and [electrospray ionisation](https://en.wikipedia.org/wiki/Electrospray_ionization) have instrument diagrams, and the article on [the dalton](https://en.wikipedia.org/wiki/Dalton_%28unit%29) explains the mass unit. [Domain Knowledge for Non-Chemists](https://www.kaggle.com/code/mohammadhmozafary/casmi-2026-domain-knowledge-for-non-chemists) follows caffeine through the measurement columns of the training file and explains SMILES, stereochemistry and InChIKeys from scratch. [A Complete Statistical Tour](https://www.kaggle.com/code/josefreitasalvesneto/casmi-2026-a-complete-statistical-tour) dissects real spectra, adducts, collision energies and neutral losses.

In [ ]:

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.set(xlim=(0, 1), ylim=(0, 1)); ax.axis("off")
steps = [(0.07,"LC\nseparation"),(0.23,"ESI\nionisation"),(0.39,"MS1\nprecursor"),
         (0.55,"CID\nfragmentation"),(0.71,"MS2\nfragments"),(0.92,"ranked\nSMILES")]
colors = sns.color_palette("crest", len(steps))
for i_step, ((x, label), color) in enumerate(zip(steps, colors)):
    ax.text(x, .57, label, ha="center", va="center", fontsize=12, color="white" if i_step >= 2 else "#16324f", zorder=3,
            bbox=dict(boxstyle="round,pad=.65", fc=color, ec="white", lw=1.5))
    if i_step < len(steps)-1:
        next_x = steps[i_step+1][0]
        ax.annotate("", xy=(next_x-.065,.57), xytext=(x+.065,.57), zorder=2,
                    arrowprops=dict(arrowstyle="-|>", mutation_scale=16, lw=1.8, color="#46566b"))
ax.axvline(.815, ymin=.20, ymax=.83, color="#b04a5a", ls="--", lw=1.5)
ax.text(.41,.13,"physical acquisition",ha="center",fontsize=12,color="#284b63",weight="bold")
ax.text(.91,.13,"inverse problem",ha="center",fontsize=12,color="#9b2c3b",weight="bold")
ax.set_title("Figure 2. From experimental acquisition to structure inference", pad=12, fontsize=16)
plt.show()


*Figure 2. From experimental acquisition to structure inference. MS1 characterises precursor ions; an isolation step selects a precursor window for fragmentation; MS2 records the resulting spectrum. The computational task begins with these measurements and ends with ranked molecular structures.*


## 2. What earns a correct prediction?

Submit one ranked list per **`molecule_id`**, combining all its spectra. The [official evaluation](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/overview) uses **mean reciprocal rank among the first 25 guesses (MRR@25)**:

$$
\operatorname{MRR@25}=\frac{1}{U}\sum_{u=1}^{U}
\begin{cases}
1/r_u, & \text{if the first correct candidate has rank }r_u\leq25,\\
0, & \text{if it is absent from the first 25.}
\end{cases}
$$

In words: each of the $U$ molecules earns one over the rank of its first correct guess, or nothing if the answer is not among the first 25, and the score is the average ([mean reciprocal rank](https://en.wikipedia.org/wiki/Mean_reciprocal_rank)). Every molecule has equal weight, however many spectra it has. A small example with three molecules:

| Molecule | Rank of the correct structure | Contribution |
|---|---:|---:|
| A | 1 | 1 |
| B | 4 | 0.25 |
| C | not among the 25 guesses | 0 |
| **MRR@25** | | **(1 + 0.25 + 0) / 3 = 0.42** |

The first ranks dominate: rank 2 is worth 0.5 and rank 25 only 0.04, so lifting one answer from rank 2 to rank 1 adds as much as finding a missing answer at rank 2.

**What counts as the same answer.** The scorer compares *connectivity*, which atoms are bonded to which. It uses **RDKit 2026.03.3**, a chemistry toolkit, to pick one standard form among **tautomers** (structures that differ mainly in where a hydrogen and a double bond sit), then compares the first 14 characters of the [InChIKey](https://en.wikipedia.org/wiki/International_Chemical_Identifier), a hashed structure identifier whose first block encodes connectivity. Keys computed with the scorer's RDKit version:

| Pair | Same answer? | Why |
|---|---|---|
| L-alanine and D-alanine | yes (`QNAYBMKLOCPYGJ`) | mirror images: stereochemistry is ignored |
| the two carvones, one smelling of spearmint and one of caraway | yes (`YAHNGTQFNICUIJ`) | mirror images again |
| 2-pyridone and 2-hydroxypyridine | yes (`UBQKCCHYAOITMY`) | tautomers collapse to one form |
| caffeine and isocaffeine | no (`RYYVLZVUVIJVGH` vs `LPHGQDQBBGAPDZ`) | same formula, one methyl group elsewhere |

Two consequences follow for modelling: get the correct connectivity into the candidate list, then move it up. Stereoisomers and tautomers need no separate guesses, and repeating an equivalent structure wastes a rank. A close analogue earns nothing: a list that offers isocaffeine but never caffeine scores zero for that molecule. Export up to 25 best-first SMILES, separated by semicolons, in `molecule_id,smiles` columns. The [primer for non-chemists](https://www.kaggle.com/code/mohammadhmozafary/casmi-2026-domain-knowledge-for-non-chemists) explains InChIKeys with examples, and [another public notebook](https://www.kaggle.com/code/dariushafshar/casmi26-10-4-of-keys-change-0-48-cost-you) measures how often the tautomer step changes the key that a structure library ships (10.4% of COCONUT) and how rarely that changes a score.

### The visible sample and the hidden task

The [official data description](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data) distinguishes three novelty classes, according to the evidence available for the answer:

**Evidence classes.**

| Available evidence for the answer | Example | Main route to a candidate | Main failure mode |
|---|---|---|---|
| Public reference spectrum | caffeine, with 645 training spectra | direct library search | acquisition differences or incorrect ranking |
| PubChem/COCONUT structure, no public spectrum | most catalogued natural products: only 8.2% of the LOTUS–NPAtlas entries have a training spectrum (Figure 16) | structure database plus indirect spectral evidence | missing candidate or weak discrimination among isomers |
| Novel structure, absent from PubChem | a newly discovered metabolite | de novo structure proposal | the correct graph is never generated |

Class membership and proportions are hidden; [one public starter notebook](https://www.kaggle.com/code/denpugovkin/three-hidden-contests-one-ranked-list) treats them as three separate contests inside one ranked list. The released `test.parquet` is only a **format example**: its spectra come from the training file, so it checks that a notebook runs, not how well it identifies new molecules. Kaggle reruns the notebook against a hidden test file for its leaderboard score.

For an offline estimate, keep complete molecules out of the reference library and training features before ranking them. Section 11 describes such a protocol and reports a development benchmark; Section 12 checks execution. [Data Pitfalls That Break Your CV](https://www.kaggle.com/code/kitopl/casmi-2026-data-pitfalls-that-break-your-cv) shows, step by step, why simpler splits give optimistic numbers.

### Files used in this notebook

The [competition data](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data) provide `train.parquet` (measured spectra with known structures; several spectra can belong to one molecule), `test.parquet` and `sample_submission.csv`. When it scores an accepted Code submission, Kaggle [replaces the visible format example with hidden spectra](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/discussion/741404). Everything else comes from attached datasets:

| Dataset | What it gives this notebook |
|---|---|
| [CASMI26 attributed v4g resource fork](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets) | five public source packages as 134 hash-pinned files (next table) |
| [Open Model and Candidate Resources](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-open-model-and-candidate-resources) | the pinned RDKit 2026.03.3 offline wheel and the inputs of the second engine: COCONUT fingerprints, two spectrum encoders and ranker training rows, all unchanged public files |
| [ChEBI and LIPID MAPS candidates by prvsiyan](https://www.kaggle.com/datasets/prvsiyan/chebi-lipidmaps-casmi26) (CC BY-NC-SA 4.0) | additional candidate structures for the second engine |
| [GLACIER package](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-glacier) | a MassSpecGym-trained forward-spectrum checkpoint with offline inference code, reusing ICEBERG's isolated dependencies; it supplies predicted spectra, not new references or candidates |
| [PubChem popularity prior](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior) (ours) | for every PubChem-tier and pool row, `log(1 + substance records)` and `log(1 + PubMed articles)` from the official [PubChem `CID-SID` and `CID-PMID` files](https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/) |
| [natural-product knowledge table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table), [identity-disjoint validation folds](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-identity-disjoint-validation-folds), [natural-product panel benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark) | analysis only: 152,717 catalogued natural products with classes and organisms, the validation folds and the candidate tables behind Section 11 |

**Mounted resource packages.**

| Package in the fork | What it supplies | Interpretation |
|---|---|---|
| `v2_pool` | 710,701 structure rows, fragments and fingerprint arrays | Train-derived and COCONUT candidate structures; the measured reference spectra come from the separately mounted competition training file. |
| `v3_models` | Earlier code and FPNet A+B weights | An isolated PubChem proposal channel; its package does not replace the main engine. |
| `v4b` | Main engine, FPNet A, `fe_v4` features and a four-booster LightGBM ranker | Pretrained candidate scoring; this notebook does not fit the checkpoints. |
| `pubchem_tier` | Mass-sorted PubChem structure arrays | A second, structure-only candidate universe; its entries are not measured reference spectra. |
| `iceberg` | Forward-spectrum code, checkpoints and offline wheels | Optional bounded same-formula reranking for covered positive-mode spectra. |

The notebook checks every input hash and restores the original paths with symlinks, so the large NumPy arrays stay memory-mapped in Kaggle Input. **Terms.** The five upstream packages carry Kaggle `Other` terms with component-specific restrictions; the fork keeps their source notices, does not relicense their weights or competition-derived data, and its card gives the original links and roles. See its `SOURCE-LICENSES.md` before reuse beyond this non-commercial competition workflow. GLACIER's `LICENSE-NOTICE.txt` explains its MIT source and the uploader's basis for using the checkpoint; Kaggle lists that package under **Other** terms. The popularity card documents the release date, the matching rules (stereo-free SMILES for PubChem rows, the InChIKey connectivity block for pool rows; stereoisomers that share one scored connectivity pool their counts), the matching rates and the build script. Section 14 names other resources as further reading.

## 3. File-level audit and schema

The first question is what a row represents. Here, a row is a **spectrum**, while the prediction target is a **molecule**. Several acquisitions can therefore support one answer.

`ingest_lib` identifies the source collection, `adduct` describes the measured ion, and `inchikey14` supplies a practical training-connectivity group. Keeping these roles separate prevents two common mistakes: counting repeated spectra as independent molecules and treating library provenance as if it were a molecular property. For example, a GNPS tag says where a spectrum came from, not which chemical class the molecule belongs to.

Tables 1 and 2 give the file sizes and explain each stored field.


In [ ]:
train_pf = pq.ParquetFile(TRAIN_PATH)
test_pf = pq.ParquetFile(TEST_PATH)

FIELD_MEANING = {
    "ingest_lib": "source spectral library", "normalized_smiles": "standardised molecular graph",
    "inchikey": "full hashed structure identifier", "inchikey14": "stored connectivity key for training grouping; scoring recomputes after tautomer canonicalisation",
    "molecular_formula": "elemental composition", "ionization_mode": "positive or negative ion polarity",
    "instrument_type": "mass-spectrometer family", "adduct": "normalised measured ion form",
    "adduct_orig": "ion form as supplied by the source", "precursor_mz": "selected precursor mass-to-charge ratio",
    "precursor_error_ppm": "precursor mass error in parts per million", "ms2_mzs": "fragment m/z array",
    "ms2_normalized_intensities": "base-peak-normalised fragment intensities", "num_peaks": "number of fragment peaks",
    "base_peak_intensity": "absolute intensity of the strongest peak", "collision_energy_ev": "collision energies mapped to eV",
    "collision_energy_orig": "source collision-energy text", "collision_energy_orig_units": "source energy unit",
    "molecule_id": "anonymous test-molecule group", "spectrum_id": "anonymous acquisition identifier",
}
FIELD_EXAMPLE = {"ingest_lib":"gnps", "normalized_smiles":"CC(=O)O", "inchikey14":"QTBSBXVTEAMEQO",
    "molecular_formula":"C2H4O2", "ionization_mode":"positive", "instrument_type":"timsTOF",
    "adduct":"[M+H]+", "precursor_mz":"375.0709", "precursor_error_ppm":"1.5",
    "num_peaks":"230", "collision_energy_ev":"[20, 40, 60]", "molecule_id":"m_005e53",
    "spectrum_id":"s_4d19d522"}

def schema_table(pf):
    schema = pf.schema_arrow
    return pd.DataFrame({"field": schema.names, "storage type": [str(x.type) for x in schema],
                         "experimental meaning": [FIELD_MEANING.get(x, "source metadata") for x in schema.names],
                         "example": [FIELD_EXAMPLE.get(x, "—") for x in schema.names]})

summary = pd.DataFrame([
    {"file": "train.parquet", "rows": train_pf.metadata.num_rows,
     "columns": train_pf.metadata.num_columns, "row_groups": train_pf.metadata.num_row_groups},
    {"file": "test.parquet", "rows": test_pf.metadata.num_rows,
     "columns": test_pf.metadata.num_columns, "row_groups": test_pf.metadata.num_row_groups},
])
display(Markdown("**Table 1. Physical layout of the competition files.**"))
display(summary.style.format({"rows": "{:,}"}))

display(Markdown("**Table 2. Field dictionary: storage, scientific meaning, and representative values.**"))
display(pd.concat({"train": schema_table(train_pf), "test": schema_table(test_pf)}, names=["file"]))

*Tables 1–2. File layout and field dictionary. Table 1 establishes dataset scale. Table 2 translates storage fields into experimental meaning and gives an illustrative value where useful; the examples are independent of each other and do not describe one spectrum record.*

**Read the schema before making joins.** Training carries molecular labels and source metadata; test carries anonymous `molecule_id` and `spectrum_id` values. Shared measurement fields use the same names, but the complete column sets differ. In particular, the training file has no test-style molecule ID, so its connectivity key is used for grouping.

The paired arrays `ms2_mzs` and `ms2_normalized_intensities` hold each spectrum. Their lengths must agree, and every element must remain paired during filtering or sorting. The remaining fields describe acquisition conditions, provenance, or labels; they are not all available as test-time features. Table 3 then checks the label counts and the integrity of the test peak arrays.


In [ ]:
TRAIN_SCALAR_COLS = [
    "ingest_lib", "normalized_smiles", "inchikey", "inchikey14", "molecular_formula",
    "ionization_mode", "instrument_type", "adduct", "adduct_orig", "precursor_mz",
    "precursor_error_ppm", "num_peaks", "base_peak_intensity",
    "collision_energy_orig", "collision_energy_orig_units",
]

# The two peak arrays dominate disk volume. Scalar metadata is enough for most EDA and keeps memory bounded.
train = pd.read_parquet(TRAIN_PATH, columns=TRAIN_SCALAR_COLS)
test = pd.read_parquet(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

assert test["spectrum_id"].is_unique
assert sample["molecule_id"].is_unique
assert set(test["molecule_id"]) == set(sample["molecule_id"])

test_peak_counts = test["ms2_mzs"].str.len()
alignment_ok = test_peak_counts.eq(test["ms2_normalized_intensities"].str.len())
normalisation_ok = test["ms2_normalized_intensities"].map(lambda x: len(x) > 0 and np.isclose(np.max(x), 1.0))
sorted_ok = test["ms2_mzs"].map(lambda x: np.all(np.diff(x) >= 0))

audit = pd.Series({
    "train spectra (rows)": len(train),
    "train unique SMILES labels": train["normalized_smiles"].nunique(),
    "train unique stored InChIKey14 groups": train["inchikey14"].nunique(),
    "train unique molecular formulae": train["molecular_formula"].nunique(),
    "visible-test spectra": len(test),
    "visible-test molecules": test["molecule_id"].nunique(),
    "sample-submission rows": len(sample),
    "test peak arrays aligned": int(alignment_ok.sum()),
    "test spectra base-peak normalised": int(normalisation_ok.sum()),
    "test m/z arrays sorted": int(sorted_ok.sum()),
}, name="value")
display(Markdown("**Table 3. Dataset integrity and label cardinality.**"))
display(audit.to_frame().style.format({"value": "{:,}"}))

*Table 3. File integrity and label cardinality. Counts distinguish spectra from training connectivities. Peak-array checks describe the mounted test file and should be rerun whenever that file changes.*


Section 2 showed that the scorer compares tautomer-standardised connectivity keys. The stored `inchikey14` column skips the tautomer step, so for some structures it differs from the key the scorer computes: the training file stores carvone as `ULDHMXUKGWMISQ`, while the scorer computes `YAHNGTQFNICUIJ`. Joins used for modelling and leakage-safe validation must therefore use the recomputed key; the stored field is still fine for the preliminary file counts above. [This notebook](https://www.kaggle.com/code/dariushafshar/casmi26-10-4-of-keys-change-0-48-cost-you) and [section 4 of the data-pitfalls notebook](https://www.kaggle.com/code/kitopl/casmi-2026-data-pitfalls-that-break-your-cv) measure how often the two keys disagree.

## 4. Visible test: molecules, spectra, adducts, and quality

The released visible test is a **format example**: its molecule groups, acquisition settings and peak arrays show what the model must accept, but there are no structure answers here. Molecule groups differ in how much evidence they provide. Repeated acquisitions can vary in collision energy (CE), adduct or ionisation polarity. They are complementary views of one target, not additional targets.

Figure 3 shows one group of spectra. Figure 4 counts acquisitions per `molecule_id` and shows the range of spectrum sizes; Table 4 summarises the same counts. A richer peak list offers more possible clues, although extra peaks can also reflect noise or co-isolation (other ions entering the precursor isolation window alongside the target).

In [ ]:
full_example_group = max((group for _, group in test.groupby("molecule_id", sort=False)), key=len)
example_group = full_example_group.head(3)
n_views = len(example_group)
fig, axes = plt.subplots(1, n_views + 1, figsize=(5 + 3.4 * n_views, 3.5),
                         gridspec_kw={"width_ratios":[.75] + [1] * n_views})
axes = np.atleast_1d(axes)
axes[0].axis("off")
axes[0].text(.5, .60, "one underlying\nmolecule", ha="center", va="center", fontsize=14, weight="bold",
             bbox=dict(boxstyle="circle,pad=.62", fc="#dcefe7", ec="#2a9d8f", lw=2))
axes[0].annotate(f"{n_views} of {len(full_example_group)} measured views shown", xy=(.50,.23), xytext=(.50,.08), ha="center", color="#46566b",
                 arrowprops=dict(arrowstyle="-|>", color="#2a9d8f", lw=1.6))
for ax, (_, row) in zip(axes[1:], example_group.iterrows()):
    mz = np.asarray(row["ms2_mzs"], dtype=float); intensity = np.asarray(row["ms2_normalized_intensities"], dtype=float)
    keep = np.argsort(intensity)[-45:]
    ax.vlines(mz[keep], 0, intensity[keep], color="#d1495b", lw=1.1)
    energy_values = np.atleast_1d(np.asarray(row["collision_energy_ev"], dtype=float))
    energy = ", ".join(f"{x:g}" for x in energy_values)
    ax.set(title=f"{row['adduct']} · CE {energy} eV", xlabel="fragment m/z", ylim=(0,1.05))
    ax.spines[["top","right"]].set_visible(False)
axes[1].set_ylabel("relative intensity")
fig.suptitle("Figure 3. One molecule can produce several complementary MS/MS spectra", fontsize=16, weight="bold")
plt.tight_layout()
plt.show()

*Figure 3. Multiple acquisitions assigned to one molecule. The underlying molecular identity is shared; the measured precursor ion and fragment pattern can change with adduct and acquisition conditions. The output is one ranked list for the whole group. Each view draws its 45 strongest peaks.*


In [ ]:
spectra_per_molecule = test.groupby("molecule_id").size()
test_summary = pd.DataFrame({
    "statistic": ["min", "median", "mean", "max"],
    "spectra per molecule": [spectra_per_molecule.min(), spectra_per_molecule.median(),
                              spectra_per_molecule.mean(), spectra_per_molecule.max()],
    "peak count": [test_peak_counts.min(), test_peak_counts.median(),
                   test_peak_counts.mean(), test_peak_counts.max()],
})
display(Markdown("**Table 4. Replicate and peak-count summary for the visible test.**"))
display(test_summary.style.format({"spectra per molecule": "{:.2f}", "peak count": "{:.2f}"}))

display(Markdown("**Figure 4. Replication and spectral-complexity profile of the visible test.**"))
fig, axes = plt.subplots(3, 1, figsize=(10.5, 13))
sns.histplot(spectra_per_molecule, discrete=True, ax=axes[0], color="#2a9d8f")
axes[0].set(title="Replicate spectra per molecule", xlabel="spectra", ylabel="molecules")
sns.histplot(test_peak_counts, bins=40, ax=axes[1], color="#457b9d")
axes[1].set(title="Fragment peaks per visible-test spectrum", xlabel="number of peaks")
sns.histplot(test["precursor_mz"], bins=35, ax=axes[2], color="#e76f51")
axes[2].set(title="Visible-test precursor m/z", xlabel="precursor m/z")
plt.tight_layout()
plt.show()

*Table 4 and Figure 4. Replication, peak counts, and precursor m/z in the mounted test file. The first panel counts separate acquisitions per molecule; the other panels describe individual spectra. These are sample observations, not fixed limits for inference. Figure 5 and Table 5 then show which adducts and polarities these acquisitions contain.*


In [ ]:
display(Markdown("**Figure 5. Adduct and polarity composition of the visible test.**"))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
test["adduct"].value_counts().sort_values().plot.barh(ax=axes[0], color="#577590")
axes[0].set(title="Visible-test adducts", xlabel="spectra", ylabel="adduct")
test["ionization_mode"].value_counts().plot.pie(ax=axes[1], autopct="%1.1f%%",
                                                colors=["#43aa8b", "#f9c74f"])
axes[1].set(title="Ionisation modes", ylabel="")
plt.tight_layout()
plt.show()

display(Markdown("**Table 5. Cross-tabulation of normalised adduct and ionisation polarity.**"))
display(pd.crosstab(test["adduct"], test["ionization_mode"], margins=True))

*Figure 5 and Table 5. Ion forms in the visible test. Adduct and polarity determine the equation used to recover neutral molecular mass, so they constrain candidate generation before spectral ranking.*

Check Table 3's integrity counts before interpreting the adduct and polarity distributions in Figure 5. A well-formed array can still contain uninformative or contaminated peaks; format checks and chemical quality checks serve different purposes.

The organisers describe hidden molecules spanning **157–1,159 Da** in neutral monoisotopic mass and **1–16 spectra per molecule**, acquired on timsTOF. These published ranges provide context; the visible precursor-m/z histogram is a different quantity measured on a demonstration sample. For `[M+H]+`, subtracting the proton mass from the measured m/z gives the neutral mass; sodium adducts and dimers need different equations (Section 7). Inference should process every supplied group and adduct without assuming the visible sample's limits. [Official data description](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data).

## 5. Training corpus: scale, provenance, and overlap

`ingest_lib` names source collections, not molecular classes. The training corpus includes experimental libraries such as [GNPS](https://ccms-ucsd.github.io/GNPSDocumentation/browselibraries/), [MassBank](https://github.com/MassBank/MassBank-data) and [MoNA](https://massbank.us/), Enveda acquisitions, and upstream labels such as RIKEN, MS-DIAL and `pluskal_ms2`. These collections overlap in chemistry but differ in instruments, processing and replication; Table 6 shows their actual labels and counts.

Table 6 asks two separate questions: which library contributes the most **spectra**, and which contributes the most **distinct stored InChIKey14 groups**? Their ratio measures replication. A large row count can mean broader chemistry, more repeated measurements, or both. Figure 6 shows the same counts as bars.

In [ ]:
library_stats = (train.groupby("ingest_lib")
                 .agg(spectra=("inchikey14", "size"), structures=("inchikey14", "nunique"))
                 .sort_values("spectra", ascending=False))
library_stats["spectra_per_structure"] = library_stats["spectra"] / library_stats["structures"]
LIBRARY_DESCRIPTION = {"enveda-180":"Enveda synthetic, instrument-matched collection",
 "pluskal_ms2":"Pluskal laboratory MS/MS compilation", "riken":"RIKEN metabolomics / natural products",
 "gnps":"Global Natural Products Social Molecular Networking", "massbank":"curated public high-resolution spectra",
 "mona":"MassBank of North America", "spectraverse":"community spectrum repository",
 "msdial":"MS-DIAL public library", "drug_plus":"drug-like reference set",
 "enveda-np-examples":"organiser natural-product examples", "masaryk":"Masaryk University spectra"}
library_stats.insert(0, "collection", [LIBRARY_DESCRIPTION.get(x, "source collection") for x in library_stats.index])
display(Markdown("**Table 6. Spectra, stored InChIKey14 groups, and replication by source library.**"))
display(library_stats.style.format({
    "spectra": "{:,}", "structures": "{:,}", "spectra_per_structure": "{:.1f}"
}).background_gradient(subset=["spectra", "structures"], cmap="Blues"))

display(Markdown("**Figure 6. Relative scale of the source libraries.**"))
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
library_stats.sort_values("spectra")["spectra"].plot.barh(ax=axes[0], color="#277da1")
axes[0].set(title="Spectra by source library", xlabel="rows", ylabel="")
library_stats.sort_values("structures")["structures"].plot.barh(ax=axes[1], color="#90be6d")
axes[1].set(title="Unique stored InChIKey14 groups by source", xlabel="structures", ylabel="")
for ax in axes:
    ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
plt.tight_layout()
plt.show()

*Table 6 and Figure 6. Training-library scale. `spectra` counts acquisitions; `structures` counts unique stored InChIKey14 groups; their ratio measures experimental replication. Table 7 and Figure 7 then look at repeated measurements and recurrence across libraries.*

In [ ]:
structure_repeats = train.groupby("inchikey14").size()
library_pairs = train[["inchikey14", "ingest_lib"]].drop_duplicates()
libraries_per_structure = library_pairs.groupby("inchikey14").size()

repeat_table = pd.DataFrame({
    "spectra_per_structure": structure_repeats.describe(percentiles=[.25, .5, .75, .9, .95, .99]),
})
display(Markdown("**Table 7. Spectra per stored InChIKey14 group (before scorer tautomer canonicalisation).**"))
display(repeat_table)

display(Markdown("**Figure 7. Replication depth and cross-library recurrence.**"))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.histplot(np.log10(structure_repeats), bins=50, ax=axes[0], color="#4d908e")
axes[0].set(title="Long-tailed spectral replication", xlabel="log10(spectra per stored InChIKey14)")
libraries_per_structure.value_counts().sort_index().plot.bar(ax=axes[1], color="#f9844a")
axes[1].set(title="Source libraries per stored structure key", xlabel="number of libraries", ylabel="stored keys")
plt.tight_layout()
plt.show()

multi_library = int((libraries_per_structure > 1).sum())
print(f"Stored InChIKey14 groups found in >1 source library: {multi_library:,} "
      f"({multi_library / len(libraries_per_structure):.2%})")

*Table 7 and Figure 7. Repeated measurements and cross-library overlap. A connectivity appearing in several sources may provide useful acquisition diversity, but provenance alone does not establish that all records are independent or correctly labelled.*


The long tail of spectra per connectivity explains why a random row split can be misleading. It may leave close replicas of a validation molecule in the search library. That is a useful test of **reference retrieval**, but an optimistic test of **generalisation to a molecule with no reference spectrum**.

In the measured corpus, `enveda-180` is the largest single source by spectrum count; it is not a majority of all rows. Instrument agreement also does not guarantee chemical-domain agreement. Holding out the smaller `enveda-np-examples` collection alone does not remove matching structures or spectra from other sources, and frozen model artifacts may already have seen them; Appendix A illustrates the limitation. For a structure-disjoint test, all spectra of each held-out connectivity must be removed from every library, not just from the named source; [section 6 of the data-pitfalls notebook](https://www.kaggle.com/code/kitopl/casmi-2026-data-pitfalls-that-break-your-cv) works through this case.


## 6. Missingness, label-quality signals, and outliers

Before filtering, distinguish missing metadata from suspicious measurements. A source can omit raw intensity while supplying a perfectly usable normalised spectrum. Conversely, a finite precursor mass can still conflict with the labelled structure and adduct.

Tables 8–10 and Figures 8–9 expose these cases. The decision to remove a record should depend on the task, the source, and measured validation impact, rather than on how unusual its value looks in a global histogram.


In [ ]:
missing = (train.isna().mean().mul(100).sort_values(ascending=False).rename("missing_%"))
display(Markdown("**Table 8. Missing metadata by field.**"))
display(missing.to_frame().style.format("{:.2f}"))

display(Markdown("**Figure 8. Fields with incomplete training metadata.**"))
fig, ax = plt.subplots(figsize=(10, 5))
missing[missing > 0].sort_values().plot.barh(ax=ax, color="#bc4749")
ax.set(title="Training metadata missingness", xlabel="missing rows (%)", ylabel="")
plt.tight_layout()
plt.show()

*Table 8 and Figure 8. Missing metadata by field. The pattern identifies which inputs need explicit missing-value handling; it does not by itself establish whether the associated spectra are poor quality.*


In [ ]:
numeric_summary = train[["precursor_mz", "precursor_error_ppm", "num_peaks", "base_peak_intensity"]].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]
).T
display(Markdown("**Table 9. Numerical metadata summary with robust quantiles.**"))
display(numeric_summary.style.format("{:.3g}"))

quality_counts = pd.Series({
    "|precursor error| > 5 ppm": train["precursor_error_ppm"].abs().gt(5).sum(),
    "|precursor error| > 20 ppm": train["precursor_error_ppm"].abs().gt(20).sum(),
    "precursor m/z > 2,000": train["precursor_mz"].gt(2000).sum(),
    "fewer than 5 peaks": train["num_peaks"].lt(5).sum(),
    "more than 2,000 peaks": train["num_peaks"].gt(2000).sum(),
}, name="rows")
display(Markdown("**Table 10. Explicit quality-control flags.**"))
display(quality_counts.to_frame().assign(percent=lambda x: 100*x["rows"]/len(train)).style.format({"rows": "{:,}", "percent": "{:.3f}%"}))

*Table 9 reports robust numerical ranges; Table 10 counts explicit quality flags. Together they separate the typical acquisition regime from records that deserve clipping, filtering, or dedicated validation.*

In [ ]:
display(Markdown("**Figure 9. Peak-count, precursor-error, and precursor-mass distributions.**"))
fig, axes = plt.subplots(3, 1, figsize=(10.5, 13))
sns.histplot(np.log10(train["num_peaks"]), bins=60, ax=axes[0], color="#577590")
axes[0].set(title="Peak-count tail", xlabel="log10(number of peaks)")

ppm_clip = train["precursor_error_ppm"].dropna().clip(-25, 25)
sns.histplot(ppm_clip, bins=80, ax=axes[1], color="#43aa8b")
axes[1].set(title="Precursor error (clipped for display)", xlabel="error (ppm)")

mass_cap = train["precursor_mz"].quantile(.995)
sns.histplot(train.loc[train["precursor_mz"] <= mass_cap, "precursor_mz"], bins=80, ax=axes[2], color="#f9844a")
axes[2].set(title="Precursor m/z (≤99.5th percentile)", xlabel="precursor m/z")
plt.tight_layout()
plt.show()

*Figure 9. Display transforms reveal the central distributions: peak counts are log10-transformed, precursor errors outside ±25 ppm are clipped to the boundary bins, and precursor m/z values above the 99.5th percentile are omitted from the bottom panel. These plotting choices do not filter the model inputs.*

Extreme values flag records for investigation; they do not by themselves establish the cause. Plausible explanations include unusual acquisition settings, inconsistent units, incorrect adducts, and erroneous labels. Inspect precursor-to-formula agreement and source-specific distributions before choosing thresholds.

`precursor_error_ppm` depends on the labelled structure and is available for reference-library quality control, not as a hidden-query label-derived feature. Likewise, missing `base_peak_intensity` can reflect how a source exported its spectra. A model can exploit these patterns as source shortcuts, so validate quality filters and source priors on held-out domains.


## 7. Instruments, collision energy, ion mode, and adduct shift

A good match must account for how the spectrum was acquired. Polarity and adduct affect ion chemistry; collision energy affects fragmentation; instrument and preprocessing affect which signals survive. Figure 10 shows how widely these conditions vary across the corpus, and Table 11 lists the most common adducts with their polarity. This motivates separate evidence channels and acquisition-aware scoring.


In [ ]:
display(Markdown("**Figure 10. Experimental domains in the training corpus.**"))
fig, axes = plt.subplots(3, 1, figsize=(10.5, 13))
train["ionization_mode"].value_counts().plot.bar(ax=axes[0], color=["#277da1", "#f9c74f"])
axes[0].set(title="Ionisation mode", xlabel="", ylabel="spectra")
train["collision_energy_orig_units"].value_counts().plot.bar(ax=axes[1], color="#90be6d")
axes[1].set(title="Original collision-energy units", xlabel="", ylabel="spectra")
train["instrument_type"].fillna("missing").value_counts().head(15).sort_values().plot.barh(ax=axes[2], color="#f3722c")
axes[2].set(title="Top 15 instrument labels", xlabel="spectra", ylabel="")
plt.tight_layout()
plt.show()

adduct_table = pd.crosstab(train["adduct"], train["ionization_mode"]).assign(total=lambda x: x.sum(axis=1))
display(Markdown("**Table 11. Most frequent adducts by ionisation polarity.**"))
display(adduct_table.sort_values("total", ascending=False).head(25).style.format("{:,}"))
print("Distinct free-text instrument labels:", train["instrument_type"].nunique(dropna=True))

*Figure 10 and Table 11. Acquisition heterogeneity. Polarity, energy units, instruments, and adducts form distinct experimental domains that a robust search score should recognise.*

eV (electron volt), NCE (normalised collision energy), and V (volt) are different quantities. Use the provided `collision_energy_ev` for a common input, while preserving its approximate nature and the original metadata. A list of energies can describe a merged spectrum; reducing it to one number loses that distinction.

Mass filtering must use the full adduct equation. For a singly protonated molecule, neutral mass is precursor m/z minus the proton mass. For a singly charged `[M+Na]+`, subtract the sodium-ion mass instead; for a dimer such as `[2M+H]+`, subtract the proton mass and divide the remainder by two. An incorrect equation can discard the correct structure before the ranker sees it. The [Fiehn lab adduct table](https://fiehnlab.ucdavis.edu/staff/kind/metabolomics/ms-adduct-calculator) lists the equation for each common adduct, and [a public notebook on mass windows](https://www.kaggle.com/code/dariushafshar/casmi26-your-ppm-window-drops-121-805-spectra) shows that a purely relative window also loses 121,805 training spectra whose precursor m/z is reported at coarse absolute precision or disagrees with its adduct label, and proposes a hybrid 10 ppm / 0.01 Da tolerance.

Normalise harmless instrument spelling differences, but do not collapse genuinely different acquisition regimes. Support the complete published adduct set rather than only the adducts observed in the demonstration file.


## 8. What an MS/MS spectrum looks like in these files

An MS/MS spectrum is a set of fragment mass-to-charge ratios with associated intensities. Each vertical line is one detected fragment ion; its horizontal position is fragment m/z and its height is abundance relative to the base peak. Figures 11 and 12 move from individual acquisitions to the repeated views available for one molecule. [Section 8 of A Complete Statistical Tour](https://www.kaggle.com/code/josefreitasalvesneto/casmi-2026-a-complete-statistical-tour) shows one spectrum before and after removing low-intensity noise peaks.

In [ ]:
def read_rows_by_position(path, positions, columns):
    # Read a few global Parquet row positions without materialising all peak arrays.
    pf = pq.ParquetFile(path)
    positions = np.asarray(sorted(set(map(int, positions))), dtype=np.int64)
    chunks = []
    start = 0
    for rg in range(pf.metadata.num_row_groups):
        n = pf.metadata.row_group(rg).num_rows
        mask = (positions >= start) & (positions < start + n)
        if mask.any():
            local = pa.array(positions[mask] - start)
            chunks.append(pf.read_row_group(rg, columns=columns).take(local))
        start += n
    if not chunks:
        return pd.DataFrame(columns=columns)
    return pa.concat_tables(chunks).to_pandas()


def plot_spectrum(ax, mzs, intensities, title, color="#264653", top_n=None):
    mzs = np.asarray(mzs, dtype=float)
    intensities = np.asarray(intensities, dtype=float)
    if top_n is not None and len(mzs) > top_n:
        keep = np.argpartition(intensities, -top_n)[-top_n:]
        mzs, intensities = mzs[keep], intensities[keep]
        order = np.argsort(mzs)
        mzs, intensities = mzs[order], intensities[order]
    ax.vlines(mzs, 0, intensities, lw=0.8, color=color)
    ax.set(title=title, xlabel="fragment m/z", ylabel="relative intensity")
    ax.set_ylim(0, 1.05)


display(Markdown("**Figure 11. MS/MS spectra with low, median and high peak counts.**"))

# Pick three visible-test spectra spanning low/median/high peak counts.
q_targets = test_peak_counts.quantile([.1, .5, .9]).to_numpy()
chosen = []
for target in q_targets:
    idx = (test_peak_counts - target).abs().idxmin()
    if idx not in chosen:
        chosen.append(idx)

fig, axes = plt.subplots(len(chosen), 1, figsize=(10.5, 3.2 * len(chosen)))
axes = np.atleast_1d(axes)
for ax, idx in zip(axes, chosen):
    row = test.loc[idx]
    plot_spectrum(ax, row["ms2_mzs"], row["ms2_normalized_intensities"],
                  f"{row['spectrum_id']} · {row['adduct']} · precursor {row['precursor_mz']:.4f} · "
                  f"{len(row['ms2_mzs'])} peaks", top_n=500)
plt.tight_layout()
plt.show()

*Figure 11. Three measured spectra spanning the visible sample's peak-count distribution. Compare fragment density and m/z coverage; peak richness describes an acquisition, not identification confidence or molecular complexity. At most the 500 strongest peaks of a spectrum are drawn.*

In [ ]:
display(Markdown("**Figure 12. Multiple collision-energy and adduct views of one molecule.**"))

# Multiple spectra of one molecule: evidence changes with adduct and collision energy.
example_id = spectra_per_molecule.sort_values(ascending=False).index[0]
example = test[test["molecule_id"].eq(example_id)].head(6)
fig, axes = plt.subplots(len(example), 1, figsize=(10.5, 2.7 * len(example)), sharex=True)
axes = np.atleast_1d(axes)
for ax, (_, row) in zip(axes, example.iterrows()):
    energy_unit = (row['collision_energy_orig_units']
                   if pd.notna(row['collision_energy_orig_units']) else 'unit not recorded')
    plot_spectrum(ax, row["ms2_mzs"], row["ms2_normalized_intensities"],
                  f"{example_id} · {row['adduct']} · CE {row['collision_energy_orig']} {energy_unit}", top_n=350)
plt.tight_layout()
plt.show()

*Figure 12. Repeated MS/MS views of one molecule. The panel titles report each acquisition's adduct and collision setting; their fragment patterns can differ, so molecule-level inference should fuse these views rather than select one arbitrarily. At most the 350 strongest peaks of a view are drawn.*

Several spectra of one molecule are complementary views, not independent prediction targets. Section 10 combines their evidence into one candidate list per `molecule_id` and explains how reference, analogue and fingerprint signals enter the score.

## 9. Molecular labels: structures, descriptors, and chemical space

The spectra describe the measurement; the labels show which molecular graphs are represented. Molecular drawings, descriptors, and fingerprints help connect a source library to its region of chemical space.

The key modelling question is whether our reference molecules resemble the compounds we hope to identify. A model can transfer poorly even when train and test use the same instrument, if their underlying chemistry differs. Figure 13 starts with one labelled structure from each source library.


In [ ]:
display(Markdown("**Figure 13. One example molecular structure per source library.**"))

# Draw deterministic representatives from several source libraries.
representatives = (train.query("150 <= precursor_mz <= 700 and 8 <= num_peaks <= 500")
                   .drop_duplicates(["ingest_lib", "inchikey14"])
                   .groupby("ingest_lib", group_keys=False)
                   .sample(n=1, random_state=SEED))
mols, legends = [], []
for _, row in representatives.iterrows():
    mol = Chem.MolFromSmiles(row["normalized_smiles"])
    if mol is not None:
        mols.append(mol)
        legends.append(f"{row['ingest_lib']}\n{row['molecular_formula']} · {row['inchikey14']}")

display(Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(340, 250), legends=legends, useSVG=True))

*Figure 13. One selected structure per source library, not a summary of the library's chemical domain. The bounded descriptor sample in Table 12/Figure 14 and source-aware projection in Figure 15 provide broader context; this gallery shows individual molecular graphs.*

In [ ]:
# Descriptor analysis on a fixed, bounded sample of unique stored InChIKey14 groups.
unique_structures = train.drop_duplicates("inchikey14").copy()
n_descriptor = min(20_000, len(unique_structures))
descriptor_sample = unique_structures.sample(n=n_descriptor, random_state=SEED)

records = []
invalid = 0
for _, row in descriptor_sample.iterrows():
    mol = Chem.MolFromSmiles(row["normalized_smiles"])
    if mol is None:
        invalid += 1
        continue
    records.append({
        "ingest_lib": row["ingest_lib"],
        "inchikey14": row["inchikey14"],
        "mol": mol,
        "MolWt": Descriptors.MolWt(mol),
        "cLogP": Descriptors.MolLogP(mol),
        "heavy_atoms": rdMolDescriptors.CalcNumHeavyAtoms(mol),
        "rings": rdMolDescriptors.CalcNumRings(mol),
        "rotatable_bonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
    })
desc = pd.DataFrame(records)
display(Markdown(f"**Table 12. Molecular-descriptor summary for {len(desc):,} sampled connectivities.**"))
print(f"Descriptor sample: {len(desc):,} valid molecules; {invalid:,} parse failures")
display(desc.drop(columns="mol").describe().T.style.format("{:.2f}"))

display(Markdown("**Figure 14. Molecular-property distributions.**"))
fig, axes = plt.subplots(3, 2, figsize=(10.5, 12))
for ax, col, color in zip(axes.flat,
                          ["MolWt", "cLogP", "heavy_atoms", "rings", "rotatable_bonds", "TPSA"],
                          sns.color_palette("viridis", 6)):
    cap = desc[col].quantile(.995)
    sns.histplot(desc.loc[desc[col] <= cap, col], bins=45, ax=ax, color=color)
    ax.set_title(f"{col} (≤99.5th percentile)")
plt.tight_layout()
plt.show()

*Table 12 and Figure 14. Molecular-descriptor summary and distributions. The sampled training structures are described by molecular weight (g/mol), predicted lipophilicity (cLogP, the preference for oil-like over aqueous surroundings), heavy-atom (non-hydrogen) count, ring count, rotatable-bond count and topological polar surface area (TPSA, Å², a structure-derived estimate of the polar surface). The values describe sampled training structures, not the complete candidate pool.*

In [ ]:
display(Markdown("**Figure 15. Source-aware projection of molecular substructure space.**"))

# Morgan fingerprints (circular substructure fingerprints) projected only for visual EDA.
n_fp = min(6_000, len(desc))
fp_sample = desc.sample(n=n_fp, random_state=SEED).reset_index(drop=True)
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=512)
X = np.zeros((n_fp, 512), dtype=np.uint8)
for i, mol in enumerate(fp_sample["mol"]):
    DataStructs.ConvertToNumpyArray(generator.GetFingerprint(mol), X[i])
projection = TruncatedSVD(n_components=2, random_state=SEED).fit_transform(X)
fp_sample[["SVD1", "SVD2"]] = projection

top_libs = library_stats.head(7).index
fp_sample["source"] = fp_sample["ingest_lib"].where(fp_sample["ingest_lib"].isin(top_libs), "other")
fig, ax = plt.subplots(figsize=(12, 7))
sns.scatterplot(data=fp_sample, x="SVD1", y="SVD2", hue="source", s=13, alpha=.55, linewidth=0, ax=ax)
ax.set_title("Approximate training chemical space (Morgan fingerprint projection)")
ax.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

*Figure 15. A two-dimensional projection of sampled Morgan fingerprints, which encode local molecular substructures, using truncated singular value decomposition (SVD). Each point takes the colour of the source of its first retained training record; the same connectivity can also occur in other sources. Overlap and separation suggest questions about chemical coverage; they are not quantitative estimates of hidden-test similarity.*


A fingerprint records local molecular substructures. Its projection makes broad patterns visible, but compressing many dimensions into two distorts distances and can create apparent clusters. Source labels may also overlap for the same connectivity.

Use this figure to formulate validation questions: do errors concentrate in sparsely represented chemistry, do source-held-out scores fall, and does a natural-product candidate database add missing answers? Answer those questions with held-out molecules and candidate-coverage measurements, not the appearance of the plot alone.


### Natural products: what is known and what has been measured

The hidden test comes from biological extracts, so most of its answers are expected to be natural products: compounds made by living organisms. The [natural-product knowledge table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table) combines LOTUS and NPAtlas into 152,717 entries, one per InChIKey connectivity block. Where LOTUS supplies it, an entry carries its NPClassifier pathway, a broad biosynthetic grouping such as terpenoids or alkaloids. Figure 16 compares the size of each pathway group with the number of its entries that have at least one training spectrum.

In [ ]:
# Figure 16: which natural-product classes are known, and which of them have a training spectrum?
import glob, json

def _first(patterns):
    for pattern in patterns:
        hits = sorted(glob.glob(pattern, recursive=True), key=len)
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(patterns[0])

np_table = pd.read_parquet(
    _first(['/kaggle/input/**/casmi26-natural-product-knowledge-table/**/natural_products.parquet',
            os.environ.get('CASMI_NP_TABLE', 'natural_products.parquet')]),
    columns=['np_pathway', 'in_casmi_train', 'in_casmi_pool'])
by_class = (np_table.dropna(subset=['np_pathway']).groupby('np_pathway')
            .agg(known=('in_casmi_train', 'size'), with_spectrum=('in_casmi_train', 'sum')).sort_values('known'))
by_class['share'] = by_class.with_spectrum / by_class.known
display(Markdown('**Figure 16. Known natural products by biosynthetic class, and the part that has a training spectrum.**'))
fig, ax = plt.subplots(figsize=(10.5, 5.2))
ax.barh(by_class.index, by_class.known / 1e3, color='#cfe3e6', label='known structures (LOTUS and NPAtlas)')
ax.barh(by_class.index, by_class.with_spectrum / 1e3, color='#287a8c', label='with a training spectrum')
for y, (known, share) in enumerate(zip(by_class.known, by_class.share)):
    ax.text(known / 1e3 + by_class.known.max() / 1e3 * .012, y, f'{share:.0%} measured', va='center', fontsize=11)
ax.set(xlabel='distinct connectivities (thousands)', xlim=(0, by_class.known.max() / 1e3 * 1.22))
ax.grid(axis='y', visible=False); ax.legend(frameon=False, loc='lower right')
plt.tight_layout(); plt.show()
print(f"{len(np_table):,} known natural products: {np_table.in_casmi_train.mean():.1%} have a training spectrum and "
      f"{np_table.in_casmi_pool.mean():.1%} are in the candidate pool; {int(np_table.np_pathway.isna().sum()):,} lack a pathway annotation.")
del np_table


*Figure 16. Reference-spectrum coverage within the LOTUS–NPAtlas catalogue, by NPClassifier pathway. Light bars count catalogue entries; dark bars count entries with a training spectrum. The 11,101 entries without a pathway annotation are left out of the bars. Across the whole table, 8.2% of the entries have a training spectrum and 98.2% occur in the candidate pool: the catalogue usually supplies a candidate without supplying a reference spectrum, so identification depends on evidence other than a matching reference. These percentages describe the catalogue, not the novelty mixture of the hidden test.*

## 10. The model: a candidate list with several kinds of evidence

**In short.** The model works like a search engine with several reviewers:

1. **Find candidates.** Take every structure within 10 ppm of the neutral mass from a 710,701-structure pool of training structures and COCONUT natural products, and generate a few more by modifying close spectral matches.
2. **Score them.** A transformer (FPNet) predicts which substructures the spectrum implies; four LightGBM models combine this with fragment, formula and library-match features.
3. **Look further.** A separate search over 105.9 million PubChem structures proposes candidates that the pool lacks.
4. **Prefer documented compounds.** A small bonus goes to structures with many PubChem records and articles.
5. **Check isomers with predicted spectra.** For suitable positive-mode spectra, ICEBERG and GLACIER predict the spectrum of a shortlist of same-formula candidates and compare it with the measurement.
6. **Ask for a second opinion.** When no strong library match exists, the list is fused with the ranking of an independently built engine.

The rest of this section gives the exact rules; Figure 17 draws the flow.

A tandem mass spectrum records fragment ions, not a molecular graph. The precursor's measured mass and its adduct (for example, the added proton in `[M+H]+`) give an estimate of the neutral molecule's mass; when a molecule has several spectra, both channels search around the median of these estimates. A search window of **10 parts per million (ppm)** allows a mass difference of roughly 0.0037 Da at 370 Da; the engine widens it to 30 ppm only when the narrow window has no candidates. This removes most impossible structures, but leaves isomers that share a formula and mass.

The main pool has **710,701 structure rows**: 274,186 linked to training structures and 436,515 from [COCONUT](https://coconut.naturalproducts.net/), an open natural-products collection. The competition training file separately provides **measured reference spectra**. A database structure is a proposal; it becomes a measured reference only if the training file actually contains its spectrum. The engine can also derive a new mass-compatible structure from a close spectral analogue. That generation step matters when the answer is absent from the starting pool: a ranker cannot recover a structure it never receives.

The engine compares a query with measured spectra, including near matches that differ in precursor mass. **FPNet**, a transformer over the peak list, predicts molecular fingerprint bits from up to 16 spectra and their polarity, adduct, precursor and collision-energy fields. A fingerprint bit indicates whether a particular molecular substructure is likely present. Fragment features compare the observed peaks with fragments of each candidate; subformula features test whether a peak's elemental composition could be drawn from the candidate's formula. Further feature families describe how a candidate was generated, its relation to measured analogues, the agreement among fingerprint predictions and predicted fragment evidence. Four pretrained **LightGBM models**, each an ensemble of boosted trees, combine 160 selected features into a candidate score. These weights are supplied with the source package; the notebook uses them for inference.

A separate [PubChem](https://pubchem.ncbi.nlm.nih.gov/) search looks beyond that pool. Its FPNet A+B models first screen the database rows of a 10-ppm window, then fully score up to 5,000 fingerprint-selected rows, remove repeated identities and retain up to 25 connectivity keys that are new to the main pool. For suitable positive-mode spectra, **ICEBERG** ([Goldman et al., 2024](https://doi.org/10.1021/acs.analchem.3c04654)) predicts fragments for a bounded shortlist of same-formula isomers and compares its predicted spectrum with the measured peaks; it can reorder candidates within a formula group. **GLACIER** supplies a second predicted spectrum for the same candidates and covered acquisitions. Within each eligible formula group, the score is `z(channel score) + 0.5 · z(ICEBERG) + 0.5 · z(GLACIER)`; GLACIER scores only `[M+H]+` acquisitions, so a molecule measured only as other adducts receives no GLACIER term (following [wangpenghua's v27](https://www.kaggle.com/code/wangpenghua/casmi26-v27-gl-mh) and [v29](https://www.kaggle.com/code/wangpenghua/casmi26-v29-fusion-pop-glmh)). The channel score already includes the popularity term described below. Here, `z` centres each score and divides it by its sample standard deviation within the group. Missing or constant scores contribute zero; ties keep the original order. Each formula group keeps its original list positions. GLACIER is applied where the ICEBERG branch is active; otherwise, the ICEBERG ordering is retained. These forward models reorder existing candidates; they do not expand the structure pool.

**Popularity prior.** Metabolomics identification has long used the fact that the compounds people measure are usually compounds people have already studied. [MetFrag](https://doi.org/10.1186/s13321-016-0115-9) combines fragmentation evidence with literature and patent counts. [CASMI 2016](https://pmc.ncbi.nlm.nih.gov/articles/PMC5368104/) reported higher identification rates when methods could use additional metadata. [PubChemLite](https://doi.org/10.1186/s13321-021-00489-0) selects compounds using annotation categories relevant to environmental and biological exposure. Here the prior is `pop = log(1 + substance records) + log(1 + PubMed articles)`. The PubChem shortlist is ranked by the fingerprint score alone, as in the [public v4m notebook](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4m-inference). On the configuration of the [public v29 notebook](https://www.kaggle.com/code/wangpenghua/casmi26-v29-fusion-pop-glmh), the same pipeline scored 0.405 without a PubChem-channel prior ([analyticaobscura](https://www.kaggle.com/code/analyticaobscura/enveda-casmi-2026)), 0.415 with private popularity arrays and 0.393 with ours (our version 22), so the prior is used in the main list only. Within the main ranker's top 60 distinct candidates, pool structures are reordered by `z(ranker) + 0.15 · pop`; generated analogues receive no popularity lookup and retain their slots during this step. The standardisation runs over the retained main list and uses the population standard deviation. The main-list weight comes from the [public v21 notebook](https://www.kaggle.com/code/wangpenghua/casmi26-v21-v4i-pop), which also used 0.25 for the PubChem shortlist; version 26 tested the main-list weight of 0.25 used by the [public notebook of huseyinemreaksoy](https://www.kaggle.com/code/huseyinemreaksoy/casmi26-sub-v4b-0-409-on-the-public) and did not keep it. Popularity adds a preference for documented compounds to the spectral score. Its effect depends on the score differences and can change more than near-ties.

**Merge procedure.** The merge looks at two quantities. `lib_max` is the strongest measured-library match. `Δ` is the best raw FPNet score among eligible non-pool PubChem candidates minus the best raw FPNet score in the pool window; the popularity bonus is excluded from Δ, although adding candidates to the PubChem shortlist can change its raw maximum. These are model scores, not probabilities. The steps run in this order; positions are one-based and refer to the merged list before the fusion.

| Step | Rule |
|---|---|
| 1. Keep the main list when it is trusted | With `lib_max ≥ 0.9` the main list is final: a strong measured match takes priority. Without PubChem proposals and with a weaker match, steps 2 and 3 are skipped and the fusion of step 4 still runs. |
| 2. Choose insertion positions | Positions 2, 4, 6, 8 and 10 when `Δ > 600` or when no finite pool score is available; positions 4, 8, 12, 16 and 20 otherwise. |
| 3. Merge | PubChem proposals enter at those positions in their forward-model order; keys already in the main list are skipped. If the main list runs out, the remaining proposals fill the list. Up to 25 candidates are kept. |
| 4. Fuse with the second engine | The merged list and the second engine's list are combined by reciprocal-rank fusion and re-ordered inside same-formula groups, as described below; the first 25 distinct structures are written. |

The merge preserves the relative order of the remaining main candidates and removes duplicate connectivity keys. The slot positions and both thresholds come from the [public v4g inference method](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4g-inference). The five source packages and their separate terms are documented in [the attributed asset fork](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets).

**A second engine and rank fusion.** Two independently built rankers make different mistakes, and a structure that both place near the top is more likely to be right. The second engine is the [two-ranker pipeline](https://www.kaggle.com/code/megayak/casmi26-two-rankers-one-engine) built on [prvsiyan's analog-propagation baseline](https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline), in the form packaged by the [public fusion notebook of seyitkaangunes](https://www.kaggle.com/code/seyitkaangunes/casmi26-v4n-engine-fusion-union-lb-0-399). It searches training, COCONUT, ChEBI and LIPID MAPS structures with its own spectrum encoders, propagates structural evidence from spectral analogues, and ranks candidates with two ensembles of gradient-boosted trees fitted at run time on supplied feature rows. It returns up to 40 structures per molecule. For a molecule without a strong library match, its list is combined with the merged list of the procedure above by weighted reciprocal-rank fusion ([Cormack et al., 2009](https://doi.org/10.1145/1571941.1572114)): a structure at rank `r` in the merged list and rank `e` in the engine's list receives `1/(3 + r) + 0.6/(3 + e)`, and a list that lacks the structure contributes nothing. Agreement moves a structure up; in the fused order, before the forward re-ordering, a structure proposed by the second engine alone cannot stand above rank 4 while the merged list has three candidates. The 40 best fused structures are then re-ordered once more inside same-formula groups by `z(fusion score) + 0.5 · z(ICEBERG) + 0.5 · z(GLACIER)`; for this purpose the second engine's candidates are added to the input of the forward models. The fusion weights and the procedure come from that notebook. Here the fusion is limited to molecules with `lib_max < 0.9`, and the scorer keys of the fused list are recomputed before writing.

**Forward-model weights.** ICEBERG and GLACIER enter all three re-orderings (main list, PubChem list and fused list) with weight 0.5; version 27 tested 0.25 for the fused list and did not keep it. Several public notebooks use weight 1, but at that weight the two predicted spectra can overrule a strong library match: the fusion score keeps the order of the candidates but not how confident the library match was. Every spectrum of the visible format example is a copy of a training spectrum, so its answers are known. There, fusing every molecule with unit forward weights pushed the correct first candidate down for 21 of the 400 molecules and fixed none (MRR@25 0.969 instead of 0.998 with the settings used here); unit weights in the main list alone cost three of them. All visible molecules have reference spectra, so this test says nothing about molecules without one, where public notebooks with unit weights score well.

The GLACIER integration follows the [public v4h notebook](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4h-inference). Its [public model package](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-glacier) includes the checkpoint, source, compatibility shims, a manifest and source-license notices. The model was trained on MassSpecGym; complete independence from CASMI training structures is not established.

Figure 17 summarises how the two candidate channels, the popularity term, the forward models, the merge rule and the second engine connect.


In [ ]:
# Figure 17: the ranking method, including the separate proposal branch.
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(10.5, 9))
ax.set(xlim=(0,10), ylim=(0,10)); ax.axis('off')
boxes = [
 (0.5,7.9,9,1.1,'Measured MS/MS + precursor + adduct'),
 (0.5,5.7,5.4,1.35,'Main candidates\nTrain / COCONUT + analogue generation'),
 (6.5,5.7,3,1.35,'PubChem candidates\nFPNet A+B'),
 (0.5,3.5,5.4,1.35,'Spectral + fingerprint + fragment evidence\nFour LightGBM models + popularity'),
 (0.5,1.4,9,1.3,'Same-formula ordering: ICEBERG + GLACIER\nGated merge, then fusion with a second engine → up to 25 structures'),
]
for x,y,w,h,label in boxes:
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=.12',fc='#e6f1f2',ec='#287a8c',lw=1.5))
    ax.text(x+w/2,y+h/2,label,ha='center',va='center',fontsize=12,color='#16324f')
for a,b in [((3.2,7.8),(3.2,7.15)),((8,7.8),(8,7.15)),((3.2,5.55),(3.2,4.95)),((3.2,3.35),(3.2,2.8)),((8,5.55),(8,2.8))]:
    ax.annotate('',xy=b,xytext=a,arrowprops=dict(arrowstyle='-|>',lw=1.8,color='#287a8c'))
ax.set_title('Figure 17. Two candidate channels, one ranked answer',fontsize=16,pad=10)
plt.tight_layout(); plt.show()


*Figure 17. Main-pool candidates receive learned spectral and structural scores; a separate PubChem channel proposes additional structures. A popularity term adjusts the scores of the main channel, ICEBERG and GLACIER reorder eligible same-formula groups, and a gate merges the lists. Without a strong library match, the merged list is then fused with a second engine's ranking. The main channel also proposes new structures by modifying spectral analogues.*

### What PubChem documentation tells us

Reference libraries and natural-product catalogues preferentially contain compounds that have already been studied. PubChem substance records and linked articles provide a measurable description of that documentation. Figure 18 measures that trail for the three structure populations the search uses: structures linked to training spectra, COCONUT natural products without a training spectrum, and a random sample of the 105.9-million-row PubChem tier. The hidden test comes from biological extracts, which makes the COCONUT column the most relevant comparison, although its documentation profile need not match the hidden test. Table 13 summarises the counts shown in Figure 18.

In [ ]:
# Figure 18: how well documented are the structures in each candidate population?
import glob

def _first(patterns):
    for pattern in patterns:
        hits = sorted(glob.glob(pattern, recursive=True), key=len)
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(patterns[0])

POP_FIG_DIR = _first(['/kaggle/input/**/casmi26-pubchem-popularity-prior/**/pool_lsid.npy',
                      os.environ.get('CASMI_POP_DIR', '.') + '/pool_lsid.npy']).parent
pool_meta_path = _first(['/kaggle/input/**/v2_pool__pool_meta.parquet', '/kaggle/input/**/v2_pool/pool_meta.parquet',
                         os.environ.get('CASMI_POOL_META', 'pool_meta.parquet')])
pool_src = pd.read_parquet(pool_meta_path, columns=['src']).src.values
pool_pop = (np.load(POP_FIG_DIR / 'pool_lsid.npy').astype(np.float32)
            + np.load(POP_FIG_DIR / 'pool_lpmid.npy').astype(np.float32))
pool_pmid = np.load(POP_FIG_DIR / 'pool_lpmid.npy').astype(np.float32)
tier_lsid = np.load(POP_FIG_DIR / 'pc_lsid.npy', mmap_mode='r')
tier_lpmid = np.load(POP_FIG_DIR / 'pc_lpmid.npy', mmap_mode='r')
tier_rows = np.sort(rng.choice(len(tier_lsid), size=500_000, replace=False))
tier_pop = tier_lsid[tier_rows].astype(np.float32) + tier_lpmid[tier_rows].astype(np.float32)
tier_pmid = tier_lpmid[tier_rows].astype(np.float32)

groups = {
    'Training-spectrum structures': (pool_pop[pool_src != 2], pool_pmid[pool_src != 2]),
    'COCONUT natural products': (pool_pop[pool_src == 2], pool_pmid[pool_src == 2]),
    'PubChem tier (random 500k)': (tier_pop, tier_pmid),
}
pop_table = pd.DataFrame({
    name: {'structures': f'{len(p):,}', 'median pop': f'{np.median(p):.2f}',
           'nonzero documentation counts': f'{(p > 0).mean():.1%}', 'with ≥1 PubMed article': f'{(m > 0).mean():.1%}'}
    for name, (p, m) in groups.items()}).T
display(Markdown('**Table 13. PubChem documentation by candidate population.**'))
display(pop_table)

display(Markdown('**Figure 18. Known and natural-product chemistry is far better documented than a typical PubChem structure.**'))
colors = ['#287a8c', '#6a994e', '#b74759']
fig, axes = plt.subplots(2, 1, figsize=(10.5, 9.5), gridspec_kw={'height_ratios': [1.35, 1]})
grid_x = np.linspace(0, 8, 161)
for (name, (p, _)), color in zip(groups.items(), colors):
    ps = np.sort(p)
    share_above = 1 - np.searchsorted(ps, grid_x, side='left') / len(ps)
    axes[0].plot(grid_x, share_above * 100, lw=2.4, color=color, label=name)
axes[0].set(xlabel='pop = log(1 + substance records) + log(1 + PubMed articles)',
            ylabel='structures with pop ≥ x (%)', title='A. How much documentation each population carries',
            xlim=(0, 8), ylim=(0, 102))
axes[0].legend(frameon=False)
share = [(m > 0).mean() * 100 for _, m in groups.values()]
axes[1].barh(list(groups), share, color=colors)
for y, v in enumerate(share):
    axes[1].text(v + max(share) * .02, y, f'{v:.1f}%', va='center', fontsize=11)
axes[1].set(xlabel='structures with at least one PubMed article (%)', xlim=(0, max(share) * 1.25),
            title='B. Literature coverage')
axes[1].invert_yaxis(); axes[1].grid(axis='y', visible=False)
plt.tight_layout(); plt.show()
del tier_lsid, tier_lpmid


*Table 13 and Figure 18. PubChem documentation of three candidate populations, with `pop = log(1 + substance records) + log(1 + PubMed articles)`. Linked PubMed articles occur for 13.2% of training-spectrum structures, 22.3% of COCONUT structures and 3.5% of the sampled PubChem rows. This enrichment motivates a documentation prior; the figure measures database composition, and Section 11 measures its effect on a development panel of documented natural products. Rows with zero stored counts receive no bonus. The prior favours documented compounds and may disadvantage rarely studied natural products.*

## 11. Validation: separate coverage, ordering and execution

Ask three questions in order. **Coverage:** is the answer's scorer-canonical connectivity present in the actual mass-window or generated candidates? **Ordering:** when it is present, how high does it rank? **Full-list accuracy:** over every held-out molecule, including those whose answer never appears, how often is the answer ranked first (top-1) or among the first 25 guesses (top-25), and what is MRR@25? A coverage miss contributes zero MRR regardless of how good the ranking model is. Looking at coverage and ordering separately tells us whether to improve candidate generation or scoring.

The supplied pretrained models come without a documented held-out split, so the benchmark at the end of this section is a development proxy, not a clean estimate. Section 12 only checks that the run works and writes valid lists; the leaderboard measures the hidden test, and the visible `test.parquet` has no answer structures.

A clean local test puts **all** spectra of one identity on one side of the split, refits every learned component without the held-out identities (named external catalogues stay unchanged), chooses settings on inner folds and reports candidate presence and MRR@25 on an outer fold that no choice has seen; Section 13 lists the steps. Only such a test can tell whether a proposed improvement finds new structures or merely reorders familiar ones.

### Keeping a molecule on one side of the split

A molecule in the training file has nine spectra on average, and some structures are stored under an `inchikey14` that differs from the key the scorer computes. The [identity-disjoint folds](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-identity-disjoint-validation-folds) assign every spectrum of one scored identity to one fold, and a second variant also keeps each Bemis–Murcko scaffold (the rings and the linkers between them, with side chains removed) in one fold; molecules without a ring keep their identity grouping. Records count as one identity when they share either the scorer key or the stored key. Figure 19 holds out approximately one fifth of the spectra in four ways and measures what the training side already knows about the held-out identity groups.

In [ ]:
# Figure 19: how much of a validation set is already known to the training side under four ways of splitting?
folds_path = _first(['/kaggle/input/**/casmi26-identity-disjoint-validation-folds/**/structures.parquet',
                     os.environ.get('CASMI_FOLDS', 'structures.parquet')])
fold_tab = (pd.read_parquet(folds_path, columns=['inchikey14', 'identity_group', 'scaffold_group', 'fold_identity', 'fold_scaffold'])
            .drop_duplicates('inchikey14').set_index('inchikey14'))
_row = fold_tab.reindex(train.inchikey14.values)
_known = _row.identity_group.notna().values
ident = _row.identity_group.values[_known].astype(np.int64)
scaf = _row.scaffold_group.values[_known].astype(np.int64)
stored = train.inchikey14.values[_known]
split_rng = np.random.default_rng(SEED)
stored_keys = pd.unique(stored)
held_keys = set(split_rng.choice(stored_keys, len(stored_keys) // 5, replace=False))
splits = {
    'random spectra': split_rng.random(len(ident)) < 0.2,
    'stored inchikey14': pd.Series(stored).isin(held_keys).values,
    'scored identity (fold 0)': _row.fold_identity.values[_known] == 0,
    'scaffold group (fold 0)': _row.fold_scaffold.values[_known] == 0,
}
leak_rows = []
for name, val in splits.items():
    val_ident = np.unique(ident[val])
    seen_identity = np.isin(val_ident, np.unique(ident[~val])).mean()
    val_scaf = pd.Series(scaf[val], index=ident[val]).groupby(level=0).first()
    seen_scaffold = np.isin(val_scaf.values, np.unique(scaf[~val])).mean()
    leak_rows.append({'split': name, 'held-out spectra (%)': 100 * val.mean(), 'held-out identity groups': len(val_ident),
                      'same identity in training (%)': 100 * seen_identity, 'same scaffold group in training (%)': 100 * seen_scaffold})
leak = pd.DataFrame(leak_rows).set_index('split')
display(Markdown('**Figure 19. What the training side already knows about a validation set.**'))
fig, axes = plt.subplots(2, 1, figsize=(10.5, 7.8))
for ax, col, color, title in ((axes[0], 'same identity in training (%)', '#b74759', 'A. Held-out identity groups that also have spectra in training'),
                              (axes[1], 'same scaffold group in training (%)', '#287a8c', 'B. Held-out identity groups whose scaffold group occurs in training')):
    ax.barh(leak.index, leak[col], color=color)
    for y, v in enumerate(leak[col]):
        ax.text(v + 1.2, y, f'{v:.1f}%', va='center', fontsize=11)
    ax.set(xlim=(0, 112), xlabel='share of held-out identity groups (%)', title=title)
    ax.invert_yaxis(); ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()
del _row, ident, scaf, stored, splits


*Figure 19. Overlap under four ways of holding out approximately one fifth of the training spectra. With a random split of spectra, 97.7% of the held-out identity groups also have spectra on the training side, so the score measures library lookup. Splitting by the stored key reduces this to 0.9%, a residue of tautomer aliases; the scored-identity folds and the scaffold folds reduce it to zero. Identity folds still share a scaffold group with training for 58.2% of the held-out identities, which the scaffold folds also remove: a stricter separation, although close analogues can still have different scaffolds. The figure checks the separation of the records; a clean model evaluation also requires refitting every learned component without the held-out identities.*

### A development benchmark: 250 natural products with known answers

The training file contains a small library of natural products measured on the instrument family of the hidden test, `enveda-np-examples`. The [natural-product panel benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark) stores, for each of its 250 molecules, the cached main-ranker and PubChem-channel candidates with their scores in three simulated situations: the molecule's own panel spectra are hidden while matching spectra in other libraries stay available; the structure is in the pool and has no reference spectrum; the structure is removed from the pool, so that only the PubChem channel or the analogue generator can find it. Changes to shortlist depth, score weights and merging can then be replayed from the cached scores without rerunning the models; a new checkpoint or generator needs new candidate scores. The replay leaves out the forward models.

The benchmark separates retrieval, scoring and merging. In the PubChem channel, deepening the fingerprint-only shortlist from 1,000 to 5,000 rows raises the share of answers inside the shortlist from 90% to 98%, yet MRR@25 stays at 0.51: the fingerprint model alone places the answer first for 40% of the molecules. Admitting and scoring candidates with the popularity prior raises top-1 accuracy to 93% and MRR@25 to 0.95 (Table 14). All 250 molecules are in the denominator; 246 answers occur in the PubChem mass window.

Table 15 replays the merge. With fixed insertion positions a PubChem answer enters no higher than rank 2 or 4, which gives MRR@25 of 0.519 when the answer is removed from the pool. A promotion rule was then tuned on this scenario. With `S = z(FPNet) + 0.25 · pop` for the best PubChem proposal, it moved that proposal to rank 1 when `S > 6` and `pop ≥ 5`, and used the earlier positions when `S > 5`. In the replay it raised the pool-removal result to 0.718 and changed neither pool scenario (Figure 20); it fired for 100 molecules, each time with the correct structure.

The leaderboard did not confirm it. Version 20 of this notebook applied the rule together with unit forward-model weights, and its public score fell from 0.386 to 0.374 (Appendix A). The two changes were made together, so the score cannot say which of them cost more. The replay, however, rests on a situation that hardly occurs: a well-documented natural product that is missing from the pool. Of the 250 panel answers, 249 are COCONUT structures, so a documented natural product is normally found in the pool itself. Removing a known answer from the pool therefore simulates a case the real candidate sources rarely produce. The rule is not used in this version; Table 15 and Figure 20 remain as a record of how far a convenient development scenario can mislead.

In [ ]:
# Tables 14-15 and Figure 20: replayed variants on 250 natural products with known answers.
bench_dir = _first(['/kaggle/input/**/casmi26-natural-product-panel-benchmark/**/results_merge_rules.json',
                    os.environ.get('CASMI_BENCH_DIR', '.') + '/results_merge_rules.json']).parent
pc_res = json.loads((bench_dir / 'results_pubchem_channel.json').read_text())
merge_res = json.loads((bench_dir / 'results_merge_rules.json').read_text())
pc_names = {
    'v18: N1=1000 rows': 'FPNet only, 1,000-row shortlist',
    'v4i: N1=5000 rows': 'FPNet only, 5,000-row shortlist',
    'full window, no prior': 'FPNet only, whole mass window',
    'v19: N1=5000 + pop 0.25 + union 200': 'FPNet + popularity 0.25 (versions 19-28)',
    'full window + pop 1.00': 'FPNet + popularity 1.00',
    'full + (pop + patents) 0.25': 'FPNet + popularity and patents',
}
by_name = {v['name']: v for v in pc_res['variants']}
table14 = pd.DataFrame([{'PubChem channel variant': label, 'MRR@25': f"{by_name[k]['mrr25']:.3f}", 'top-1': f"{by_name[k]['top1']:.0%}",
                         'answer in shortlist': f"{by_name[k]['in_shortlist']:.0%}"} for k, label in pc_names.items()]).set_index('PubChem channel variant')
display(Markdown(f"**Table 14. PubChem channel on {pc_res['summary']['panel']} natural products whose answer is withheld from the pool.**"))
display(table14)

scen = {'lib': 'panel spectra hidden,\nother libraries kept', 'nospec': 'in the pool,\nno reference spectrum', 'dbdrop': 'removed from the pool\n(PubChem or generation)'}
rule_names = {'no PubChem insertion': 'main list with popularity, no PubChem insertion', 'v19 gate': 'fixed slots (rank 2 or 4)', 'promote if S>6 and pop>=5; early slots if S>5': 'promotion rule of version 20 (no forward models)'}
rules = {r['name']: r for r in merge_res['rules']}
table15 = pd.DataFrame({label: {scen[s].replace('\n', ' '): f"{rules[k][s]['mrr25']:.3f}" for s in scen} for k, label in rule_names.items()}).T
display(Markdown('**Table 15. MRR@25 of the final list under three merge rules (replay without the forward models).**'))
display(table15)
fired = merge_res['promotion']['fired']
display(Markdown(f"In the replay, the promotion condition of version 20 (weak library match, `S > 6`, `pop ≥ 5`) fired for **{fired['dbdrop']}** molecules of the pool-removal scenario, "
                 f"**{merge_res['promotion']['correct_when_removed_from_pool']}** of them with the correct structure, "
                 f"and for **{fired['nospec']}** and **{fired['lib']}** molecules of the two scenarios whose answer stays in the pool."))

display(Markdown('**Figure 20. A promotion rule improves the replayed pool-removal scenario; the leaderboard did not confirm it.**'))
fig, ax = plt.subplots(figsize=(10.5, 5.4))
width = 0.27
for j, ((k, label), color) in enumerate(zip(rule_names.items(), ['#c9d3da', '#6fa8b5', '#16324f'])):
    vals = [rules[k][s]['mrr25'] for s in scen]
    bars = ax.bar(np.arange(len(scen)) + (j - 1) * width, vals, width, color=color, label=label)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.012, f'{v:.3f}', ha='center', fontsize=9.5)
ax.set_xticks(np.arange(len(scen))); ax.set_xticklabels(list(scen.values()))
ax.set(ylabel='MRR@25', ylim=(0, 1.42), yticks=np.arange(0, 1.01, 0.2)); ax.grid(axis='x', visible=False); ax.legend(frameon=False, loc='upper center', ncol=1, fontsize=10)
plt.tight_layout(); plt.show()


*Tables 14–15 and Figure 20. Replayed variants on 250 natural products with known answers. Popularity is the largest single effect in the PubChem channel, and the merge rule decides how much of it reaches the final list. The cached replay uses FPNet A in the main engine and FPNet A+B in the PubChem channel; it leaves out the forward models and the second engine. The panel is a development proxy: the main ranker and its feature resources may have seen these molecules, which makes the two pool scenarios optimistic. The answers are well-documented natural products, so this panel may overstate the benefit of the prior for rarely studied compounds, and the pool-removal scenario also credits the analogue generator with rebuilding known structures. Tables 14 and 15 come from separate replays with slightly different candidate-exclusion rules and are not an exact decomposition of one candidate list. Contamination, the choice of thresholds on the same molecules and the artificial pool-removal scenario limit transfer to the hidden test, as the public score of version 20 showed.*

### Fingerprint-only ranking on training and panel queries

The spectrum-to-fingerprint model is the main learned signal in both channels. Table 16 compares 3,000 structure–polarity queries sampled from training structures with 499 queries from the 250 natural-product panel molecules. Each query merges up to eight spectra. The answer is present in the pool, and the fingerprint score alone ranks the pool structures within 10 ppm of the answer's mass, so the table measures ordering when the candidate is available. According to the supplied training code, FPNet A and B exclude the panel library from fitting; [full1](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-fpnet-full1), a public checkpoint of the same architecture from the [v4m notebook](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4m-inference), was fitted on all training folds including the panel. The exact rosters of the public checkpoints are not published. The numbers and the evaluation script are stored in the [benchmark dataset](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark).

In [ ]:
# Table 16: pool-window retrieval by the fingerprint model alone, on training structures and on the natural-product panel.
fp_res = json.loads((bench_dir / 'results_fingerprint_seen_unseen.json').read_text())
table16 = pd.DataFrame([{
    'Fingerprint checkpoint': m['name'].split(' (')[0], 'panel in training': 'yes' if m['trained_on_panel'] else 'no',
    'training queries: top-1': f"{m['training_sample']['top1']:.0%}", 'training queries: MRR@25': f"{m['training_sample']['mrr25']:.3f}",
    'panel queries: top-1': f"{m['panel']['top1']:.0%}", 'panel queries: top-25': f"{m['panel']['top25']:.1%}", 'panel queries: MRR@25': f"{m['panel']['mrr25']:.3f}",
} for m in fp_res['models']]).set_index('Fingerprint checkpoint')
display(Markdown(f"**Table 16. Ranking by the fingerprint model alone: {fp_res['models'][0]['training_sample']['n']:,} training queries "
                 f"and {fp_res['models'][0]['panel']['n']} natural-product panel queries.**"))
display(table16)
del fp_res, table16


*Table 16. FPNet A ranks the answer first for 89% of the sampled training queries and for 42% of the panel queries; the answer is inside the first 25 for 95% of the panel queries. Full1, whose training includes the panel, reaches 89% top-1 on the panel. The difference is consistent with an effect of training exposure: the fingerprint model orders structures it was fitted on far better than new ones, although the two query sets also differ in chemistry and window size. For a hidden answer without reference spectra, the panel figure of FPNet A is the more relevant one, and its errors are errors of ordering inside the mass window. This notebook keeps FPNet A in the main engine and A+B in the PubChem channel; full1 appears here only to show what training exposure does.*

In [ ]:
# EDA has finished: release scalar tables and RDKit objects before constructing the search index.
import gc
for _name in ("train", "unique_structures", "descriptor_sample", "desc", "fp_sample", "records",
              "preview_pool", "representatives", "mols", "X", "projection", "library_pairs"):
    globals().pop(_name, None)
plt.close("all")
gc.collect()


## 12. Run the complete model

To reproduce the notebook, attach the **CASMI26 competition data**, the [attributed v4g asset fork](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets), and [Open Model and Candidate Resources](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-open-model-and-candidate-resources) for its RDKit 2026.03.3 offline wheel and the inputs of the second engine. The figures of Sections 9 and 11 read three small datasets: the [natural-product knowledge table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-knowledge-table), the [identity-disjoint folds](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-identity-disjoint-validation-folds) and the [natural-product panel benchmark](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-natural-product-panel-benchmark). Also attach the [ChEBI and LIPID MAPS candidates](https://www.kaggle.com/datasets/prvsiyan/chebi-lipidmaps-casmi26) of the second engine, the [public GLACIER package](https://www.kaggle.com/datasets/ahmedberatozer/casmi26-glacier) and the [PubChem popularity prior](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior). Select a Kaggle GPU and leave Internet off. Run all cells. The current model reads `train.parquet` and `test.parquet`, writes `/kaggle/working/submission.csv`, and records component counts and file hashes in `/kaggle/working/v29_execution_receipt.json`. Submit the **completed version's** `submission.csv` as a Code competition submission to obtain its public score.

The executable cells verify the fork's manifest and every file hash, restore source paths with symlinks, and construct a cleaned measured-spectrum cache from the **currently mounted** competition training file. PubChem A+B proposals run first in an isolated process, followed by the second engine in another. The main engine returns up to 60 candidates per molecule, the popularity prior re-orders its pool candidates, ICEBERG and GLACIER try to reorder same-formula groups, the merge rule inserts PubChem proposals, rank fusion combines the result with the second engine where no strong library match exists, and up to 25 scorer-distinct structures are written in sample-submission order. The run checks that both required channels produce lists for every visible ID; ICEBERG and GLACIER coverage and any failures are reported separately. Table 1 counts the mounted training file (2,539,608 spectra at the time of writing). The PubChem arrays exceed 6 GiB, so expect a substantial GPU run and enough working memory for the measured-spectrum cache. The output checks establish that the job ran on the right files and produced valid lists; the accuracy of the complete pipeline on the hidden test is measured by Kaggle.

ICEBERG has a 4,000-second budget and GLACIER, which scores `[M+H]+` acquisitions only, a 3,000-second budget. Both run in isolated subprocesses, using their packaged RDKit 2025.03.6 dependencies; the main process and final connectivity validation use RDKit 2026.03.3. A successful notebook run can still have incomplete forward-model coverage, so inspect the saved component receipts as well as the final CSV.


In [ ]:
"""Verify the attributed v4g asset fork and restore its source paths by symlink.

The Kaggle dataset stores each original file as one flat transport name. The
symlinks keep the large NumPy arrays in /kaggle/input for memory mapping.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path, PurePosixPath


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def restore(input_root: Path, work_root: Path, expected_manifest_sha256: str,
            *, verify_files: bool = True, create_links: bool = True) -> dict:
    """Check the exact source files, then make the original directory layout.

    ``create_links=False`` is a read-only preflight; the normal Kaggle run
    creates one symlink per file. ``verify_files`` always defaults to True.
    """
    input_root = Path(input_root).resolve(strict=True)
    work_root = Path(work_root)
    manifest_path = input_root / "MANIFEST.json"
    assert manifest_path.is_file()
    assert sha256(manifest_path) == expected_manifest_sha256, "resource manifest changed"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert manifest["dataset"] == "dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets"
    assert manifest["layout"] == "flat_transport_names_with_double_underscore_path_separator"
    assert manifest["file_count"] == 134 and manifest["total_bytes"] == 11060095811
    rows = manifest["files"]
    assert len(rows) == manifest["file_count"]
    seen_transport: set[str] = set()
    for original, row in sorted(rows.items()):
        rel = PurePosixPath(original)
        assert not rel.is_absolute() and all(p not in ("", ".", "..") for p in rel.parts)
        transport = row["transport_name"]
        assert transport == original.replace("/", "__")
        assert transport not in seen_transport and Path(transport).name == transport
        seen_transport.add(transport)
        source = input_root / transport
        assert source.is_file() and source.stat().st_size == row["bytes"], original
        if verify_files:
            assert sha256(source) == row["sha256"], f"resource changed: {original}"
        if create_links:
            link = work_root.joinpath(*rel.parts)
            link.parent.mkdir(parents=True, exist_ok=True)
            if link.is_symlink():
                assert link.resolve(strict=True) == source, f"wrong existing link: {link}"
            else:
                assert not link.exists(), f"existing resource would be overwritten: {link}"
                link.symlink_to(source)
            assert link.resolve(strict=True) == source
    assert sum(row["bytes"] for row in rows.values()) == manifest["total_bytes"]
    return {
        "source_files": len(rows),
        "source_bytes": manifest["total_bytes"],
        "manifest_sha256": expected_manifest_sha256,
        "file_hashes_checked": bool(verify_files),
        "links_created": bool(create_links),
        "folders": {name: str(work_root / name) for name in
                    ("v2_pool", "v3_models", "v4b", "pubchem_tier", "iceberg")},
    }

In [ ]:
# Verify every attributed source file before constructing model state.
import os, sys, time, subprocess, json, pickle, hashlib
ASSET_INPUT = Path('/kaggle/input/casmi26-attributed-v4g-reproduction-assets')
ASSET_TREE = Path('/kaggle/working/v4g_sources')
resource_receipt = restore(ASSET_INPUT, ASSET_TREE, 'd193977598848224e794ba50bd55dfd1b9e46498c3d8ca8f20593f4e21a99b7e')
T0 = time.time()
LIB_TAU, REL_TH = 0.9, 600.0
TOPN, ICE_LAM, ICE_BUDGET, ICE_PC = 60, 0.5, 4000, True
POP_LAM, POP_UNION, POOLPOP_MU = 0.0, 200, 0.15      # no prior in the PubChem channel (POP_LAM 0 also disables the union); main list 0.15
SLOTS_AGG, SLOTS_GENTLE = [2, 4, 6, 8, 10], [4, 8, 12, 16, 20]
os.makedirs('/kaggle/working/numba_cache', exist_ok=True)
os.environ['NUMBA_CACHE_DIR'] = '/kaggle/working/numba_cache'
V4CODE = str(ASSET_TREE/'v4b/code')
V4MOD = str(ASSET_TREE/'v4b')
V3CODE = str(ASSET_TREE/'v3_models/code')
V3MOD = str(ASSET_TREE/'v3_models')
POOL_DIR = str(ASSET_TREE/'v2_pool')
PC_DIR = str(ASSET_TREE/'pubchem_tier')
COMP = str(DATA)
MAN = json.loads((ASSET_TREE/'v4b/MANIFEST.json').read_text())
assert MAN['ranker']['n_features'] == 160 and MAN['ranker']['n_boosters'] == 4
print('Verified source files:', resource_receipt['source_files'])

# Upstream's ICEBERG lookup resolves to our hash-verified restored source tree.
def find(pattern):
    if pattern == 'casmi26-iceberg/**/ice_runner.py':
        return str(ASSET_TREE / 'iceberg/ice_runner.py')
    import glob
    hits = sorted(glob.glob('/kaggle/input/**/' + pattern, recursive=True), key=len)
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]

# Fail closed if the attached GLACIER package differs from the reviewed manifest.
GL_ROOT = Path(find('casmi26-glacier/**/gl_runner.py')).parent
assert sha256(GL_ROOT / 'MANIFEST.json') == '53bf3f8724cb0d8ddbb104254deda88b0a90502e3ab031aa42cbbf010b38634d', 'GLACIER manifest changed'
gl_manifest = json.loads((GL_ROOT / 'MANIFEST.json').read_text())
for rel, spec in gl_manifest['files'].items():
    # Kaggle does not mount the uploader's dataset-metadata.json.
    if rel == 'dataset-metadata.json':
        continue
    assert sha256(GL_ROOT / rel) == spec['sha256'], f'GLACIER file changed: {rel}'


# Popularity prior: log1p PubChem substance and PubMed counts, aligned row-by-row with the PubChem tier and pool.
POP_ROOT = Path(find('casmi26-pubchem-popularity-prior/**/MANIFEST.json')).parent
assert sha256(POP_ROOT / 'MANIFEST.json') == 'b4d3aa114e7ba04c618bb022d7943f0d43f0333ecd578dacbe15b8a666b3d67f', 'popularity manifest changed'
pop_manifest = json.loads((POP_ROOT / 'MANIFEST.json').read_text())
for rel, spec in pop_manifest['files'].items():
    assert sha256(POP_ROOT / rel) == spec['sha256'], f'popularity file changed: {rel}'
os.environ.update(CASMI_POP_DIR=str(POP_ROOT), CASMI_POP_LAM=str(POP_LAM), CASMI_POP_UNION=str(POP_UNION))
print('Popularity prior', pop_manifest['pubchem_release'], 'lambda', POP_LAM, 'union', POP_UNION, 'pool mu', POOLPOP_MU)


### PubChem proposals in a separate process

The PubChem channel runs in a separate process to isolate its model dependencies. It returns ranked structures and connectivity keys for each molecule before the main ranking begins.

In [ ]:
# Isolated PubChem A+B proposal channel; its v3 casmi package cannot shadow the main v4b engine.
CORE = '"""P_pubchem probe core (lever 3). Embedded verbatim into the Kaggle notebook (written to /kaggle/working and imported,\nso worker processes can import it) and imported directly by the local smoke test.\n\nPer molecule:\n  1. f.z logits of the A+B FPNet ensemble, exactly like Engine.model_logits: z = 0.5 * (mean per-spectrum logits +\n     mean per-polarity merged-spectrum logits); models averaged inside ModelBank.\n  2. PubChem window: +-10 ppm around the target neutral mass (median over the molecule\'s spectra), no subsampling.\n  3. pass 1 (src/casmi/pubchem.py tier): partial f.z on the ECFP4 block of the selected bits, keep top PC_N1 = 1000.\n  4. pass 2: full selected-bit fingerprints, full f.z, sort descending.\n  5. walk down the f.z order, compute the metric score key (tautomer-canonical InChIKey14); keep a structure only if\n     its key is NOT in the candidate pool (train U COCONUT) and not already listed; stop at 25.\n  An empty list becomes the single \'CCO\' fallback (never padded).\nCandidates are ranked without using test answers.\n"""\nfrom __future__ import annotations\n\nimport os\nimport time\n\nimport numpy as np\n\nPC_N1 = 5000\nPPM = 10.0\nK = 25\n\n_W = {}\n\n\n# ------------------------------------------------------------------------------------------ model part (main process)\ndef molecule_logits(bank, spectra):\n    """spectra: list of dict(mz, it, mode, adduct, prec, ce, ce_n). Returns z_avg (nbits,) float32 or None."""\n    from casmi import chem, fpnet\n    from casmi.spectra import merge_spectra\n    items_s = []\n    for s in spectra:\n        pm, pi = fpnet.prep_peaks(s[\'mz\'], s[\'it\'], s[\'prec\'])\n        ce_ok = s[\'ce\'] is not None and not np.isnan(s[\'ce\'])\n        items_s.append(dict(mz=pm, it=pi, prec=float(s[\'prec\']), adduct_ix=chem.adduct_index(s[\'adduct\']),\n                            ce=float(s[\'ce\']) if ce_ok else 0.0, ce_known=1.0 if ce_ok else 0.0,\n                            n_merged=min(int(s.get(\'ce_n\', 1)), 8), mode=float(s[\'mode\'])))\n    items_m = []\n    for md in (1, -1):\n        grp = [s for s in spectra if s[\'mode\'] == md]\n        if not grp:\n            continue\n        mm, ii = merge_spectra([(s[\'mz\'], s[\'it\']) for s in grp])\n        prec = float(np.median([s[\'prec\'] for s in grp]))\n        pm, pi = fpnet.prep_peaks(mm, ii, prec)\n        ces = [s[\'ce\'] for s in grp if s[\'ce\'] is not None and not np.isnan(s[\'ce\'])]\n        adducts = [s[\'adduct\'] for s in grp]\n        ad = max(adducts, key=adducts.count)          # deterministic tie-break (first occurrence)\n        items_m.append(dict(mz=pm, it=pi, prec=prec, adduct_ix=chem.adduct_index(ad),\n                            ce=float(np.mean(ces)) if ces else 0.0, ce_known=1.0 if ces else 0.0,\n                            n_merged=min(sum(max(1, int(s.get(\'ce_n\', 1))) for s in grp), 8), mode=float(md)))\n    if not items_s:\n        return None\n    zs, _ = bank.logits_el(items_s)\n    zm, _ = bank.logits_el(items_m)\n    return (0.5 * (zs.mean(0) + zm.mean(0))).astype(np.float32)\n\n\n# ------------------------------------------------------------------------------------------ PubChem part (workers)\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\n    import sys\n    if code_dir and code_dir not in sys.path:\n        sys.path.insert(0, code_dir)\n    import pandas as pd\n    from casmi import chem\n    _W[\'chem\'] = chem\n    _W[\'mass\'] = np.load(os.path.join(pc_dir, \'pc_mass.npy\'), mmap_mode=\'r\')\n    _W[\'off\'] = np.load(os.path.join(pc_dir, \'pc_off.npy\'), mmap_mode=\'r\')\n    _W[\'buf\'] = np.load(os.path.join(pc_dir, \'pc_smiles.npy\'), mmap_mode=\'r\')\n    bits = np.load(bits_path)\n    _W[\'bits\'] = bits\n    _W[\'sel_e\'] = np.where(bits < 4096)[0]\n    _W[\'raw_e\'] = bits[_W[\'sel_e\']]\n    _W[\'pool_keys\'] = set(pd.read_parquet(pool_meta_path, columns=[\'key\']).key.values.tolist())\n    r = chem._rdkit()\n    _W[\'ecfp4\'] = r[\'gen\'].GetMorganGenerator(radius=2, fpSize=4096)\n    _W[\'Chem\'] = r[\'Chem\']\n\n\ndef _window(target):\n    tol = target * PPM * 1e-6\n    a = int(np.searchsorted(_W[\'mass\'], target - tol, \'left\')); b = int(np.searchsorted(_W[\'mass\'], target + tol, \'right\'))\n    off = np.asarray(_W[\'off\'][a:b + 1]).astype(np.int64)\n    if b <= a:\n        return []\n    raw = bytes(_W[\'buf\'][off[0]:off[-1]])\n    base = off[0]\n    return [raw[off[i] - base:off[i + 1] - base].decode(\'ascii\') for i in range(b - a)]\n\n\ndef probe_one(task):\n    """task = (molecule_id, target_mass, z (nbits,) float32). Returns (molecule_id, [smiles], [f.z], [score keys], diag dict)."""\n    mid, target, z = task\n    t0 = time.time()\n    chem = _W[\'chem\']; Chem = _W[\'Chem\']; gen = _W[\'ecfp4\']\n    smis = _window(float(target))\n    n = len(smis)\n    diag = dict(molecule_id=mid, n_window=n, n_pass1=0, n_pass2=0, n_keyed=0, n_in_pool=0, n_listed=0)\n    if n == 0:\n        diag[\'secs\'] = time.time() - t0\n        return mid, [], [], [], diag\n    raw_e = _W[\'raw_e\']; z_e = z[_W[\'sel_e\']].astype(np.float32)\n    E = np.zeros((n, len(raw_e)), np.float32); ok = np.zeros(n, bool)\n    for i, s in enumerate(smis):\n        m = Chem.MolFromSmiles(s)\n        if m is None:\n            continue\n        E[i] = gen.GetFingerprintAsNumPy(m)[raw_e]; ok[i] = True\n    sc1 = E @ z_e\n    sc1[~ok] = -np.inf\n    n1 = min(PC_N1, int(ok.sum()))\n    diag[\'n_pass1\'] = n1\n    if n1 == 0:\n        diag[\'secs\'] = time.time() - t0\n        return mid, [], [], [], diag\n    top1 = np.argpartition(-sc1, n1 - 1)[:n1] if n1 < n else np.where(ok)[0]\n    bits = _W[\'bits\']\n    fz, idx = [], []\n    for i in top1:\n        fp = chem.raw_fingerprint(smis[i])\n        if fp is None:\n            continue\n        fz.append(float(fp[bits].astype(np.float32) @ z)); idx.append(int(i))\n    diag[\'n_pass2\'] = len(idx)\n    order = np.argsort(-np.asarray(fz), kind=\'stable\')\n    out, out_fz, out_k, seen = [], [], [], set()\n    for o in order:\n        s = smis[idx[o]]\n        k = chem.score_key(s)\n        diag[\'n_keyed\'] += 1\n        if k is None or k in seen:\n            continue\n        seen.add(k)\n        if k in _W[\'pool_keys\']:\n            diag[\'n_in_pool\'] += 1\n            continue\n        out.append(s); out_fz.append(float(fz[o])); out_k.append(k)\n        if len(out) >= K:\n            break\n    diag[\'n_listed\'] = len(out)\n    diag[\'secs\'] = time.time() - t0\n    return mid, out, out_fz, out_k, diag\n'
RUNNER = '"""PubChem-only channel for the final submission (runs in its own process so its casmi package\n(S1 snapshot, FPNet A+B) never collides with the v1 engine\'s casmi package).\n\nPer molecule: z = A+B logits (probe_core2.molecule_logits, using the supplied A+B weights);\nbest_pool_fz = max f.z over the pool window (+-10 ppm, fallback 30 ppm) with the same z (as P1 logits.py);\nPubChem-only list = probe_core2.probe_one (top 25 by f.z, keys not in the pool) with f.z values and score keys.\nWrites a json {molecule_id: {pc, pc_fz, pc_keys, best_pool_fz, target}}.\nusage: python pc_runner.py <v3_code_dir> <v3_models_dir> <pool_dir> <pubchem_dir> <test_parquet> <out_json> <n_workers> [max_mols]\n"""\nimport os, sys, json, time\nimport numpy as np, pandas as pd\n\nif __name__ == \'__main__\':\n    V3CODE, V3MOD, POOL_DIR, PC_DIR, TEST, OUT, NW = sys.argv[1:8]\n    MAXM = int(sys.argv[8]) if len(sys.argv) > 8 else 0\n    NW = int(NW)\n    sys.path.insert(0, V3CODE); sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\n    import torch, glob\n    import probe_core2 as pc\n    from casmi import fpnet, chem\n    T0 = time.time()\n    models = sorted(glob.glob(os.path.join(V3MOD, \'fpnet_*.pt\')))\n    assert len(models) == 2, models\n    dev = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    bank = fpnet.ModelBank(models, device=dev)\n    bits = np.load(os.path.join(POOL_DIR, \'fp_bits.npy\')); nbits = len(bits)\n    pmass = pd.read_parquet(os.path.join(POOL_DIR, \'pool_meta.parquet\'), columns=[\'mass\']).mass.values.astype(np.float64)\n    assert np.all(np.diff(pmass) >= 0), \'pool not mass-sorted\'\n    pfp = np.load(os.path.join(POOL_DIR, \'pool_fp.npy\'), mmap_mode=\'r\')\n    te = pd.read_parquet(TEST)\n    te[\'nm\'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)\n    tasks, extra = [], {}\n    groups = list(te.groupby(\'molecule_id\', sort=False))\n    if MAXM:\n        groups = groups[:MAXM]\n    for mid, sub in groups:\n        nms = sub.nm.values[np.isfinite(sub.nm.values)]\n        if not len(nms):\n            continue\n        target = float(np.median(nms))\n        spectra = []\n        for r in sub.itertuples():\n            ce = r.collision_energy_ev\n            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)\n            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\n                                mode=1 if r.ionization_mode == \'positive\' else -1, adduct=r.adduct,\n                                prec=float(r.precursor_mz), ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,\n                                ce_n=int(len(ce_arr)) if len(ce_arr) else 1))\n        spectra = spectra[:16]\n        try:\n            z = pc.molecule_logits(bank, spectra)\n        except Exception as e:\n            print(\'ERROR logits\', mid, repr(e), flush=True); z = None\n        if z is None:\n            continue\n        tol = target * 10e-6\n        a = np.searchsorted(pmass, target - tol, \'left\'); b = np.searchsorted(pmass, target + tol, \'right\')\n        if b <= a:\n            tol = target * 30e-6\n            a = np.searchsorted(pmass, target - tol, \'left\'); b = np.searchsorted(pmass, target + tol, \'right\')\n        if b > a:\n            F = np.unpackbits(np.asarray(pfp[a:b]), axis=1)[:, :nbits].astype(np.float64)\n            best = float((F @ z.astype(np.float64)).max())\n        else:\n            best = float(\'nan\')\n        extra[mid] = dict(best_pool_fz=best, target=target)\n        tasks.append((mid, target, z))\n    del bank; torch.cuda.empty_cache()\n    print(f\'logits for {len(tasks)} molecules {time.time()-T0:.0f}s\', flush=True)\n    res = {}\n    import multiprocessing as mp\n    ctx = mp.get_context(\'fork\' if sys.platform != \'win32\' else \'spawn\')\n    with ctx.Pool(NW, initializer=pc.init_worker, initargs=(PC_DIR, os.path.join(POOL_DIR, \'fp_bits.npy\'),\n                                                            os.path.join(POOL_DIR, \'pool_meta.parquet\'), V3CODE)) as P:\n        for k, (mid, smis, fzs, keys, d) in enumerate(P.imap_unordered(pc.probe_one, tasks, chunksize=1)):\n            res[mid] = dict(pc=smis, pc_fz=fzs, pc_keys=keys, fz_top=d.get(\'fz_top\'), top_pop=d.get(\'top_pop\'), top_z=d.get(\'top_z\'), **extra[mid])\n            if (k + 1) % 25 == 0 or k + 1 == len(tasks):\n                print(f\'  pubchem {k+1}/{len(tasks)} {time.time()-T0:.0f}s\', flush=True)\n    json.dump(res, open(OUT, \'w\'))\n    print(\'pc_runner done\', len(res), f\'{time.time()-T0:.0f}s\', flush=True)\n'
POP_PATCH = "\n\n# ---- v17 patch: popularity prior (log1p SID + log1p PMID from PubChem, mapped onto this store by stereo-stripped SMILES) ----\n_POP_DIR = os.environ.get('CASMI_POP_DIR'); _POP_LAM = float(os.environ.get('CASMI_POP_LAM', '0')); _POP_UNION = int(os.environ.get('CASMI_POP_UNION', '0'))\n_init_worker0 = init_worker\n\n\ndef init_worker(pc_dir, bits_path, pool_meta_path, code_dir=None):\n    _init_worker0(pc_dir, bits_path, pool_meta_path, code_dir)\n    if _POP_DIR:\n        _W['ls'] = np.load(os.path.join(_POP_DIR, 'pc_lsid.npy'), mmap_mode='r')\n        _W['lp'] = np.load(os.path.join(_POP_DIR, 'pc_lpmid.npy'), mmap_mode='r')\n\n\ndef probe_one(task):\n    mid, target, z = task\n    t0 = time.time()\n    chem = _W['chem']; Chem = _W['Chem']; gen = _W['ecfp4']\n    tol = float(target) * PPM * 1e-6\n    a0 = int(np.searchsorted(_W['mass'], float(target) - tol, 'left'))\n    smis = _window(float(target))\n    n = len(smis)\n    diag = dict(molecule_id=mid, n_window=n, n_pass1=0, n_pass2=0, n_keyed=0, n_in_pool=0, n_listed=0, fz_top=None)\n    if n == 0:\n        diag['secs'] = time.time() - t0\n        return mid, [], [], [], diag\n    raw_e = _W['raw_e']; z_e = z[_W['sel_e']].astype(np.float32)\n    E = np.zeros((n, len(raw_e)), np.float32); ok = np.zeros(n, bool)\n    for i, s in enumerate(smis):\n        m = Chem.MolFromSmiles(s)\n        if m is None:\n            continue\n        E[i] = gen.GetFingerprintAsNumPy(m)[raw_e]; ok[i] = True\n    sc1 = E @ z_e\n    sc1[~ok] = -np.inf\n    n1 = min(PC_N1, int(ok.sum()))\n    diag['n_pass1'] = n1\n    if n1 == 0:\n        diag['secs'] = time.time() - t0\n        return mid, [], [], [], diag\n    top1 = np.argpartition(-sc1, n1 - 1)[:n1] if n1 < n else np.where(ok)[0]\n    use_pop = ('ls' in _W) and _POP_LAM > 0\n    pop = (np.asarray(_W['ls'][a0:a0 + n], np.float32) + np.asarray(_W['lp'][a0:a0 + n], np.float32)) if use_pop else np.zeros(n, np.float32)\n    if use_pop and _POP_UNION > 0:\n        pp = np.where(ok, pop, -np.inf); ku = min(_POP_UNION, int(ok.sum()))\n        top1 = np.union1d(top1, np.argpartition(-pp, ku - 1)[:ku])\n    bits = _W['bits']\n    fz, idx = [], []\n    for i in top1:\n        fp = chem.raw_fingerprint(smis[i])\n        if fp is None:\n            continue\n        fz.append(float(fp[bits].astype(np.float32) @ z)); idx.append(int(i))\n    diag['n_pass2'] = len(idx)\n    if not idx:\n        diag['secs'] = time.time() - t0\n        return mid, [], [], [], diag\n    fz = np.asarray(fz, np.float64); idx = np.asarray(idx)\n    keyc = {}\n    def key(j):\n        if j not in keyc:\n            keyc[j] = chem.score_key(smis[idx[j]]); diag['n_keyed'] += 1\n        return keyc[j]\n    for j in np.argsort(-fz, kind='stable'):        # gate quantity = v16's pc_fz[0] (best non-pool structure by raw f.z)\n        k = key(j)\n        if k is not None and k not in _W['pool_keys']:\n            diag['fz_top'] = float(fz[j]); break\n    zf = (fz - fz.mean()) / (fz.std() + 1e-9)\n    sc = zf + _POP_LAM * pop[idx]\n    order = np.argsort(-sc, kind='stable')\n    out, out_s, out_k, seen = [], [], [], set()\n    for j in order:\n        k = key(j)\n        if k is None or k in seen:\n            continue\n        seen.add(k)\n        if k in _W['pool_keys']:\n            diag['n_in_pool'] += 1\n            continue\n        out.append(smis[idx[j]]); out_s.append(float(sc[j])); out_k.append(k)\n        if len(out) == 1:                      # popularity and standardised fingerprint score of the best proposal\n            diag['top_pop'] = float(pop[idx[j]]); diag['top_z'] = float(zf[j])\n        if len(out) >= K:\n            break\n    diag['n_listed'] = len(out)\n    diag['secs'] = time.time() - t0\n    return mid, out, out_s, out_k, diag\n"
open('/kaggle/working/probe_core2.py', 'w').write(CORE + POP_PATCH)
open('/kaggle/working/pc_runner.py', 'w').write(RUNNER)
PC = {}
try:
    rr = subprocess.run([sys.executable, '/kaggle/working/pc_runner.py', V3CODE, V3MOD, POOL_DIR, PC_DIR,
                         os.path.join(COMP, 'test.parquet'), '/kaggle/working/pc_lists.json', '4'],
                        capture_output=True, text=True, timeout=4 * 3600)
    print(rr.stdout[-3000:]); print(rr.stderr[-3000:])
    PC = json.load(open('/kaggle/working/pc_lists.json'))
except Exception as e:
    print('PUBCHEM CHANNEL FAILED -> base lists only:', repr(e))
print('pubchem lists', len(PC), f'{time.time()-T0:.0f}s')
assert 'rr' in globals() and rr.returncode == 0, 'PubChem process failed; inspect its diagnostic output above'
assert PC, 'PubChem process produced no proposal lists'


### A second candidate engine in a separate process

The second engine loads its own candidate arrays, spectrum encoders and ranker rows, so it runs in a process of its own before the main engine starts. It returns up to 40 ranked structures per molecule; if it fails, the lists of the main pipeline are used unchanged.

In [ ]:
# Second candidate engine in its own process: the two-ranker pipeline built on prvsiyan's analog-propagation baseline,
# as packaged in the public fusion notebook by seyitkaangunes (training, COCONUT, ChEBI and LIPID MAPS structures).
# Any failure leaves the single-engine lists in place.
import signal
ENG_DIR = '/kaggle/working/eng'; os.makedirs(ENG_DIR, exist_ok=True)
ENG_FILES = {'pv.py': '"""prvsiyan analog-propagation pipeline components (public notebook, version of 2026-09-16), copied\nverbatim where possible so local simulations and the Kaggle notebook share one source of truth.\nAdditions are marked `# ADDED`.\n"""\nimport math\nimport numpy as np\nfrom numba import njit, prange\n\n\nclass CFG:\n    PPM_WIN = 10.0\n    PPM_FALLBACK = 30.0\n    INT_FLOOR = 0.002\n    MAX_PEAKS = 256\n    MZ_TOL = 0.01\n    INT_POWER = 1.0\n    ENT_WEIGHT = True\n    ANALOG_WIN = 200.0\n    N_ANALOG = 80\n    SIM_POWER = 3.0\n    W1_PRIORS = (0.30, 0.60)\n    SEEDS = (0, 1, 2, 3)\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\n    TOPN = 25\n\n\n# ----------------------------------------------------------------------------------------------\n# similarity kernels\n# ----------------------------------------------------------------------------------------------\n@njit(cache=True, fastmath=True)\ndef _clean(mz, it, floor, topk, power, ent_weight):\n    n = len(mz)\n    if n == 0: return np.empty(0, np.float32), np.empty(0, np.float32)\n    mx = 0.0\n    for i in range(n):\n        if it[i] > mx: mx = it[i]\n    if mx <= 0: return np.empty(0, np.float32), np.empty(0, np.float32)\n    thr = floor * mx; c = 0\n    for i in range(n):\n        if it[i] >= thr: c += 1\n    idx = np.empty(c, np.int64); j = 0\n    for i in range(n):\n        if it[i] >= thr: idx[j] = i; j += 1\n    if c > topk:\n        v = np.empty(c, np.float32)\n        for i in range(c): v[i] = it[idx[i]]\n        o = np.argsort(v)[c - topk:]\n        k2 = np.empty(topk, np.int64)\n        for i in range(topk): k2[i] = idx[o[i]]\n        k2.sort(); idx = k2; c = topk\n    om = np.empty(c, np.float32); oi = np.empty(c, np.float32)\n    s = 0.0\n    for i in range(c):\n        om[i] = mz[idx[i]]; v = it[idx[i]] ** power; oi[i] = v; s += v\n    if s > 0:\n        for i in range(c): oi[i] /= s\n    if ent_weight:\n        S = 0.0\n        for i in range(c):\n            if oi[i] > 0: S -= oi[i] * np.log(oi[i])\n        if S < 3.0:\n            w = 0.25 + 0.25 * S; s2 = 0.0\n            for i in range(c): oi[i] = oi[i] ** w; s2 += oi[i]\n            if s2 > 0:\n                for i in range(c): oi[i] /= s2\n    return om, oi\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim(qmz, qp, cmz, cp, tol):\n    i = 0; j = 0; n = len(qmz); m = len(cmz)\n    SA = 0.0\n    for x in range(n):\n        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])\n    SB = 0.0\n    for x in range(m):\n        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])\n    SAB = 0.0; tot = 0.0\n    buf = np.empty(n + m, np.float64); b = 0\n    while i < n and j < m:\n        d = qmz[i] - cmz[j]\n        if d < -tol: buf[b] = qp[i]; i += 1; b += 1\n        elif d > tol: buf[b] = cp[j]; j += 1; b += 1\n        else: buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1\n    while i < n: buf[b] = qp[i]; i += 1; b += 1\n    while j < m: buf[b] = cp[j]; j += 1; b += 1\n    for x in range(b): tot += buf[x]\n    if tot <= 0: return 0.0\n    for x in range(b):\n        v = buf[x] / tot\n        if v > 0: SAB -= v * np.log(v)\n    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)\n\n\n@njit(cache=True, fastmath=True, parallel=True)\ndef search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]; a = off[c]; b = off[c + 1]\n        if b <= a: continue\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0: continue\n        out[k] = entropy_sim(qmz, qp, cm, cp, tol)\n    return out\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):\n    a = entropy_sim(qmz, qp, cmz, cp, tol)\n    if shift > -0.001 and shift < 0.001: return a\n    sm = np.empty(len(cmz), np.float32)\n    for i in range(len(cmz)): sm[i] = cmz[i] + shift\n    b = entropy_sim(qmz, qp, sm, cp, tol)\n    return a if a > b else b\n\n\n# ADDED: search over a pre-cleaned store (reps cleaned once instead of on every call)\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift_pre(qmz, qp, lo, hi, roff, rmz, rit, tol, shift):\n    out = np.zeros(hi - lo, np.float32)\n    for k in prange(hi - lo):\n        a = roff[lo + k]; b = roff[lo + k + 1]\n        if b <= a: continue\n        out[k] = entropy_sim_shift(qmz, qp, rmz[a:b], rit[a:b], tol, shift[k])\n    return out\n\n\n# ADDED: library search where each reference is shifted by (query precursor - reference precursor):\n# the same molecule measured as a different adduct keeps its neutral losses (modified-cosine idea)\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift_rows(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]; a = off[c]; b = off[c + 1]\n        if b <= a: continue\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0: continue\n        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])\n    return out\n\n\n# ADDED: clean a list of spectra into a flat store\n@njit(cache=True)\ndef clean_store(rows, off, allmz, allin, floor, topk, power, ent_weight):\n    n = len(rows)\n    lens = np.zeros(n + 1, np.int64)\n    tmp_m = []\n    tmp_i = []\n    for k in range(n):\n        r = rows[k]\n        cm, cp = _clean(allmz[off[r]:off[r + 1]], allin[off[r]:off[r + 1]], floor, topk, power, ent_weight)\n        tmp_m.append(cm); tmp_i.append(cp)\n        lens[k + 1] = lens[k] + len(cm)\n    om = np.empty(lens[n], np.float32); oi = np.empty(lens[n], np.float32)\n    for k in range(n):\n        om[lens[k]:lens[k + 1]] = tmp_m[k]; oi[lens[k]:lens[k + 1]] = tmp_i[k]\n    return lens, om, oi\n\n\n# ----------------------------------------------------------------------------------------------\n# adducts\n# ----------------------------------------------------------------------------------------------\nMASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,\n            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,\n            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)\nE = 0.00054857990; PROTON = MASS[\'H\'] - E; H2O = 2 * MASS[\'H\'] + MASS[\'O\']\nNH4 = MASS[\'N\'] + 4 * MASS[\'H\']; FORMATE = MASS[\'C\'] + 2 * MASS[\'H\'] + 2 * MASS[\'O\']\nACETATE = 2 * MASS[\'C\'] + 4 * MASS[\'H\'] + 2 * MASS[\'O\']\nADDUCTS = {\n    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E), "[M+Na]+": (1, 1, MASS[\'Na\'] - E),\n    "[M+K]+": (1, 1, MASS[\'K\'] - E), "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2 * H2O),\n    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),\n    "[M+CH3OH+H]+": (1, 1, PROTON + MASS[\'C\'] + 4 * MASS[\'H\'] + MASS[\'O\']),\n    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS[\'C\'] + 3 * MASS[\'H\'] + MASS[\'N\']),\n    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),\n    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS[\'Cl\'] + E), "[M]-": (1, 1, E),\n    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS[\'Na\'] - 2 * PROTON),\n    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS[\'Na\'] - E), "[2M+NH4]+": (2, 1, NH4 - E),\n    "[2M+K]+": (2, 1, MASS[\'K\'] - E), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),\n    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS[\'Na\'] - 2 * PROTON),\n    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),\n}\n\n\ndef neutral_mass(mz, adduct):\n    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)\n    for a, (n, z, d) in ADDUCTS.items():\n        m = (ad == a)\n        if m.any(): out[m] = (mz[m] * z - d) / n\n    return out\n\n\n# ----------------------------------------------------------------------------------------------\n# ranking features (31)\n# ----------------------------------------------------------------------------------------------\nN_ANALOG = 80\nP_SIM = 3.0\nN_FEAT = 31\n\n\ndef _rank_norm(x):\n    o = np.argsort(-x); r = np.empty(len(x)); r[o] = np.arange(len(x)); return r / max(1, len(x) - 1)\n\n\ndef _z(x):\n    s = x.std()\n    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)\n\n\ndef rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None):\n    nc = cand_fp.shape[0]\n    cf = cand_fp.astype(np.float32); cs = cf.sum(1)\n    lv = np.asarray(cand_lib, np.float32)\n    lvmax = float(lv.max()) if nc else 0.0\n    if analog_fp is not None and len(analog_sim):\n        af = analog_fp.astype(np.float32); asum = af.sum(1)\n        inter = cf @ af.T\n        tan = inter / (cs[:, None] + asum[None, :] - inter + 1e-9)\n        w = np.clip(np.asarray(analog_sim, np.float32), 0, None)\n        ap = (tan * (w ** P_SIM)[None, :]).max(1)\n        a1 = (tan * w[None, :]).max(1)\n        best_tan = tan.max(1); top_tan = tan[:, 0]; top_sim = float(w[0])\n        mean_tan = (tan * (w ** P_SIM)[None, :]).sum(1) / ((w ** P_SIM).sum() + 1e-9)\n    else:\n        ap = a1 = best_tan = top_tan = mean_tan = np.zeros(nc, np.float32); top_sim = 0.0\n    apmax = float(ap.max()) if nc else 0.0\n    if model_logits is not None:\n        raw = cf @ np.asarray(model_logits, np.float32)\n        nrm = raw / np.sqrt(np.maximum(cs, 1.0))\n        mfeat = [_z(raw), _rank_norm(raw), raw - raw.max(), _z(nrm), _rank_norm(nrm),\n                 (raw == raw.max()).astype(np.float32)]\n    else:\n        mfeat = [np.zeros(nc, np.float32)] * 6\n    if model_logits is not None and nc:\n        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))\n        lbest = int(np.argmax(lv)) if lvmax > 0 else -1\n        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0\n        abest = int(np.argmax(ap)) if apmax > 0 else -1\n        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0\n        xfeat = [lv * (1.0 - mr), ap * (1.0 - mr), np.full(nc, agree), np.full(nc, agree_a),\n                 np.full(nc, agree * lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0, 1]) if lv.std() > 1e-9 else 0.0)]\n    else:\n        xfeat = [np.zeros(nc, np.float32)] * 6\n    if frag is not None:\n        fr = np.asarray(frag, np.float32)\n        ffeat = [fr, _rank_norm(fr), fr - fr.max() if nc else fr, _z(fr)]\n    else:\n        ffeat = [np.zeros(nc, np.float32)] * 4\n    return np.column_stack([\n        lv, _rank_norm(lv), np.full(nc, lvmax), lv - lvmax, (lv > 0).astype(float),\n        ap, _rank_norm(ap), np.full(nc, apmax), ap - apmax,\n        a1, best_tan, top_tan, mean_tan, np.full(nc, top_sim),\n        np.full(nc, np.log(max(nc, 1))),\n        *mfeat, *ffeat, *xfeat,\n    ]).astype(np.float32)\n\n\n# ----------------------------------------------------------------------------------------------\n# MetFrag-lite\n# ----------------------------------------------------------------------------------------------\nAMU = {\'C\': 12.0, \'H\': 1.00782503207, \'N\': 14.0030740048, \'O\': 15.9949146196, \'P\': 30.97376163,\n       \'S\': 31.97207100, \'F\': 18.99840322, \'Cl\': 34.96885268, \'Br\': 78.9183371, \'I\': 126.904473,\n       \'Na\': 22.9897692809, \'K\': 38.96370668, \'Si\': 27.9769265325, \'B\': 11.0093054, \'Se\': 79.9165213}\nH = AMU[\'H\']\n\n\ndef mol_graph(smi):\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    m = Chem.MolFromSmiles(smi)\n    if m is None: return None\n    n = m.GetNumAtoms()\n    w = np.zeros(n)\n    for a in m.GetAtoms():\n        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H\n    if (w == 0).any(): return None\n    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]\n    return w, bonds, n\n\n\ndef _components(n, bonds, drop):\n    adj = [[] for _ in range(n)]\n    for i, (a, b) in enumerate(bonds):\n        if i in drop: continue\n        adj[a].append(b); adj[b].append(a)\n    seen = np.zeros(n, bool); comps = []\n    for s in range(n):\n        if seen[s]: continue\n        stack = [s]; seen[s] = True; cur = [s]\n        while stack:\n            u = stack.pop()\n            for v in adj[u]:\n                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)\n        comps.append(cur)\n    return comps\n\n\ndef fragment_masses(smi, max_breaks=2, max_bonds=34):\n    g = mol_graph(smi)\n    if g is None: return np.zeros(0)\n    w, bonds, n = g\n    nb = len(bonds)\n    if nb == 0 or nb > max_bonds: return np.array([w.sum()])\n    out = {w.sum()}\n    for i in range(nb):\n        for c in _components(n, bonds, {i}):\n            out.add(float(w[c].sum()))\n    if max_breaks >= 2:\n        for i in range(nb):\n            for j in range(i + 1, nb):\n                for c in _components(n, bonds, {i, j}):\n                    out.add(float(w[c].sum()))\n    return np.array(sorted(out))\n\n\ndef frag_masses_safe(smi):\n    try: return fragment_masses(smi)\n    except Exception: return np.zeros(0)\n\n\n# ADDED: charge carriers a fragment may keep, per precursor adduct (prvsiyan only used +-proton)\nCARRIERS = {\n    "[M+H]+": (PROTON,), "[M-H2O+H]+": (PROTON,), "[M-2H2O+H]+": (PROTON,),\n    "[M+NH4]+": (PROTON, NH4 - E), "[M+Na]+": (PROTON, MASS[\'Na\'] - E), "[M+K]+": (PROTON, MASS[\'K\'] - E),\n    "[M-H]-": (-PROTON,), "[M-H2O-H]-": (-PROTON,), "[M+CH2O2-H]-": (-PROTON, FORMATE - PROTON),\n    "[M+Cl]-": (-PROTON, MASS[\'Cl\'] + E),\n}\n\n\ndef explain_score_adduct(frag_mass, peak_mz, peak_int, adduct, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\n    car = CARRIERS.get(adduct, (PROTON,) if "+" in adduct[-2:] else (-PROTON,))\n    ion = np.sort(np.concatenate([frag_mass + dh * H + c for dh in h_shifts for c in car]))\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\n    if tot <= 0: return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)\n\n\ndef explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\n    ion = []\n    for dh in h_shifts:\n        ion.append(frag_mass + dh * H + (PROTON if mode > 0 else -PROTON))\n    ion = np.sort(np.concatenate(ion))\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\n    if tot <= 0: return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)', 'pv_fp.py': '"""Spectrum -> fingerprint model (prvsiyan FPNet), kept apart so CPU worker processes never import torch."""\nimport math\nimport numpy as np\n# ----------------------------------------------------------------------------------------------\n# spectrum -> fingerprint model\n# ----------------------------------------------------------------------------------------------\nimport torch, torch.nn as nn, torch.nn.functional as F\n\nFP_MAX_PEAKS = 128\nADDUCT_LIST = ["[M+H]+", "[M+NH4]+", "[M+Na]+", "[M+K]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M]+",\n               "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M+Cl]-", "[M]-",\n               "[M+2H]2+", "[M-2H]-", "[2M+H]+", "[2M+Na]+", "[2M+NH4]+", "[2M-H]-", "[2M+K]+",\n               "[2M+CH2O2-H]-", "[2M+C2H4O2-H]-", "[2M+Na-2H]-", "[M+Na-2H]-", "[M-H2O]+", "<unk>"]\nADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}\nINSTR_LIST = ["timsTOF", "Orbitrap", "QTOF", "IT", "other"]\n\n\ndef instr_family(s):\n    if s is None: return 4\n    t = str(s).lower()\n    if \'timstof\' in t: return 0\n    if \'orbitrap\' in t or \'qft\' in t or \'ftms\' in t or \'hybrid ft\' in t or \'itft\' in t or \'exactive\' in t: return 1\n    if \'tof\' in t: return 2\n    if \'trap\' in t or \'qq\' in t: return 3\n    return 4\n\n\ndef prep_peaks(mz, inten, prec_mz, max_peaks=FP_MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):\n    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = (mz <= prec_mz + 1.5)\n    mz, it = mz[keep], it[keep]\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    mx = it.max()\n    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = it >= floor * mx\n    mz, it = mz[keep], it[keep]\n    if len(mz) > max_peaks:\n        order = np.argsort(-it)\n        bucket = (mz // win).astype(np.int64)\n        cnt = {}; sel = []\n        for i in order:\n            b = bucket[i]; c = cnt.get(b, 0)\n            if c < per_win: cnt[b] = c + 1; sel.append(i)\n        sel = np.array(sel)\n        if len(sel) > max_peaks:\n            sel = sel[np.argsort(-it[sel])[:max_peaks]]\n        elif len(sel) < max_peaks:\n            rest = np.array([i for i in order if i not in set(sel.tolist())])\n            need = max_peaks - len(sel)\n            if len(rest): sel = np.concatenate([sel, rest[:need]])\n        mz, it = mz[sel], it[sel]\n    o = np.argsort(mz)\n    mz, it = mz[o], it[o]\n    v = np.sqrt(it / it.max())\n    return mz.astype(np.float32), v.astype(np.float32)\n\n\nclass SinEmb(nn.Module):\n    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):\n        super().__init__()\n        n = dim // 2\n        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)\n        self.register_buffer(\'inv\', (2 * math.pi) / wav)\n\n    def forward(self, x):\n        a = x.unsqueeze(-1) * self.inv\n        return torch.cat([torch.sin(a), torch.cos(a)], -1)\n\n\nclass Block(nn.Module):\n    def __init__(self, d, h, drop):\n        super().__init__(); self.h = h\n        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)\n        self.n2 = nn.LayerNorm(d)\n        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))\n        self.drop = nn.Dropout(drop)\n\n    def forward(self, x, pad):\n        B, N, D = x.shape; y = self.n1(x)\n        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)\n        m = (~pad)[:, None, None, :]\n        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)\n        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))\n        return x + self.drop(self.ff(self.n2(x)))\n\n\nclass FPNet(nn.Module):\n    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):\n        super().__init__()\n        self.d = d\n        self.mz_emb = SinEmb(d)\n        self.nl_emb = SinEmb(d)\n        self.pk = nn.Linear(2 * d + 1, d)\n        self.prec_emb = SinEmb(d)\n        self.ad = nn.Embedding(len(ADDUCT_LIST), d)\n        self.ins = nn.Embedding(len(INSTR_LIST), d)\n        self.gl = nn.Linear(d + 3, d)\n        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])\n        self.norm = nn.LayerNorm(d)\n        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))\n\n    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):\n        B, N = mz.shape\n        nl = (prec[:, None] - mz).clamp(min=0)\n        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))\n        g = self.gl(torch.cat([self.prec_emb(prec),\n                               (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),\n                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)\n        x = torch.cat([g.unsqueeze(1), p], 1)\n        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)\n        for b in self.blocks: x = b(x, pad)\n        x = self.norm(x)\n        cls = x[:, 0]\n        msk = (~pad[:, 1:]).float().unsqueeze(-1)\n        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)\n        return self.head(torch.cat([cls, mean], -1))\n\n\ndef load_fp_models(paths, dev):\n    single, merged = [], []\n    nbits = None\n    for pth in paths:\n        ck = torch.load(pth, map_location=\'cpu\', weights_only=False)\n        net = FPNet(ck[\'nbits\'], d=ck[\'d\'], layers=ck[\'layers\']).to(dev).eval()\n        net.load_state_dict(ck[\'model\'])\n        (merged if \'merged\' in str(pth).replace(\'\\\\\', \'/\').split(\'/\')[-1] else single).append(net)\n        nbits = ck[\'nbits\']\n    return single, merged, nbits\n\n\ndef merge_peaks(specs):\n    """specs: list of (mz, it). All peaks collapsed into one pseudo-spectrum."""\n    specs = [(a, b) for a, b in specs if len(a)]\n    if not specs: return np.zeros(0), np.zeros(0)\n    mz = np.concatenate([np.asarray(a, float) for a, _ in specs])\n    it = np.concatenate([np.asarray(b, float) / max(float(np.asarray(b, float).max()), 1e-9) for _, b in specs])\n    o = np.argsort(mz); mz, it = mz[o], it[o]\n    keep = np.ones(len(mz), bool)\n    for j in range(1, len(mz)):\n        if mz[j] - mz[j - 1] < 0.005:\n            if it[j] >= it[j - 1]: keep[j - 1] = False\n            else: keep[j] = False\n    return mz[keep], it[keep]\n\n\n@torch.no_grad()\ndef logits_batch(P, nets, dev, prec, adduct, instr, ce, mode):\n    """P: list of (mz, it) already prepped; per-row arrays prec/adduct(str)/instr(str)/ce/mode."""\n    keep = [i for i, (a, _) in enumerate(P) if len(a)]\n    if not keep: return None\n    P = [P[i] for i in keep]\n    B = len(P); N = max(len(a) for a, _ in P)\n    mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)\n    for i, (a, b) in enumerate(P):\n        mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False\n    T = lambda x: torch.as_tensor(x, device=dev)\n    args = (T(mz), T(it), T(pad),\n            T(np.asarray(prec, np.float32)[keep]),\n            T(np.array([ADDUCT_IX.get(a, ADDUCT_IX[\'<unk>\']) for a in np.asarray(adduct, object)[keep]])),\n            T(np.array([instr_family(s) for s in np.asarray(instr, object)[keep]])),\n            T(np.asarray(ce, np.float32)[keep]),\n            T(np.asarray(mode, np.float32)[keep]))\n    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)\n\n\ndef molecule_logits(models, specs, prec, adduct, instr, ce, mode):\n    """prvsiyan model_logits(): mean of (single-model per-spectrum mean) and (merged-model on merged list).\n    specs: list of (mz, it); prec/adduct/instr/ce/mode: per-spectrum."""\n    single, merged, dev = models\n    out = []\n    if single:\n        P = [prep_peaks(a, b, p) for (a, b), p in zip(specs, prec)]\n        za = logits_batch(P, single, dev, prec, adduct, instr, ce, mode)\n        if za is not None: out.append(za)\n    if merged:\n        mz, it = merge_peaks(specs)\n        pm = float(np.median(prec))\n        P = [prep_peaks(mz, it, pm)]\n        zb = logits_batch(P, merged, dev, [pm], [adduct[0]], [instr[0]], [25.0], [float(np.mean(mode))])\n        if zb is not None: out.append(zb)\n    return np.mean(out, axis=0) if out else None', 'casmi_engine.py': '"""End-to-end inference (Kaggle notebook body). Mirrors e02_channels.py + e05_rank.py feature blocks exactly,\nbut reads train.parquet / test.parquet directly. Runs locally too (set CASMI_LOCAL=1).\n\nStages: library -> candidate pool -> index -> per-molecule channels -> MetFrag-lite -> features -> ranker -> submission\n"""\nimport os, sys, glob, time, pickle, math\nos.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\nimport numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa\nimport pv\nfrom pv import CFG\nimport re as _re\n_MONO = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, S=31.97207100, P=30.97376163,\n             Cl=34.96885268, Br=78.9183371, F=18.99840322, I=126.904473, Si=27.9769265325, Se=79.9165213,\n             Na=22.9897692809, K=38.96370668, B=11.0093054, As=74.9215965, Fe=55.9349375, D=2.0141017778)\n_TOK = _re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef formula_mass(f):\n    """ADDED: neutral monoisotopic mass by arithmetic on molecular_formula (prvsiyan fix; immune to\n    riken\'s 0.005 Da precursor precision and gnps adduct mislabels). NaN if an element is untabulated."""\n    if not isinstance(f, str) or not f: return np.nan\n    m = 0.0\n    for el, n in _TOK.findall(f):\n        if el not in _MONO: return np.nan\n        m += _MONO[el] * (int(n) if n else 1)\n    return m\n\nPOOL = None\n\n\nclass RANK:\n    W_A = (0.35, 0.55)      # weight on class-1 simulation rows; two priors averaged\n    SEEDS = (0, 1)\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\n\nT0 = time.time()\nLOCAL = os.environ.get("CASMI_LOCAL") == "1"\nROOTS = [r"C:\\Users\\HW-LEE\\Desktop\\CASMI"] if LOCAL else ["/kaggle/input"]\n\n\ndef find(name):\n    for root in ROOTS:\n        hits = glob.glob(os.path.join(root, "**", name), recursive=True)\n        if hits: return sorted(hits, key=len)[0]\n    raise FileNotFoundError(name)\n\n\ndef log(msg):\n    print(f"[{time.time()-T0:6.0f}s] {msg}", flush=True)\n\n\n# ---------------------------------------------------------------------------------------------\n# library\n# ---------------------------------------------------------------------------------------------\ndef load_library(path):\n    """Row group by row group, peaks straight to float32: peak RAM stays near the final ~3.5 GB."""\n    f = pq.ParquetFile(path)\n    n_rows = f.metadata.num_rows\n    npk = 0\n    for i in range(f.num_row_groups):\n        c = f.read_row_group(i, columns=["ms2_mzs"]).column(0).combine_chunks()\n        npk += len(c.values)\n    off = np.zeros(n_rows + 1, np.int64)\n    allmz = np.empty(npk, np.float32); allin = np.empty(npk, np.float32)\n    prec, add, pol, ik, smi = [], [], [], [], []\n    fo, tims = [], []\n    r0 = p0 = 0\n    for i in range(f.num_row_groups):\n        t = f.read_row_group(i, columns=["inchikey14", "normalized_smiles", "adduct", "precursor_mz", "ionization_mode",\n                                         "ms2_mzs", "ms2_normalized_intensities", "molecular_formula", "instrument_type"])\n        mzc = t.column("ms2_mzs").combine_chunks(); itc = t.column("ms2_normalized_intensities").combine_chunks()\n        o = mzc.offsets.to_numpy().astype(np.int64)\n        n = len(o) - 1; k = int(o[-1] - o[0])\n        allmz[p0:p0 + k] = mzc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\n        allin[p0:p0 + k] = itc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\n        off[r0 + 1:r0 + n + 1] = p0 + (o[1:] - o[0])\n        prec.append(t.column("precursor_mz").to_numpy(zero_copy_only=False).astype(np.float64))\n        add += t.column("adduct").cast(pa.string()).to_pylist()\n        pol.append(np.array(t.column("ionization_mode").cast(pa.string()).to_pylist(), dtype=object) == "positive")\n        ik += t.column("inchikey14").cast(pa.string()).to_pylist()\n        smi += t.column("normalized_smiles").cast(pa.string()).to_pylist()\n        fo += t.column("molecular_formula").cast(pa.string()).to_pylist()\n        tims.append(np.array([isinstance(x, str) and x.lower().replace("-", "").replace(" ", "") == "timstof"\n                              for x in t.column("instrument_type").cast(pa.string()).to_pylist()]))\n        r0 += n; p0 += k\n        del t, mzc, itc\n    prec = np.concatenate(prec); add = np.asarray(add, dtype=object)\n    pol = np.where(np.concatenate(pol), 1, -1).astype(np.int8)\n    nmp = pv.neutral_mass(prec, add)\n    _fm = {x: formula_mass(x) for x in set(v for v in fo if isinstance(v, str))}\n    nmf = np.array([_fm.get(x, np.nan) if isinstance(x, str) else np.nan for x in fo])\n    nm = np.where(np.isfinite(nmf), nmf, nmp)\n    tims = np.concatenate(tims)\n    log(f"library mass index: formula {np.isfinite(nmf).mean():.1%}, timsTOF rows {tims.mean():.1%}")\n    del fo\n    return dict(off=off, mz=allmz, it=allin, prec=prec, nm=nm, pol=pol, tims=tims,\n                ik=np.asarray(ik, dtype=object), smi=np.asarray(smi, dtype=object))\n\n\n# ---------------------------------------------------------------------------------------------\n# candidate pool = COCONUT (prvsiyan fingerprints) U training structures (fingerprinted here)\n# ---------------------------------------------------------------------------------------------\n_g = {}\n\n\ndef _fp_init():\n    from rdkit import RDLogger\n    from rdkit.Chem import rdFingerprintGenerator\n    RDLogger.DisableLog("rdApp.*")\n    _g["bits"] = np.load(find("fp_bits.npy"))\n    _g["m2"] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)\n    _g["m3"] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)\n    _g["rk"] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)\n\n\ndef fp_and_mass(smi):\n    from rdkit import Chem\n    from rdkit.Chem import MACCSkeys\n    from rdkit.Chem.Descriptors import ExactMolWt\n    if not _g: _fp_init()\n    m = Chem.MolFromSmiles(smi)\n    if m is None: return None\n    try:\n        fp = np.concatenate([_g["m2"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             _g["m3"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             _g["rk"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[_g["bits"]]\n        return np.packbits(fp), float(ExactMolWt(m))\n    except Exception:\n        return None\n\n\ndef canon_key(smi):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    RDLogger.DisableLog("rdApp.*")\n    if "te" not in _g: _g["te"] = rdMolStandardize.TautomerEnumerator()\n    try:\n        m = Chem.MolFromSmiles(smi)\n        return Chem.MolToInchiKey(_g["te"].Canonicalize(m))[:14] if m is not None else None\n    except Exception:\n        return None\n\n\ndef build_pool(L, workers):\n    from multiprocessing import Pool as MPool\n    d = os.path.dirname(find("coco_fp.npy"))\n    cm = pickle.load(open(os.path.join(d, "coco_meta.pkl"), "rb"))\n    co_fp = np.load(os.path.join(d, "coco_fp.npy")); co_mass = np.load(os.path.join(d, "coco_mass.npy"))\n    co_keys = np.asarray(cm["keys"], dtype=object); co_smi = np.asarray(cm["smiles"], dtype=object)\n    # ADDED: ChEBI + LIPID MAPS (prvsiyan/chebi-lipidmaps-casmi26), same 6,930-bit packed fingerprints\n    bd = os.path.dirname(find("bio_fp.npy")); bm = pickle.load(open(os.path.join(bd, "bio_meta.pkl"), "rb"))\n    bkeys = np.asarray(bm["keys"], dtype=object); bnew = ~pd.Series(bkeys).isin(set(co_keys)).values\n    co_fp = np.vstack([co_fp, np.load(os.path.join(bd, "bio_fp.npy"))[bnew]])\n    co_mass = np.concatenate([co_mass, np.load(os.path.join(bd, "bio_mass.npy"))[bnew]])\n    co_keys = np.concatenate([co_keys, bkeys[bnew]])\n    co_smi = np.concatenate([co_smi, np.asarray(bm["smiles"], dtype=object)[bnew]])\n    log(f"+ ChEBI/LIPID MAPS {int(bnew.sum()):,} new structures")\n    tr = pd.DataFrame({"ik": L["ik"], "smi": L["smi"]}).dropna().drop_duplicates("ik")\n    tr = tr[~tr.ik.isin(set(co_keys))]\n    log(f"COCONUT {len(co_keys):,}; training structures to fingerprint {len(tr):,}")\n    with MPool(workers) as mp:\n        res = mp.map(fp_and_mass, list(tr.smi), chunksize=500)\n    ok = np.array([r is not None for r in res])\n    tr_fp = np.stack([r[0] for r in res if r is not None]); tr_mass = np.array([r[1] for r in res if r is not None])\n    fp = np.vstack([co_fp, tr_fp]); mass = np.concatenate([co_mass, tr_mass])\n    keys = np.concatenate([co_keys, tr.ik.values[ok]]); smis = np.concatenate([co_smi, tr.smi.values[ok]])\n    good = np.isfinite(mass)\n    o = np.argsort(np.where(good, mass, 1e18), kind="mergesort")[: good.sum()]\n    P = dict(fp=fp[o], mass=mass[o], keys=keys[o], smiles=smis[o])\n    P["k2i"] = pd.Series(np.arange(len(o)), index=P["keys"]); P["k2i"] = P["k2i"][~P["k2i"].index.duplicated()]\n    log(f"pool {len(o):,} structures")\n    return P\n\n\ndef pool_fps(idx):\n    return np.unpackbits(np.asarray(POOL["fp"][np.asarray(idx, np.int64)]), axis=1)[:, :6930]\n\n\ndef pool_window(t, ppm):\n    a = np.searchsorted(POOL["mass"], t * (1 - ppm / 1e6), "left")\n    b = np.searchsorted(POOL["mass"], t * (1 + ppm / 1e6), "right")\n    return np.arange(a, b)\n\n\n# ---------------------------------------------------------------------------------------------\n# index: neutral-mass sorted library rows + cleaned representatives (all / positive / negative)\n# ---------------------------------------------------------------------------------------------\ndef _reps(L, rows):\n    npk = np.diff(L["off"])[rows]\n    df = pd.DataFrame({"row": rows, "p": L["row_p"][rows], "npk": npk, "tims": L["tims"][rows]})\n    # ADDED: prefer the test instrument (timsTOF) representative, then the richest spectrum\n    df = df.sort_values(["p", "tims", "npk", "row"], ascending=[True, False, False, True]).drop_duplicates("p")\n    rep = df.row.values; rep_nm = L["nm"][rep]\n    o = np.argsort(rep_nm, kind="mergesort"); rep = rep[o]\n    roff, rmz, rit = pv.clean_store(rep, L["off"], L["mz"], L["it"], CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\n    return dict(rep=rep, rep_p=L["row_p"][rep], rep_nm=L["nm"][rep], roff=roff, rmz=rmz, rit=rit)\n\n\ndef build_index(L):\n    ok = np.isfinite(L["nm"]) & (L["row_p"] >= 0)\n    rows = np.where(ok)[0]\n    o = np.argsort(L["nm"][rows], kind="mergesort")\n    I = dict(lib_rows=rows[o], lib_nm=L["nm"][rows[o]])\n    I["all"] = _reps(L, rows)\n    I["pos"] = _reps(L, rows[L["pol"][rows] == 1])\n    I["neg"] = _reps(L, rows[L["pol"][rows] == -1])\n    log(f"index: {len(rows):,} library rows; representatives {len(I[\'all\'][\'rep\']):,}")\n    return I\n\n\n# ---------------------------------------------------------------------------------------------\n# per-molecule channels (identical to e02_channels.molecule_channels)\n# ---------------------------------------------------------------------------------------------\ndef clean(mz, it):\n    return pv._clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),\n                     CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\n\n\ndef analog_search(R, Q, target):\n    lo = np.searchsorted(R["rep_nm"], target - CFG.ANALOG_WIN, "left")\n    hi = np.searchsorted(R["rep_nm"], target + CFG.ANALOG_WIN, "right")\n    asim = np.zeros(hi - lo, np.float32)\n    if hi > lo and Q:\n        shift = (target - R["rep_nm"][lo:hi]).astype(np.float32)\n        for qm, qp in Q:\n            asim = np.maximum(asim, pv.search_shift_pre(qm, qp, lo, hi, R["roff"], R["rmz"], R["rit"], CFG.MZ_TOL, shift))\n    return R["rep_p"][lo:hi], asim\n\n\ndef top_analogs(p, s):\n    if len(p) == 0: return []\n    df = pd.DataFrame({"p": p, "s": s}).groupby("p", sort=False).s.max().sort_values(ascending=False, kind="mergesort")\n    return list(zip(df.index[:CFG.N_ANALOG].astype(int), df.values[:CFG.N_ANALOG].astype(float)))\n\n\ndef molecule_channels(L, I, specs, precs, pols, target):\n    Q = [clean(a, b) for a, b in specs]\n    keep = [i for i, (a, _) in enumerate(Q) if len(a)]\n    Q = [Q[i] for i in keep]; precs = np.asarray(precs)[keep]; pols = np.asarray(pols)[keep]\n    tol = target * CFG.PPM_WIN / 1e6\n    lo = np.searchsorted(I["lib_nm"], target - tol, "left"); hi = np.searchsorted(I["lib_nm"], target + tol, "right")\n    crow = I["lib_rows"][lo:hi]\n    u, inv = np.unique(L["row_p"][crow], return_inverse=True)\n    M = np.zeros((max(len(Q), 1), len(u)), np.float32); MS = np.zeros_like(M)\n    for j, (qm, qp) in enumerate(Q):\n        if len(crow) == 0: break\n        s = pv.search(qm, qp, crow, L["off"], L["mz"], L["it"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\n                      CFG.INT_POWER, CFG.ENT_WEIGHT)\n        np.maximum.at(M[j], inv, s)\n        sh = (precs[j] - L["prec"][crow]).astype(np.float32)\n        s2 = pv.search_shift_rows(qm, qp, crow, L["off"], L["mz"], L["it"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\n                                  CFG.INT_POWER, CFG.ENT_WEIGHT, sh)\n        np.maximum.at(MS[j], inv, s2)\n    cnt = np.bincount(inv, minlength=len(u)).astype(np.float32)\n    z = np.zeros(0, np.float32)\n    lib = (u, M.max(0) if len(u) else z, M.mean(0) if len(u) else z, MS.max(0) if len(u) else z, cnt)\n    p, s = analog_search(I["all"], Q, target)\n    ana = top_analogs(p, s)\n    ps, ss = [], []\n    for sign, name in ((1, "pos"), (-1, "neg")):\n        QQ = [q for q, pl in zip(Q, pols) if pl == sign]\n        if QQ:\n            p, s = analog_search(I[name], QQ, target); ps.append(p); ss.append(s)\n    anapol = top_analogs(np.concatenate(ps), np.concatenate(ss)) if ps else []\n    return lib, ana, anapol\n\n\ndef parse_ce(v):\n    try:\n        a = np.abs(np.atleast_1d(np.asarray(v, float)))\n        return float(a.mean()) if len(a) else 25.0\n    except Exception:\n        return 25.0\n\n\n\n\n# ---------------------------------------------------------------------------------------------\n# ranking features: prvsiyan\'s 31 + our blocks (identical to e05_rank.py BLOCKS)\n# ---------------------------------------------------------------------------------------------\ndef _rel(v):\n    v = np.asarray(v, np.float32)\n    return [v, pv._rank_norm(v).astype(np.float32), v - (v.max() if len(v) else 0.0)]\n\n\ndef _analog_feats(cfp, ana):\n    nc = len(cfp)\n    if not ana: return np.zeros((nc, 5), np.float32)\n    afp = pool_fps([p for p, _ in ana]).astype(np.float32); cf = cfp.astype(np.float32)\n    inter = cf @ afp.T\n    tan = inter / (cf.sum(1)[:, None] + afp.sum(1)[None, :] - inter + 1e-9)\n    w = np.clip(np.array([s for _, s in ana], np.float32), 0, None)\n    ap = (tan * (w ** 3)[None, :]).max(1)\n    ap6 = (tan * (w ** 6)[None, :]).max(1)\n    return np.column_stack(_rel(ap) + [ap6, np.full(nc, w[0])]).astype(np.float32)\n\n\ndef blk_base(r, cfp):\n    ana = r["ana"]\n    afp = pool_fps([p for p, _ in ana]) if ana else None\n    sims = np.array([s for _, s in ana], np.float32)\n    return pv.rank_features(cfp, r["lib"], afp, sims, r["zlog"], r["frag"])\n\n\ndef blk_mass(r, cfp):\n    ppm = (POOL["mass"][r["cand"]] - r["target"]) / r["target"] * 1e6\n    return np.column_stack([ppm, np.abs(ppm), np.abs(ppm + 1.5)]).astype(np.float32)\n\n\ndef blk_libx(r, cfp):\n    cnt = np.log1p(r["lib_cnt"])\n    return np.column_stack(_rel(r["lib_mean"]) + _rel(r["libsh"]) + [cnt, cnt - cnt.max()]).astype(np.float32)\n\n\ndef blk_anapol(r, cfp):\n    return _analog_feats(cfp, r["anapol"])\n\n\ndef blk_fragx(r, cfp):\n    return np.column_stack(_rel(r["frag_ad"]) + _rel(r["frag_mean"]) + [r["frag_pol"]]).astype(np.float32)\n\n\nBLOCKS = {"base": blk_base, "mass": blk_mass, "libx": blk_libx, "anapol": blk_anapol, "fragx": blk_fragx}\n\n\n# ---------------------------------------------------------------------------------------------\n# run\n# ---------------------------------------------------------------------------------------------\ndef compute_channels(L, I, models, te):\n    mols = list(te.groupby("molecule_id", sort=True))\n    recs = []\n    for gi, (mid, sub) in enumerate(mols):\n        specs = []\n        for r in sub.itertuples():\n            a_ = np.asarray(r.ms2_mzs, np.float32); b_ = np.asarray(r.ms2_normalized_intensities, np.float32)\n            k_ = a_ <= float(r.precursor_mz) + 2.0\n            a_, b_ = a_[k_], b_[k_]\n            specs.append((a_, b_ / b_.max() if len(b_) and b_.max() > 0 else b_))\n        nms = pv.neutral_mass(sub.precursor_mz.values.astype(np.float64), sub.adduct.astype(str).values)\n        nms = nms[np.isfinite(nms)]\n        rec = dict(mid=mid, cand=np.zeros(0, np.int64))\n        if len(nms):\n            target = float(np.median(nms))\n            modes = np.where(sub.ionization_mode.astype(str).values == "positive", 1.0, -1.0)\n            lib, ana, anapol = molecule_channels(L, I, specs, sub.precursor_mz.values.astype(np.float64), modes, target)\n            cand = pool_window(target, CFG.PPM_WIN)\n            if len(cand) == 0: cand = pool_window(target, CFG.PPM_FALLBACK)\n            ces = np.array([parse_ce(v) for v in sub.collision_energy_ev.values], np.float32)\n            import pv_fp\n            zlog = pv_fp.molecule_logits(models, specs, sub.precursor_mz.values.astype(np.float32),\n                                         list(sub.adduct.astype(str).values), list(sub.instrument_type.astype(str).values),\n                                         ces, modes)\n            u, lmax, lmean, lsh, lcnt = lib\n            pos = {int(x): i for i, x in enumerate(u)}\n            ci = np.array([pos.get(int(c), -1) for c in cand], np.int64); has = ci >= 0\n            rec.update(target=target, cand=cand, zlog=None if zlog is None else zlog.astype(np.float32), specs=specs,\n                       mode=float(np.mean(modes)), modes=modes, adducts=list(sub.adduct.astype(str).values),\n                       ana=ana, anapol=anapol)\n            for nm_, arr in (("lib", lmax), ("lib_mean", lmean), ("libsh", lsh), ("lib_cnt", lcnt)):\n                v = np.zeros(len(cand), np.float32); v[has] = arr[ci[has]]; rec[nm_] = v\n        recs.append(rec)\n        if gi % 50 == 0: log(f"  channels {gi}/{len(mols)}")\n    return mols, recs\n\n\ndef compute_frag(recs, workers):\n    from multiprocessing import Pool as MPool\n    uc = np.unique(np.concatenate([r["cand"] for r in recs]))\n    log(f"MetFrag-lite on {len(uc):,} candidates")\n    with MPool(workers) as mp:\n        fr = mp.map(pv.frag_masses_safe, list(POOL["smiles"][uc]), chunksize=16)\n    fmap = dict(zip(uc.tolist(), fr))\n    for r in recs:\n        if not len(r["cand"]): continue\n        peaks = []\n        for a_, b_ in r.pop("specs"):\n            m2, i2 = pv._clean(a_, b_, CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)\n            peaks.append((np.asarray(m2, float), np.asarray(i2, float)))\n        F1 = np.zeros((len(r["cand"]), max(len(peaks), 1)), np.float32); F2 = np.zeros_like(F1); F3 = np.zeros_like(F1)\n        for j, c in enumerate(r["cand"]):\n            fm = fmap[int(c)]\n            for k, (x, y) in enumerate(peaks):\n                F1[j, k] = pv.explain_score(fm, x, y, mode=r["mode"], tol=CFG.MZ_TOL)\n                F2[j, k] = pv.explain_score_adduct(fm, x, y, r["adducts"][k], tol=CFG.MZ_TOL)\n                F3[j, k] = pv.explain_score(fm, x, y, mode=r["modes"][k], tol=CFG.MZ_TOL)\n        r["frag"] = F1.max(1); r["frag_ad"] = F2.max(1); r["frag_mean"] = F3.mean(1); r["frag_pol"] = F3.max(1)\n\n\ndef _hgb(X, Y, W, seed, gbm):\n    from sklearn.ensemble import HistGradientBoostingClassifier\n    m = HistGradientBoostingClassifier(random_state=seed, **gbm); m.fit(X, Y, sample_weight=W)\n    return m\n\n\ndef fit_rankers(rank_train_path):\n    """Our ranker: simulation rows from 7 query sets (S = 0 class-1 sim, 1 = class-2 sim)."""\n    z = np.load(rank_train_path, allow_pickle=True)\n    X, Y, S = z["X"], z["Y"], z["S"]\n    rankers = [_hgb(X, Y, np.where(S == 0, wA, 1.0 - wA), sd, RANK.GBM) for wA in RANK.W_A for sd in RANK.SEEDS]\n    log(f"our ranker: {len(rankers)} GBMs on {X.shape[0]:,} rows x {X.shape[1]} features")\n    return rankers, [str(b) for b in z["blocks"]]\n\n\ndef fit_pv_rankers(pv_rank_path, priors=(0.30, 0.60), seeds=(0, 1, 2, 3)):\n    """prvsiyan\'s ranker, exactly as in his notebook: shipped rank_train.npz (M = 0 class-1 sim)."""\n    z = np.load(pv_rank_path)\n    rankers = [_hgb(z["X"], z["Y"], np.where(z["M"] == 0, w1, 1.0 - w1), sd, CFG.GBM) for w1 in priors for sd in seeds]\n    log(f"prvsiyan ranker: {len(rankers)} GBMs on {z[\'X\'].shape[0]:,} rows x {z[\'X\'].shape[1]} features")\n    return rankers, z["X"].shape[1]\n\n\ndef score_molecules(recs, rankers, blocks, use_fp):\n    out = {}\n    for r in recs:\n        if not len(r["cand"]): continue\n        rr = r if use_fp else dict(r, zlog=None)\n        cfp = pool_fps(r["cand"])\n        Xm = np.hstack([BLOCKS[b](rr, cfp) for b in blocks]).astype(np.float32)\n        out[r["mid"]] = np.mean([m.predict_proba(Xm)[:, 1] for m in rankers], axis=0)\n    return out\n\n\ndef blend_scores(a, b, wa):\n    """Within-molecule rank blend (both inputs are per-candidate probabilities)."""\n    out = {}\n    for mid in a:\n        ra = 1.0 - pv._rank_norm(a[mid]); rb = 1.0 - pv._rank_norm(b[mid])\n        out[mid] = wa * ra + (1.0 - wa) * rb + 1e-6 * a[mid]\n    return out\n\n\ndef make_submissions(mols, recs, score_sets, sample_path, workers, topn=25):\n    """score_sets: {name: {mid: scores}} -> {name: submission DataFrame}; one canonicalisation pass for all."""\n    from multiprocessing import Pool as MPool\n    cand = {r["mid"]: r["cand"] for r in recs if len(r["cand"])}\n    ordered = {n: {mid: cand[mid][np.argsort(-sc, kind="mergesort")][:topn + 15] for mid, sc in S.items()}\n               for n, S in score_sets.items()}\n    allc = [v for o in ordered.values() for v in o.values()]\n    short = np.unique(np.concatenate(allc)) if allc else np.zeros(0, np.int64)\n    with MPool(workers) as mp:\n        ck = mp.map(canon_key, list(POOL["smiles"][short]), chunksize=64)\n    ckey = dict(zip(short.tolist(), ck))\n    samp = pd.read_csv(sample_path)\n    subs = {}\n    for n, o in ordered.items():\n        rows = []\n        for mid, _ in mols:\n            out, seen = [], set()\n            for c in o.get(mid, []):\n                k = ckey.get(int(c)) or POOL["keys"][c]\n                if k in seen: continue\n                seen.add(k); out.append(POOL["smiles"][c])\n                if len(out) == topn: break\n            while len(out) < topn:\n                out.append("CCO")\n            rows.append((mid, ";".join(out[:topn])))\n        sub = samp[["molecule_id"]].merge(pd.DataFrame(rows, columns=["molecule_id", "smiles"]), on="molecule_id", how="left")\n        sub["smiles"] = sub["smiles"].fillna("CCO")\n        assert len(sub) == len(samp) and sub.molecule_id.duplicated().sum() == 0 and sub.smiles.isnull().sum() == 0\n        assert (sub.smiles.str.split(";").map(len) <= topn).all()\n        subs[n] = sub\n    return subs\n\n\ndef main(test_path, train_path, sample_path, our_rank_path, pv_rank_path, fp_model_paths, workers=4, limit=0, w_pv=0.88):\n    global POOL\n    import torch\n    import pv_fp\n    L = load_library(train_path); log(f"library {len(L[\'prec\']):,} spectra")\n    POOL = build_pool(L, workers)\n    L["row_p"] = POOL["k2i"].reindex(L["ik"]).fillna(-1).values.astype(np.int64)\n    del L["ik"], L["smi"]\n    I = build_index(L)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    single, merged, _ = pv_fp.load_fp_models(fp_model_paths, dev)\n    log(f"fp models: {len(single)} single + {len(merged)} merged on {dev}")\n    te = pq.read_table(test_path).to_pandas()\n    if limit:\n        keep = sorted(te.molecule_id.unique())[:limit]; te = te[te.molecule_id.isin(keep)]\n    log(f"test: {len(te):,} spectra / {te.molecule_id.nunique():,} molecules")\n    mols, recs = compute_channels(L, I, (single, merged, dev), te)\n    del L, I\n    compute_frag(recs, workers)\n    ours, blocks = fit_rankers(our_rank_path)\n    s_ours = score_molecules(recs, ours, blocks, use_fp=False)\n    del ours\n    pvr, nfeat = fit_pv_rankers(pv_rank_path)\n    s_pv = score_molecules(recs, pvr, ["base"], use_fp=True)\n    del pvr\n    subs = make_submissions(mols, recs, {"blend": blend_scores(s_pv, s_ours, w_pv), "pv": s_pv, "ours": s_ours},\n                            sample_path, workers)\n    log("submissions ready")\n    return subs, recs, {"pv": s_pv, "ours": s_ours}\n', 'eng_runner.py': '"""Second candidate engine (prvsiyan analog-propagation pipeline as packaged in the public fusion notebook by\nseyitkaangunes) -> {molecule_id: {smiles: [...], keys: [...]}}: blended order, up to 40 scored identities.\n\nusage: python eng_runner.py <test.parquet> <train.parquet> <sample_submission.csv> <out.json>\n"""\nimport os, sys, json, time\nimport numpy as np\nif __name__ == \'__main__\':\n    here = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, here)\n    test, train, sample, out = sys.argv[1:5]\n    import casmi_engine as E\n    E.ROOTS = [os.environ.get(\'CASMI_INPUT_ROOT\', \'/kaggle/input\')]\n    _orig_find = E.find\n    def _find(n):                                  # the main pipeline has its own fp_bits.npy (10,226 bits): take the 6,930-bit file\n        if n == \'fp_bits.npy\':\n            return os.path.join(os.path.dirname(_orig_find(\'coco_fp.npy\')), \'fp_bits.npy\')\n        return _orig_find(n)\n    E.find = _find\n    E.RANK.W_A = (0.35, 0.55); E.RANK.SEEDS = (0, 1); E.CFG.N_ANALOG = 200\n    fp_models = [_orig_find(\'fp_merged_m1.pt\'), _orig_find(\'fp_single_s2.pt\')]\n    print(\'engine fp models\', fp_models, flush=True)\n    T0 = time.time()\n    subs, recs, S = E.main(test, train, sample, E.find(\'sim_rank_rows_nofp.npz\'), E.find(\'rank_train.npz\'), fp_models,\n                           workers=os.cpu_count(), w_pv=0.88)\n    bl = E.blend_scores(S[\'pv\'], S[\'ours\'], 0.88)\n    res = {}\n    for r in recs:\n        mid = r[\'mid\']\n        if mid not in bl:\n            continue\n        order = np.argsort(-bl[mid], kind=\'mergesort\')[:80]\n        smis, keys, seen = [], [], set()\n        for c in r[\'cand\'][order]:\n            s = E.POOL[\'smiles\'][c]; k = E.canon_key(s)\n            if k is None or k in seen:\n                continue\n            seen.add(k); smis.append(s); keys.append(k)\n            if len(smis) >= 40:\n                break\n        res[str(mid)] = dict(smiles=smis, keys=keys)\n    json.dump(res, open(out, \'w\'))\n    print(\'engine lists\', len(res), f\'{time.time()-T0:.0f}s\', flush=True)\n'}
ENG_INPUT_SHA = {'coco_fp.npy': '5d960d7355e0e10b22f1b2fa97c9fde37e86ddffbea8e562ec1873bdbb0e5f88', 'coco_mass.npy': '2b2af92c3185fa8cd52a4a3e3e7fd7eda4d94e8b9402662a3a4c55ffb13c6e5c', 'coco_meta.pkl': '29eab3083adcb329b52ac7909b4d3ce21f9f95a4008a93356bc526310ab44091', 'fp_bits.npy': 'a61e372fc41091bf5c7467e08134081266262665077cb9e2c56d73e4887f3013', 'rank_train.npz': '41ccec87fba5f8ab255aaf5a723567f25bd766064265c5cc2b83ba2cf15f57f8', 'fp_single_s2.pt': 'c19aa29016ed4d7a67bac38105b8fa5076c696c29eb66a504558c9b2e52b614a', 'fp_merged_m1.pt': '261529c823825cbe2028684bbf7214a95a7e631ca9f656336d577c507cce8294', 'sim_rank_rows_nofp.npz': 'c19ae76f69c7a3f23e67d2e27441830213eee941aa5658b47b1798872a61a7e9', 'bio_fp.npy': 'e1c0687abc73ca093920aa5b19c93db934c6064543289b7ef37dd8f6a7449f51', 'bio_mass.npy': '7b67b125f7bed307c2ac08bf41ad92044bfc2177519c5d4bb36c2c51789a2921', 'bio_meta.pkl': 'b94aaa0c68d6a0d7bb36db267d271f679e4286b0cd2de31f982f14f0f1bcc051'}
for _n, _src in ENG_FILES.items():
    open(os.path.join(ENG_DIR, _n), 'w').write(_src)
ENG, eng_status = {}, 'not run'
_eng_out = Path('/kaggle/working/eng_lists.json')
_eng_log = Path('/kaggle/working/eng_runner.log')
_eng_started = time.monotonic()
eng_seconds, eng_returncode = 0.0, None
def _kill_engine_group(proc):
    # the engine's multiprocessing workers share its process group
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except (ProcessLookupError, PermissionError):
        pass
try:
    _eng_out.unlink(missing_ok=True)               # never read the output of an earlier execution
    import glob as _glob
    for _n, _h in ENG_INPUT_SHA.items():
        _hits = sorted(_glob.glob('/kaggle/input/**/' + _n, recursive=True), key=len)
        assert _hits and sha256(Path(_hits[0])) == _h, f'second-engine input missing or changed: {_n}'
    # Keep time for the main engine, both forward models and the final checks inside the nine-hour limit.
    _reserve = 3000 + ICE_BUDGET + 300 + 3000 + 300 + 600
    _eng_timeout = min(3 * 3600, NOTEBOOK_STARTED + 9 * 3600 - 600 - time.monotonic() - _reserve)
    if _eng_timeout < 300:
        raise TimeoutError('not enough notebook time left for the second engine')
    _cmd = [sys.executable, os.path.join(ENG_DIR, 'eng_runner.py'), os.path.join(COMP, 'test.parquet'),
            os.path.join(COMP, 'train.parquet'), os.path.join(COMP, 'sample_submission.csv'), str(_eng_out)]
    with _eng_log.open('w', encoding='utf8') as _log:
        _proc = subprocess.Popen(_cmd, stdout=_log, stderr=subprocess.STDOUT, cwd=ENG_DIR, start_new_session=True)
        try:
            eng_returncode = _proc.wait(timeout=_eng_timeout)
        except BaseException:
            _kill_engine_group(_proc); _proc.wait(timeout=30)
            raise
        if eng_returncode != 0:
            _kill_engine_group(_proc)
            raise subprocess.CalledProcessError(eng_returncode, _cmd)
    _loaded = json.loads(_eng_out.read_text(encoding='utf8'))
    assert isinstance(_loaded, dict) and _loaded, 'second engine produced no candidate lists'
    for _mid, _entry in _loaded.items():           # keep only well-formed lists: aligned SMILES and 14-character keys
        _sm, _ks = (_entry.get('smiles'), _entry.get('keys')) if isinstance(_entry, dict) else (None, None)
        if (isinstance(_sm, list) and isinstance(_ks, list) and 1 <= len(_sm) == len(_ks) <= 40
                and all(isinstance(x, str) and x for x in _sm) and all(isinstance(k, str) and len(k) == 14 for k in _ks)
                and len(set(_ks)) == len(_ks)):
            ENG[str(_mid)] = dict(smiles=_sm, keys=_ks)
    assert ENG, 'second engine produced no valid candidate list'
    eng_status = 'ok'
    print('second-engine lists', len(ENG), 'of', len(_loaded))
except Exception as e:
    ENG, eng_status = {}, 'failed: ' + repr(e)[:200]
    print('SECOND ENGINE FAILED -> single-engine lists kept:', repr(e))
finally:
    eng_seconds = round(time.monotonic() - _eng_started, 1)
    if _eng_log.exists():
        with _eng_log.open('rb') as _f:
            _f.seek(max(0, _eng_log.stat().st_size - 3000))
            print(_f.read().decode('utf8', errors='replace'))
    print('second-engine wall seconds', eng_seconds, 'return code', eng_returncode)


In [ ]:
# Build the measured-reference cache from the current competition training table.
sys.path.insert(0, V4CODE)
os.makedirs('work', exist_ok=True)
for f in ['pool_meta.parquet', 'pool_fp.npy', 'pool_frag_off.npy', 'pool_frag_mass.npy', 'fp_bits.npy', 'train_structs.parquet', 'train_fp_sel.npy']:
    src = os.path.join(POOL_DIR, f)
    dst = f'work/{f}'
    if not os.path.exists(dst):
        os.symlink(src, dst)
from casmi.build import build_spec_cache
build_spec_cache(os.path.join(COMP, 'train.parquet'), 'work')
import casmi; print('casmi from', casmi.__file__)
sys.path.insert(1, os.path.join(V4CODE, 'fe_v4'))
print(f'cache built {time.time()-T0:.0f}s')

In [ ]:
# Main FPNet A engine, candidate generation, five feature families and four ranker boosters.
import numpy as np, pandas as pd, torch, lightgbm as lgb
from casmi.library import Library
from casmi.pool import Pool
from casmi.engine import Engine, EngineCfg, FEATURES
from casmi import fpnet, chem
import v1engine
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
L = Library('work'); P = Pool('work')
tfp = np.load('work/train_fp_sel.npy')
bank = fpnet.ModelBank([os.path.join(V4MOD, 'fpnet_0.pt')], device=dev)
E = Engine(L, P, tfp, EngineCfg(generate=True), bank)
V = v1engine.V1FE(E, [os.path.join(V4MOD, m['file']) for m in MAN['fe_models']], MAN['families'], device=dev)
if 'derivation' in MAN['families']:
    n_bad = V.mods['derivation'].self_test()
    print('derivation prior self-test mismatches:', n_bad)
    assert n_bad == 0, 'RDKit build differs from the one the derivation priors were built with'
RK = pickle.load(open(os.path.join(V4MOD, MAN['ranker']['file']), 'rb'))
RK_FEATS = list(RK['features']); RK_B = [lgb.Booster(model_str=s) for s in RK['boosters']]
def rank_score(X, names):
    pos = {n: i for i, n in enumerate(names)}
    miss = [f for f in RK_FEATS if f not in pos]
    assert not miss, f'ranker features not produced (would be zero-filled): {miss}'
    Xs = np.ascontiguousarray(X[:, [pos[f] for f in RK_FEATS]], dtype=np.float32)
    return np.mean([b.predict(Xs, num_threads=4) for b in RK_B], axis=0)
print('engine ready; ranker features', len(RK_FEATS), 'boosters', len(RK_B), f'{time.time()-T0:.0f}s')

In [ ]:
# Rank the mass-compatible and generated main-channel candidates for every molecule.
te = pd.read_parquet(os.path.join(COMP, 'test.parquet'))
te['nm'] = chem.neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)
mols = list(te.groupby('molecule_id', sort=False))
BASE, n_err = {}, 0
POOL_POP = np.load(POP_ROOT / 'pool_lsid.npy').astype(np.float64) + np.load(POP_ROOT / 'pool_lpmid.npy').astype(np.float64)
assert len(POOL_POP) == len(P.key), (len(POOL_POP), len(P.key))
n_pp = 0
for gi, (mid, sub) in enumerate(mols):
    nms = sub.nm.values[np.isfinite(sub.nm.values)]
    smis, keys, lib_max, scs, forms = [], [], 0.0, [], []
    pids_ = []
    if len(nms):
        target = float(np.median(nms))
        spectra = []
        for r in sub.itertuples():
            ce = r.collision_energy_ev
            ce_arr = np.atleast_1d(np.asarray(ce, dtype=float)) if ce is not None else np.zeros(0)
            spectra.append(dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),
                                mode=1 if r.ionization_mode == 'positive' else -1, adduct=r.adduct,
                                prec=float(r.precursor_mz), ce=float(ce_arr.mean()) if len(ce_arr) else np.nan,
                                ce_n=int(len(ce_arr)) if len(ce_arr) else 1))
        try:
            C, X, F, names, info = V.run(spectra, target)
            lib_max = float(info.get('lib_max', 0.0))
            if len(C.pid):
                score = rank_score(np.concatenate([X, F], 1), list(FEATURES) + list(names))
                order = np.argsort(-score, kind='stable')
                seen = set()
                for i in order:
                    k = C.key[i]
                    if k in seen: continue
                    seen.add(k); smis.append(C.smiles[i]); keys.append(k); scs.append(float(score[i])); forms.append(C.formula[i]); pids_.append(int(C.pid[i]))
                    if len(smis) >= TOPN: break
        except Exception as e:
            print('ERROR', mid, repr(e)); n_err += 1
    if POOLPOP_MU > 0 and len(scs) > 1:
        # z(ranker) + mu * popularity for pool structures; generated analogues keep their positions.
        s_ = np.asarray(scs, np.float64); z_ = (s_ - s_.mean()) / (s_.std() + 1e-9)
        pid_ = np.asarray(pids_); isg_ = pid_ < 0
        f_ = z_ + POOLPOP_MU * np.where(isg_, 0.0, POOL_POP[np.maximum(pid_, 0)])
        pool_i = np.where(~isg_)[0]; o_ = np.empty(len(pid_), np.int64)
        o_[isg_] = np.where(isg_)[0]; o_[~isg_] = pool_i[np.argsort(-f_[pool_i], kind='stable')]
        smis = [smis[i] for i in o_]; keys = [keys[i] for i in o_]; scs = [float(f_[i]) for i in o_]
        forms = [forms[i] + ('|gen' if isg_[i] else '') for i in o_]      # generated structures form their own isomer groups
        n_pp += 1
    BASE[mid] = (smis, keys, lib_max, scs, forms)
    if (gi + 1) % 25 == 0 or gi == len(mols) - 1:
        print(f'  {gi+1}/{len(mols)} molecules {time.time()-T0:.0f}s', flush=True)
print('errors', n_err, '| popularity-ordered lists', n_pp)
assert n_err == 0, f'{n_err} main-channel molecules failed'
assert all(BASE[m][0] for m in BASE), 'A main-channel candidate list is empty'
assert set(PC) == set(BASE), 'PubChem and main channels cover different molecule IDs'

In [ ]:
# ICEBERG post-ranker isomer re-scoring (casmi26-iceberg). Optional: any failure keeps the ranker order.
import gc
ICE_SCORES, ice_stats = {}, dict(molecules=0, changed_top25=0, changed_top1=0)
try:
    ICE_PKG = os.path.dirname(find('casmi26-iceberg/**/ice_runner.py'))
    sys.path.append(ICE_PKG)
    import fuse as ice_fuse
    del V, E, bank; gc.collect(); torch.cuda.empty_cache()
    # priority: molecules without a library-level match first (class 1 rarely changes), then the rest
    mids = sorted(BASE, key=lambda m: BASE[m][2])
    cands = {m: ice_fuse.ice_candidates(BASE[m][0], BASE[m][4], TOPN) for m in mids}
    if ENG:                              # second-engine candidates join the same-formula groups scored by the forward models
        from rdkit.Chem.rdMolDescriptors import CalcMolFormula as _CMF
        from rdkit import Chem as _Ch
        def _f2(s):
            try:
                return _CMF(_Ch.MolFromSmiles(s))
            except Exception:
                return s
        n_add = 0
        for m in mids:
            e = ENG.get(str(m))
            if not e or not e.get('smiles'):
                continue
            bs, bk, bf = list(BASE[m][0][:TOPN]), set(BASE[m][1][:TOPN]), list(BASE[m][4][:TOPN])
            es = [s for s, k in zip(e['smiles'][:40], e['keys'][:40]) if k not in bk]
            ef = [_f2(s) for s in es]
            extra = ice_fuse.ice_candidates(bs + es, bf + ef, len(bs) + len(es))
            before = len(cands.get(m, []))
            cands[m] = list(dict.fromkeys(list(cands.get(m, [])) + extra))
            n_add += len(cands[m]) - before
        print('second-engine candidates added to the forward-model input:', n_add, flush=True)
    if ICE_PC:                          # also the PubChem-only list of molecules the v3.6 gate may use (lib_max < LIB_TAU)
        from rdkit.Chem.rdMolDescriptors import CalcMolFormula
        from rdkit import Chem as _Chem
        def _form(s):
            try:
                return CalcMolFormula(_Chem.MolFromSmiles(s))
            except Exception:
                return s
        for m in mids:
            p = PC.get(m) if isinstance(PC, dict) else None
            if p and p.get('pc') and BASE[m][2] < LIB_TAU:
                p['pc_form'] = [_form(x) for x in p['pc']]
                extra = ice_fuse.ice_candidates(p['pc'], p['pc_form'], 25)
                cands[m] = list(dict.fromkeys(list(cands.get(m, [])) + extra))
    cands = {m: c for m, c in cands.items() if c}
    items = ice_fuse.build_ice_input(te, cands)
    print('ICE input molecules', len(items), 'candidates', sum(len(it['cands']) for it in items), f'{time.time()-T0:.0f}s', flush=True)
    ICE_SCORES = ice_fuse.run_ice(ICE_PKG, items, workdir='/kaggle/working/ice_work', device='cuda', budget_s=ICE_BUDGET,
                                  site='/kaggle/working/ice_site')
    meta_f = '/kaggle/working/ice_work/ice_out.json.meta.json'
    if os.path.exists(meta_f):
        print('ICE meta', open(meta_f).read()[:1500])
except Exception as e:
    print('ICE FAILED -> ranker order kept:', repr(e))
# v4h: GLACIER third re-scoring term (casmi26-glacier) on the SAME runner input; any GLACIER failure -> v4g (ICE only)
GL_SCORES, gl_stats, GL_LAM, GL_BUDGET = {}, dict(molecules=0, changed_vs_ice_top25=0, changed_vs_ice_top1=0), 0.5, 3000
try:
    GL_PKG = os.path.dirname(find('casmi26-glacier/**/gl_runner.py'))
    sys.path.append(GL_PKG)
    import gl_fuse
    gc.collect(); torch.cuda.empty_cache()   # ICE ran in its own (finished) process; free the notebook's cached GPU memory
    # GLACIER scores [M+H]+ acquisitions only (wangpenghua v27/v29); other adducts get no GLACIER term.
    gl_items = [dict(it, spectra=[s for s in it['spectra'] if s.get('adduct') == '[M+H]+']) for it in items]
    gl_items = [it for it in gl_items if it['spectra']]
    gl_stats['mh_molecules'] = len(gl_items)
    print('GLACIER on [M+H]+ spectra: molecules', len(gl_items), 'of', len(items))
    GL_SCORES = gl_fuse.run_gl(GL_PKG, gl_items, workdir='/kaggle/working/gl_work', device='cuda', budget_s=GL_BUDGET,
                               site='/kaggle/working/ice_site', wheels=os.path.join(ICE_PKG, 'wheels'))
    meta_f = '/kaggle/working/gl_work/gl_out.json.meta.json'
    print('GL meta', open(meta_f).read()[:1500] if os.path.exists(meta_f) else 'missing')
except Exception as e:
    GL_SCORES = {}
    print('GL FAILED -> v4g behaviour (ICE only):', repr(e))
def gl_of(m):
    g = GL_SCORES.get(str(m), {}) if isinstance(GL_SCORES, dict) else {}
    return g if isinstance(g, dict) and any(v is not None for v in g.values()) else {}
def gl_rerank(order0, smis, keys, scs, forms, ice, gl, top_n, tag=''):
    # z(scs) + ICE_LAM z(ice) + GL_LAM z(gl) in the same slots; no GL scores / any failure -> order0 (= v4g order)
    if not gl:
        return order0
    try:
        o = gl_fuse.rerank_multi(smis, keys, scs, forms, [ice, gl], [ICE_LAM, GL_LAM], top_n=top_n)
    except Exception as e:
        print('gl rerank failed', tag, repr(e)); return order0
    for k, v in (('molecules', 1), ('changed_vs_ice_top25', int(o[:25] != order0[:25])), ('changed_vs_ice_top1', int(o[:1] != order0[:1]))):
        gl_stats[tag + k] = gl_stats.get(tag + k, 0) + v
    return o
for m in list(BASE):
    smis, keys, lib_max, scs, forms = BASE[m]
    ice = ICE_SCORES.get(str(m), {}) if ICE_SCORES else {}
    gl = gl_of(m)
    if ice and any(v is not None for v in ice.values()):
        try:
            order = ice_fuse.rerank(smis, keys, scs, forms, ice, lam=ICE_LAM, top_n=TOPN)
            order = gl_rerank(order, smis, keys, scs, forms, ice, gl, TOPN)
            s2 = [smis[i] for i in order]; k2 = [keys[i] for i in order]
            ice_stats['molecules'] += 1
            ice_stats['changed_top25'] += int(s2[:25] != smis[:25]); ice_stats['changed_top1'] += int(bool(s2) and s2[0] != smis[0])
            smis, keys = s2, k2
        except Exception as e:
            print('rerank failed', m, repr(e))
    BASE[m] = (smis[:25], keys[:25], lib_max)
    p = PC.get(m) if (ICE_PC and isinstance(PC, dict)) else None
    if p and p.get('pc_form') and ice:
        try:                             # re-order the PubChem list by z(f.z) + lam z(ice) inside same-formula groups;
            o = ice_fuse.rerank(p['pc'], p['pc_keys'], p['pc_fz'], p['pc_form'], ice, lam=ICE_LAM, top_n=25)
            o = gl_rerank(o, p['pc'], p['pc_keys'], p['pc_fz'], p['pc_form'], ice, gl, 25, 'pc_')
            fz0 = p['pc_fz'][0]          # the gate keeps using the ORIGINAL top f.z (rel), so gate decisions are unchanged
            p['pc'] = [p['pc'][i] for i in o]; p['pc_keys'] = [p['pc_keys'][i] for i in o]
            p['pc_fz'] = [fz0] + [p['pc_fz'][i] for i in o][1:]
            ice_stats['pc_changed'] = ice_stats.get('pc_changed', 0) + int(o[:1] != [0])
        except Exception as e:
            print('pc rerank failed', m, repr(e))
print('ICE rerank stats', ice_stats, f'{time.time()-T0:.0f}s')
print('GL rerank stats', gl_stats, f'{time.time()-T0:.0f}s')

In [ ]:
# Merge the two channels, fuse the result with the second engine and save the ordered submission.
def merge(base, base_keys, pc, pc_keys, slots, n=25):
    bk = set(base_keys)
    pcs = [s for s, k in zip(pc, pc_keys) if k not in bk]
    out, bi, pj = [], 0, 0
    for pos in range(1, n + 1):
        if pos in slots and pj < len(pcs):
            out.append(pcs[pj]); pj += 1
        elif bi < len(base):
            out.append(base[bi]); bi += 1
        elif pj < len(pcs):
            out.append(pcs[pj]); pj += 1
    return out
from rdkit.Chem.rdMolDescriptors import CalcMolFormula as _CMF3
from rdkit import Chem as _Ch3
ALPHA, KRR, FUSE_N = 0.6, 3.0, 40      # weight of the second engine, rank offset and fused-list length (public fusion notebook)
FUSED_LAM = 0.5                        # ICEBERG and GLACIER weight in the re-ordering of the fused list (as in the main and PubChem lists)
def _formula(s):
    try:
        return _CMF3(_Ch3.MolFromSmiles(s))
    except Exception:
        return s
def merge_lists(smis, keys, lib_max, p):
    """Main list with PubChem proposals inserted at the gate's positions, and the branch taken."""
    if not p or not p.get('pc'):
        return smis, 'no_pc'
    if lib_max >= LIB_TAU:
        return smis, 'untouched'
    bp = p.get('best_pool_fz')
    rel = (p['fz_top'] if p.get('fz_top') is not None else p['pc_fz'][0]) - bp if (bp is not None and np.isfinite(bp)) else 1e9
    early = rel > REL_TH
    return merge(smis, keys, p['pc'], p['pc_keys'], SLOTS_AGG if early else SLOTS_GENTLE), 'aggressive' if early else 'gentle'
def fuse_rrf(v_smis, v_keys, e_smis, e_keys, n=FUSE_N):
    """Weighted reciprocal-rank fusion of the pipeline list and the second engine's list: one entry per scorer key,
    score 1/(KRR + rank) + ALPHA/(KRR + engine rank); ties keep the pipeline order."""
    sc, smi_of = {}, {}
    for r, (s, k) in enumerate(zip(v_smis, v_keys), 1):
        if k:
            sc[k] = sc.get(k, 0.0) + 1.0 / (KRR + r); smi_of.setdefault(k, s)
    for r, (s, k) in enumerate(zip(e_smis, e_keys), 1):
        if k:
            sc[k] = sc.get(k, 0.0) + ALPHA / (KRR + r); smi_of.setdefault(k, s)
    order = sorted(sc, key=lambda k: -sc[k])[:n]
    return [smi_of[k] for k in order], order, [sc[k] for k in order]
def forward_rerank(mid, smis, keys, scores):
    """Same-formula re-ordering of a fused list: z(fusion score) + FUSED_LAM (z(ICEBERG) + z(GLACIER)); unchanged without scores."""
    ice = ICE_SCORES.get(str(mid), {}) if isinstance(ICE_SCORES, dict) else {}
    if not ice or 'ice_fuse' not in globals():
        return smis, keys
    try:
        forms = [_formula(x) for x in smis]
        o = ice_fuse.rerank(smis, keys, scores, forms, ice, lam=FUSED_LAM, top_n=len(smis))
        g = gl_of(mid)
        if g:
            o = gl_fuse.rerank_multi(smis, keys, scores, forms, [ice, g], [FUSED_LAM, FUSED_LAM], top_n=len(smis))
        assert sorted(o) == list(range(len(smis)))
        return [smis[i] for i in o], [keys[i] for i in o]
    except Exception as ex:
        print('fused forward rerank failed', mid, repr(ex)); return smis, keys
def fused_list(smis, keys, lib_max, p, eng=None, mid=None, fuse_strong=False):
    """Final list of one molecule: the gate's merge, then fusion with the second engine when there is no strong
    library match. Returns the list, the slot branch and whether the fusion was applied."""
    final, branch = merge_lists(smis, keys, lib_max, p)
    if not (eng and eng.get('smiles') and final and (lib_max < LIB_TAU or fuse_strong)):
        return final, branch, False
    key_of = dict(zip(p['pc'], p['pc_keys'])) if (p and p.get('pc')) else {}
    key_of.update(zip(smis, keys))
    fkeys = [key_of.get(s) or chem.score_key(s) for s in final]
    fs, fk, fsc = fuse_rrf(final, fkeys, eng['smiles'], eng.get('keys') or [])
    return forward_rerank(mid, fs, fk, fsc)[0], branch, True

# Self-test on synthetic lists: the hidden rerun must not be the first time these branches execute.
_b, _bk = [f'B{i}' for i in range(30)], [f'kb{i}' for i in range(30)]
_p = dict(pc=[f'P{i}' for i in range(25)], pc_keys=[f'kp{i}' for i in range(25)], pc_fz=[7.0] * 25, fz_top=200.0, best_pool_fz=100.0)
_f, _br = merge_lists(_b, _bk, 0.5, _p)                                   # small raw-score gap: positions 4, 8, 12, 16, 20
assert _br == 'gentle' and _f[:8] == ['B0', 'B1', 'B2', 'P0', 'B3', 'B4', 'B5', 'P1'] and len(_f) == 25 and len(set(_f)) == 25
_f, _br = merge_lists(_b, _bk, 0.5, dict(_p, fz_top=900.0))               # large gap: positions 2, 4, 6, 8, 10
assert _br == 'aggressive' and _f[:4] == ['B0', 'P0', 'B1', 'P1']
assert merge_lists(_b, _bk, 0.5, dict(_p, best_pool_fz=float('nan')))[1] == 'aggressive'
assert merge_lists(_b, _bk, 0.5, dict(_p, best_pool_fz=None))[1] == 'aggressive'
assert merge_lists(_b, _bk, 0.5, dict(_p, fz_top=None, pc_fz=[800.0] * 25))[1] == 'aggressive'
_f, _br = merge_lists(_b, _bk, 0.5, dict(_p, pc=['Q2'] + _p['pc'][1:], pc_keys=['kb2'] + _p['pc_keys'][1:]))   # proposal already in the main list
assert 'Q2' not in _f and _f[:4] == ['B0', 'B1', 'B2', 'P1']
assert merge_lists(_b, _bk, 0.95, _p) == (_b, 'untouched')
assert merge_lists(_b, _bk, 0.5, None) == (_b, 'no_pc') and merge_lists(_b, _bk, 0.5, dict(_p, pc=[], pc_keys=[])) == (_b, 'no_pc')
assert merge_lists(_b[:1], _bk[:1], 0.5, dict(_p, pc=['P0'], pc_keys=['kp0']))[0] == ['B0', 'P0']
# fusion: B1 is supported by both lists, E0 only by the second engine
_e = dict(smiles=['E0', 'B1'], keys=['ke0', 'kb1'])
_f, _br, _fu = fused_list(_b, _bk, 0.5, _p, _e, 'synthetic')
assert _fu and _br == 'gentle' and _f[:5] == ['B1', 'B0', 'B2', 'E0', 'P0'] and 25 <= len(_f) <= FUSE_N and len(set(_f)) == len(_f)
_f, _br, _fu = fused_list(_b, _bk, 0.95, _p, _e, 'synthetic')             # strong library match: untouched
assert not _fu and _br == 'untouched' and _f == _b
_f, _br, _fu = fused_list(_b, _bk, 0.95, _p, _e, 'synthetic', fuse_strong=True)
assert _fu and _f[:2] == ['B1', 'B0']
_f, _br, _fu = fused_list(_b, _bk, 0.5, None, _e, 'synthetic')            # no PubChem proposals: fusion still applies
assert _fu and _br == 'no_pc' and _f[:4] == ['B1', 'B0', 'B2', 'E0']
assert fused_list(_b, _bk, 0.5, _p, None, 'synthetic') == (merge_lists(_b, _bk, 0.5, _p)[0], 'gentle', False)
assert fused_list(_b, _bk, 0.5, _p, dict(smiles=[], keys=[]), 'synthetic')[2] is False
assert fused_list(_b, _bk, 0.5, _p, dict(smiles=['E0', 'E1'], keys=[None, 'kb0']), 'synthetic')[0][0] == 'B0'   # engine entry without a key is skipped

def distinct(smiles_list):
    """One SMILES per scorer key, order kept; structures the scorer cannot key are dropped."""
    out, seen = [], set()
    for s in smiles_list:
        k = chem.score_key(s)
        if k is not None and k not in seen:
            seen.add(k); out.append(s)
    return out

rows, stats = [], dict(untouched=0, gentle=0, aggressive=0, no_pc=0)
fusion = dict(engine_lists=len(ENG), applied=0, top1_changed=0, strong_library_top1_would_change=0, errors=0)
for mid in te.molecule_id.unique():
    smis, keys, lib_max = BASE.get(mid, ([], [], 0.0))
    eng = ENG.get(str(mid)) if isinstance(ENG, dict) else None
    plain, branch = merge_lists(smis, keys, lib_max, PC.get(mid))
    final = plain
    try:
        fused_smiles, _, fused = fused_list(smis, keys, lib_max, PC.get(mid), eng, mid)
        if fused:
            fused_smiles = distinct(fused_smiles)[:25]
            if fused_smiles:
                final = fused_smiles
                fusion['applied'] += 1; fusion['top1_changed'] += int(final[:1] != plain[:1])
        elif eng and eng.get('smiles') and lib_max >= LIB_TAU:     # diagnostic only: what fusion would do to a strong library match
            what_if = distinct(fused_list(smis, keys, lib_max, PC.get(mid), eng, mid, fuse_strong=True)[0])[:25]
            fusion['strong_library_top1_would_change'] += int(what_if[:1] != plain[:1])
    except Exception as ex:                                        # any fusion problem keeps the single-engine list
        print('fusion failed', mid, repr(ex)); final = plain; fusion['errors'] += 1
    stats[branch] += 1
    assert final, f'No candidate for {mid}'
    rows.append((mid, ';'.join(final[:25])))
print('fusion', fusion)
print('merge stats', stats)
sub = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])
samp = pd.read_csv(os.path.join(COMP, 'sample_submission.csv'))
sub = samp[['molecule_id']].merge(sub, on='molecule_id', how='left')
assert len(sub) == len(samp) and sub.molecule_id.is_unique and sub.smiles.notna().all()
assert sub.smiles.map(lambda s: len(s.split(';'))).max() <= 25
sub.to_csv('submission.csv', index=False)
json.dump(dict(secs=round(time.time() - T0), merge=stats, fusion=fusion, second_engine=eng_status, popularity_lists=n_pp, n_errors=n_err, families=MAN['families'],
               gpu=torch.cuda.get_device_name(0) if dev == 'cuda' else None), open('run_manifest.json', 'w'), indent=1)
print('done', sub.shape, f'{time.time()-T0:.0f}s')
sub.head()

In [ ]:
# Validate output against the visible submission contract and record execution diagnostics.
from rdkit import Chem as _Chem
assert sub.molecule_id.tolist() == samp.molecule_id.tolist()
assert sub.molecule_id.is_unique and sub.smiles.notna().all()
for mid, answer in sub.itertuples(index=False):
    guesses = answer.split(';')
    assert 1 <= len(guesses) <= 25, mid
    keys = [chem.score_key(s) for s in guesses]
    assert all(k is not None for k in keys) and len(keys) == len(set(keys)), mid
receipt = {
    'method': 'attributed_v4i_popularity_engine_fusion_glmh_nopcpop',
    'resource_manifest_sha256': resource_receipt['manifest_sha256'],
    'train_sha256': sha256(TRAIN_PATH),
    'test_sha256': sha256(TEST_PATH),
    'submission_sha256': sha256(Path('/kaggle/working/submission.csv')),
    'molecules': len(sub), 'main_errors': n_err,
    'pubchem_lists': len(PC), 'merge': stats, 'iceberg': ice_stats,
    'glacier': gl_stats,
    'popularity': dict(lists=n_pp, pubchem_lambda=POP_LAM, union=POP_UNION, pool_mu=POOLPOP_MU,
                       manifest_sha256=sha256(POP_ROOT / 'MANIFEST.json')),
    'forward_weights': dict(iceberg=ICE_LAM, glacier=GL_LAM, fused=FUSED_LAM),
    'second_engine': eng_status, 'second_engine_seconds': eng_seconds, 'second_engine_return_code': eng_returncode,
    'fusion': dict(fusion, alpha=ALPHA, k=KRR, fused_length=FUSE_N),
    'glacier_manifest_sha256': sha256(GL_ROOT / 'MANIFEST.json'),
    'elapsed_seconds': round(time.time()-T0, 1),
}
Path('/kaggle/working/v29_execution_receipt.json').write_text(json.dumps(receipt, indent=2))
display(pd.Series({k: receipt[k] for k in ('molecules','main_errors','pubchem_lists','elapsed_seconds')}))

### One molecule through the final gate

The header below summarises one molecule from the visible, **unlabelled** format example: its acquisition count, estimated neutral mass, strongest library match, the merge branch and Δ. The table lists its first five final structures with their source channel, and Figure 21 draws the first three; the drawings make it easier to see why mass alone cannot distinguish related graphs. The example reports the merge decision and the source channels; it does not trace the score changes made by each ranking component.

In [ ]:
# Choose a gate-eligible query if one exists; otherwise show the weakest library match.
example_ids = list(te.molecule_id.drop_duplicates())
eligible_ids = [m for m in example_ids if BASE[m][2] < LIB_TAU and PC.get(m, {}).get('pc')]
example_mid = eligible_ids[0] if eligible_ids else min(example_ids, key=lambda m: BASE[m][2])
example_query = te[te.molecule_id.eq(example_mid)]
main_smiles, main_keys, example_lib = BASE[example_mid]
proposal = PC.get(example_mid, {})
pc_smiles = proposal.get('pc', [])
pc_keys = proposal.get('pc_keys', [])
_final, _branch, _fused = fused_list(main_smiles, main_keys, example_lib, proposal or None, ENG.get(str(example_mid)), example_mid)
branch = {'no_pc': 'main: no PubChem proposals', 'untouched': 'main: strong measured-library match',
          'aggressive': 'PubChem inserted at 2/4/6/8/10', 'gentle': 'PubChem inserted at 4/8/12/16/20'}[_branch]
if _fused:
    branch += ', then fused with the second engine'
_ft, _bp = proposal.get('fz_top'), proposal.get('best_pool_fz')
delta_text = f' · Δ {_ft - _bp:.0f}' if (_ft is not None and _bp is not None and np.isfinite(_bp)) else ''
display(Markdown(
    f'**{example_mid}** · {len(example_query)} measured view(s) · '
    f'neutral mass {np.nanmedian(example_query.nm):.4f} Da · '
    f'library match {example_lib:.3f} · '
    f'**{branch}**' + delta_text
))
final_smiles = sub.set_index('molecule_id').at[example_mid, 'smiles'].split(';')
main_key_set, pc_key_set = set(main_keys), set(pc_keys)
eng_key_set = set((ENG.get(str(example_mid)) or {}).get('keys', []))
example_rows = []
for rank, smiles in enumerate(final_smiles[:5], 1):
    key = chem.score_key(smiles)
    channels = [name for name, known in (('main', main_key_set), ('PubChem', pc_key_set), ('second engine', eng_key_set)) if key in known]
    example_rows.append({'Rank': rank, 'Channel': ' + '.join(channels) or 'unattributed',
                         'Connectivity key': key, 'SMILES': smiles})
display(Markdown('**Worked-example candidates.**'))
display(pd.DataFrame(example_rows))
drawn = [_Chem.MolFromSmiles(s) for s in final_smiles[:3]]
if all(m is not None for m in drawn):
    display(Draw.MolsToGridImage(drawn, molsPerRow=3, subImgSize=(360, 280), useSVG=True,
                                 legends=[f'Rank {i}' for i in range(1, len(drawn) + 1)]))

*Figure 21. The top three final proposals for the molecule summarised above, in the submitted order. The header states which branch of the merge rule produced this list.*

## 13. A practical next experiment

On the development panel, popularity was the largest single effect and a promotion rule looked decisive; the leaderboard rewarded the first modestly and did not reward the second. The next experiment should therefore measure such choices on independently held-out identities, in scenarios that the real candidate sources can actually produce.

1. **Freeze identities and permitted inputs.** Assign every spectrum of one scorer-compatible identity group to one fold. Reserve an outer fold for the final audit and use inner folds for every choice. Declare whether held-out structures may stay in the pool as structure-only candidates.
2. **Audit static candidate coverage.** Before fitting anything, measure whether each held-out answer occurs in the permitted catalogues and neutral-mass windows. This separates losses caused by database absence or mass filtering from ranking errors.
3. **Fit the learned components inside the training folds.** Rebuild reference indices, fingerprint models, feature resources and ranker-training rows from permitted identities only. The supplied weights are inference artifacts without a documented identity-disjoint recipe; audit the training overlap of any pretrained component that is kept.
4. **Separate learned retrieval from ordering.** On inner folds, measure answer coverage after fingerprint shortlisting and analogue generation, then compare scoring and merging on a fixed candidate set. Freeze the choices before opening the outer fold. Report coverage, top-1, top-25 and MRR@25 over *all* held-out molecules, including those without a correct candidate, split by instrument, adduct and polarity.

[CSI:FingerID](https://www.pnas.org/doi/10.1073/pnas.1509788112) motivates a better spectrum-to-fingerprint representation; [MS2Query](https://www.nature.com/articles/s41467-023-37446-4) suggests analogue evidence when candidates exist; [MassSpecGym](https://github.com/pluskal-lab/MassSpecGym) offers harder evaluation settings. Sections 14–16 survey broader chemistry and physics ideas, not required inputs for this immediate experiment.

## 14. Further reading: data and representations

An external resource is valuable when it adds useful chemistry or representations beyond what the competition corpus already contains. Download size alone is a poor guide.

**Further-reading resources.**

| Resource | What it can add | What must be checked |
|---|---|---|
| [Enveda-180](https://github.com/enveda/enveda-180) | acquisition context and reproducible processing | Its MS/MS data are already a major training source; extra modalities are useful only if the inference task supplies a compatible signal. |
| [GNPS libraries](https://ccms-ucsd.github.io/GNPSDocumentation/browselibraries/), [MassBank releases](https://github.com/MassBank/MassBank-data), [MoNA](https://massbank.us/) | additional experimental references or analogue coverage | Audit exact spectra, scored connectivities, source terms, and experimental versus inferred annotations. |
| [PubChem](https://pubchem.ncbi.nlm.nih.gov/docs/downloads) | a broad structure-only candidate universe, plus substance, literature and patent counts usable as priors | A database entry supplies a candidate, not a measured spectrum or evidence that it is present in the sample. |
| [COCONUT 2.0](https://doi.org/10.1093/nar/gkae1063) | natural-product structures with downloadable formulae and monoisotopic masses | Standardise structures, retain provenance, and measure additional answer coverage after deduplication. |
| [ChEBI](https://www.ebi.ac.uk/chebi/downloads), [NPAtlas](https://www.npatlas.org/), [HMDB](https://www.hmdb.ca/downloads) | focused biological chemistry and annotations | Check each database’s organism and compound scope, licensing, and overlap with the existing candidate pool. |
| [DreaMS / GeMS](https://www.nature.com/articles/s41587-025-02663-3) | pretrained spectrum representations | Its pretraining data and acquisition settings differ from the hidden task; measure transfer by ion mode, mass, and instrument. |
| [MassSpecGym](https://github.com/pluskal-lab/MassSpecGym) | benchmark code and demanding generalisation protocols | Its release holds 231,104 spectra of 28,929 InChIKey connectivity blocks; 28,854 of them (99.7%) match the competition training file through a stored or scorer-normalised key, and 68 scorer identities are absent. It adds little new labelled chemistry here, while its code and protocols remain useful. |
| [MassIVE](https://massive.ucsd.edu/) | unlabelled experimental spectra for pretraining | Most records do not supply confirmed structure labels; preprocessing and overlap control remain necessary. |

PubChem expands candidate coverage, popularity adjusts selection and ranking, and ICEBERG and GLACIER compare predicted spectra within eligible formula groups. The next challenge is to measure which component improves coverage or ordering on genuinely held-out chemistry. Report **coverage before ranking** and **MRR after ranking** so the source of an improvement is visible.

Attach all adopted resources for offline inference and record their source, version, licence, preprocessing, and overlap policy. The resource map above is a literature guide; it does not imply that every listed dataset or model is loaded by this notebook.

### Longer-term direction: molecular geometry and quantum data

Open molecular-geometry datasets can help pretrain models of conformers (different three-dimensional arrangements of one molecular connectivity) and their energies, but they do not provide experimental MS/MS peaks for this task. [GEOM](https://www.nature.com/articles/s41597-022-01288-4) offers conformers and energies; [ANI-1x/ANI-1ccx](https://www.nature.com/articles/s41597-020-0473-z) develops learned potentials; [PubChemQC](https://pubs.acs.org/doi/10.1021/acs.jcim.7b00083) offers quantum-chemical structures. A practical first use here would be a small test of fragment plausibility or protonation on ambiguous candidates, with the candidate mass and adduct matched to experimental spectra. These are further reading, not inputs to this notebook.

The geometry resources above supply structures and energies. Section 15 considers the additional modelling needed to turn these into candidate-specific fragmentation evidence.

## 15. Further reading: selective physics

**DFT (density functional theory)** can estimate molecular geometries, charge distributions, bond dissociation tendencies and energies, including the differences between protomers: ions that carry the proton at different sites. **AIMD (ab initio molecular dynamics)** or semiempirical quantum-chemical MD can simulate fragmentation trajectories after energetic activation.

[QCxMS](https://pubs.acs.org/doi/10.1021/jasms.1c00098) extends QCEIMS (Quantum Chemistry Electron Ionisation Mass Spectrometry) to CID spectra by modelling collisions and fragmentation with on-the-fly molecular dynamics and the semiempirical GFN2-xTB Hamiltonian. A 2025 [pilot study](https://pubmed.ncbi.nlm.nih.gov/40876445/) generated CID-MS/MS for 121 Blood Exposome Database compounds, demonstrating feasibility but also the need for parameter/protomer selection and experimental calibration. [CIDMD](https://pmc.ncbi.nlm.nih.gov/articles/PMC11492810/) is another ab initio CID approach evaluated on a small set of low-mass compounds.

For this competition, full DFT/MD is unlikely to be the primary engine:

- it is a **forward** calculation for a proposed structure, so it does not remove the need to generate candidates;
- a single molecule has multiple conformers, protonation sites, adduct geometries, and collision trajectories;
- simulating thousands or millions of PubChem candidates for the roughly 400 molecules in the released format-example test would be hard to fit within the competition's [9-hour offline inference limit](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/rules); the hidden set may have a different size;
- reproducing instrument- and energy-dependent intensities is substantially harder than predicting plausible fragment masses;
- large, flexible molecules can lie beyond the easy regime of published small-molecule quantum-dynamics demonstrations.

The highest-value physics use is selective: formula/adduct consistency checks, protomer ranking, fragment plausibility for a small top candidate set, or generating targeted training examples in sparse chemical classes. Fast semiempirical methods or learned interatomic potentials are more realistic than routine high-level DFT. Any simulated score must be calibrated against held-out experimental timsTOF spectra.

[ANI active learning](https://doi.org/10.1063/1.5023802) illustrates how expensive calculations can focus on uncertain examples. Disagreement between rankers alone is not a calibrated uncertainty estimate; a selective experiment still needs held-out validation.


## 16. Historical challenge context

CASMI (Critical Assessment of Small Molecule Identification) began as a blinded comparison of identification workflows. The [founding overview](https://pmc.ncbi.nlm.nih.gov/articles/PMC3901296/) and [CASMI 2013 analysis](https://pmc.ncbi.nlm.nih.gov/articles/PMC4337369/) show why successful identification involves more than matching peak lists: formula inference, database coverage, data quality, and chemical context all matter.

The [CASMI 2016 automated-methods analysis](https://pmc.ncbi.nlm.nih.gov/articles/PMC5368104/) compared chemistry-based and learned approaches under different information conditions. Its useful lesson is complementarity: structural and metadata evidence can change the ranking produced by a spectrum-only method. The [CASMI 2022 results](https://www.fiehnlab.ucdavis.edu/casmi/casmi-2022-results) likewise separate tasks such as adduct, formula, class, and structure assignment.

These earlier challenges motivate a staged pipeline, but their scores cannot be transferred to CASMI 2026. Candidate databases, acquisition conditions, test chemistry, and evaluation rules differ. Use their methods as hypotheses to test on a reproducible contemporary validation split.

## 17. Sources and reproducibility

The [competition overview](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra), [data description](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data) and [rules](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/rules) define the prediction task, data roles and offline inference limit. The current model follows [Ahmed Berat Özer's public v4g–v4m notebooks](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4g-inference), whose notebook code is marked Apache 2.0. Its five input packages are preserved in [our attributed resource fork](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-attributed-v4g-reproduction-assets); its README maps every file to the upstream Kaggle dataset and explains the distinct terms for competition-derived data, COCONUT, PubChem, DreaMS and ICEBERG. The Kaggle `Other` label and source notices remain in force for those assets.

The restriction of GLACIER to `[M+H]+` follows [wangpenghua's v27](https://www.kaggle.com/code/wangpenghua/casmi26-v27-gl-mh) and [v29](https://www.kaggle.com/code/wangpenghua/casmi26-v29-fusion-pop-glmh). The second engine and the rank fusion follow the [public fusion notebook of seyitkaangunes](https://www.kaggle.com/code/seyitkaangunes/casmi26-v4n-engine-fusion-union-lb-0-399), which packages the [two-ranker engine of megayak](https://www.kaggle.com/code/megayak/casmi26-two-rankers-one-engine) and [prvsiyan's analog-propagation baseline](https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline) with prvsiyan's public datasets. Version 26 tested the main-list popularity weight of 0.25 from [huseyinemreaksoy's notebook](https://www.kaggle.com/code/huseyinemreaksoy/casmi26-sub-v4b-0-409-on-the-public). The popularity prior follows the [public v21 notebook](https://www.kaggle.com/code/wangpenghua/casmi26-v21-v4i-pop) by wangpenghua and the deeper PubChem screen follows [v4i/v4m](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4m-inference); our [popularity dataset](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior) is rebuilt from the official PubChem downloads, which are freely downloadable (see the [PubChem downloads documentation](https://pubchem.ncbi.nlm.nih.gov/docs/downloads)).

The attached [Open Model and Candidate Resources](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-open-model-and-candidate-resources) supplies the pinned main RDKit wheel.

For scientific context, [CSI:FingerID](https://www.pnas.org/doi/10.1073/pnas.1509788112) motivates spectrum-to-fingerprint matching, [spectral entropy](https://doi.org/10.1038/s41592-021-01331-z) helps compare measured spectra, [MS2Query](https://www.nature.com/articles/s41467-023-37446-4) motivates analogue evidence and [MassSpecGym](https://github.com/pluskal-lab/MassSpecGym) offers harder generalisation tests. The [ms-pred / ICEBERG code](https://github.com/coleygroup/ms-pred) supplies the bounded forward-model component. Sections 14–16 are further reading on additional data, physics and past challenges.

The executable cells verify the asset manifest, record train/test/submission hashes, component counts and runtime, and check the visible submission schema. The worked example in Section 12 shows how the two channels combine on one unlabelled molecule. A leaderboard score belongs to the exact version that Kaggle ran and accepted.

## Appendix A. What earlier versions taught us

Earlier versions of this **same notebook** tested a two-ranker model with PubChemLite proposals, then added ChEMBL or ChEBI candidate catalogues. Those recipes used different training snapshots and search rules. They remain available in the notebook's version history, in the [source-held-out validation audit](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-source-held-out-validation-audit) and in the [FPNet and Candidate Catalogues](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-open-fpnet-and-candidate-catalogues) dataset; neither dataset is mounted for current inference. The current version uses the train/COCONUT pool, a separately scored PubChem tier, a popularity prior and the ICEBERG and GLACIER forward models; the two-ranker engine of those earlier versions returns as the second engine of the rank fusion.

**Earlier submission results.**

| Earlier completed submission | Change | Public score |
|---|---|---:|
| Version 27 | Version 24 with a forward-model weight of 0.25 in the re-ordering of the fused list | 0.398 |
| Version 26 | Version 24 with a popularity weight of 0.25 in the main list | 0.405 |
| Version 25 | Version 24 with the popularity re-order of the second engine's list (weight 0.1) | 0.395 |
| Version 24 | Version 21 with GLACIER restricted to `[M+H]+` | **0.406** |
| Version 23 | Version 22 with the fusion limited again to `lib_max < 0.9` | 0.393 |
| Version 22 | Configuration of the public v29 notebook: unit forward weights, full1 checkpoint, fusion for every molecule, GLACIER on `[M+H]+` | 0.393 |
| Version 21 | Version-19 merge and 0.5 forward weights, plus second-engine fusion for `lib_max < 0.9` | 0.401 |
| Version 20 | + promotion of documented PubChem proposals, unit forward weights | 0.374 |
| Version 19 | + deeper PubChem screen and popularity prior | 0.386 |
| Version 18 | + GLACIER second forward model (v4h) | 0.373 |
| Version 17 | v4g main model + PubChem + ICEBERG | 0.366 |
| Version 13 | Two-ranker blend plus PubChemLite | 0.341 |
| Version 14 | Primary branch order plus the same proposal rule | 0.326 |

Version 20 lost 0.012 against version 19; Section 11 explains why its promotion rule had looked convincing in a replay, and later versions keep the version 19 merge rule. Versions 22 and 23 copied the configuration of the public v29 notebook (0.415 with private popularity arrays) and scored 0.393 with ours; version 24 returned to the version 21 settings and kept only the restriction of GLACIER to `[M+H]+`. Version 25 added a popularity re-order of the second engine's list, which version 26 left out again; version 26 instead raised the popularity weight of the main list from 0.15 to 0.25, following [huseyinemreaksoy's public notebook](https://www.kaggle.com/code/huseyinemreaksoy/casmi26-sub-v4b-0-409-on-the-public), which did not help either. Version 27 returned to the version 24 method and lowered the forward-model weight in the re-ordering of the fused list from 0.5 to 0.25, which did not help; the current version returns to the version 24 method and removes the popularity prior from the PubChem channel. On a selected local set of 48 GNPS and 48 Enveda molecules, the primary order had looked better on some covered answers. That local advantage did not transfer: version 14's official score was lower (0.326 versus version 13's 0.341). That difference is why Section 11 separates candidate presence from rank and Section 13 calls for a fresh identity-disjoint outer test. It cannot be explained from hidden per-molecule labels, which are unavailable.

One more historical check illustrates why adding a catalogue is not enough. On a selected 48-molecule Enveda slice, early ChEMBL insertions pushed existing correct candidates down: MRR@25 moved from 0.1837 to 0.1534. Keeping the original first 20 positions and appending new candidates gave 0.1847 on that **same development slice**. These numbers compare an older recipe on a selected subset; they are not accuracy estimates for the method in Section 10. The practical lesson is to measure both newly covered answers and displaced answers before adopting a larger pool.